In [1]:
# ============================================================
# 020_backward_fill_high_citation_core
# ============================================================
#
# Overview
# ----------------
# This notebook implements a backward-filling workflow for building a
# high-quality research corpus around a given Research Question (RQ).
# Starting from a small set of "core papers" that are both highly cited
# and strongly aligned with the RQ, it expands backward through citation
# links to identify older, foundational, or canonical works that should
# be included in the corpus.
#
# Papers are prioritized using a composite score that combines citation
# signals (e.g., cited-by counts) and an RQ alignment score derived from
# titles, abstracts, and topic metadata. The workflow is designed to run
# on a weekly or on-demand basis as part of a PhD-style research agent,
# supporting systematic literature coverage rather than ad-hoc discovery.
#
# Inputs / Outputs
# ----------------
# Inputs:
# - Existing paper corpus (from Notion / local artifacts), including:
#   title, year, identifiers (OpenAlex ID, DOI, arXiv ID), abstract,
#   citation counts, and reference lists where available.
# - Research Question (RQ) text and configuration parameters
#   (year range, scoring weights, hop depth, top-K thresholds).
# - External metadata and citation data via the OpenAlex API.
#
# Outputs:
# - A ranked list of candidate "backfilled" papers (typically older or
#   foundational works) with detailed scoring explanations.
# - Fetch status for each candidate (e.g., OA PDF found / not found),
#   including links to stored PDFs in Drive when available.
# - Updated records in the Notion literature database (upserted),
#   reflecting newly discovered or enriched papers.
# - Reusable artifacts (CSV / logs) for auditability and downstream use.
#
# Structure
# ----------------
# 1. Configuration and environment setup (RQ, year range, weights, I/O)
# 2. Loading the existing corpus and building deduplication indices
# 3. Selection of high-impact, high-RQ-alignment core papers
# 4. Computation of a lightweight RQ alignment score
#    (keyword / abstract / topic-based)
# 5. Backward expansion via citation links (references and optional cited-by)
# 6. Metadata enrichment and candidate deduplication
# 7. Priority scoring and ranking of backfill candidates
# 8. Open-access PDF resolution and download attempts
# 9. Persistence to Drive and upsert into the Notion database
# 10. Summary tables and reports for human review and decision-making
#
# Notes
# ----------------
# - This notebook focuses on "backward" expansion (toward older literature)
#   and complements forward-looking or daily discovery pipelines.
# - RQ alignment scoring is intentionally modular: a simple heuristic
#   implementation is provided here, with hooks for more advanced or
#   private models.
# - Identifier normalization (OpenAlex ID / DOI / arXiv ID) is critical
#   to avoid duplicate entries and ensure stable linkage across systems.
# - The final decision on which papers to keep or discard is expected to
#   involve lightweight human review, preserving researcher control.


In [10]:
# ============================================================
# 1. Configuration and environment setup (RQ, year range, weights, I/O)
# ============================================================

import os
import pathlib
import requests
from dotenv import load_dotenv

# ------------------------------------------------------------
# Small helper: require environment variable
# ------------------------------------------------------------
def require_env(key: str) -> str:
    val = os.getenv(key)
    if not val or not str(val).strip():
        raise ValueError(f"Missing required environment variable: {key}")
    return val

# ------------------------------------------------------------
# Load environment variables
# ------------------------------------------------------------
# Explicitly load env.txt (instead of default .env)
load_dotenv("env.txt")
print("🔧 Environment variables loaded from env.txt")


# ============================================================
# Research configuration (RQ, year range, scoring weights, I/O)
# ============================================================

# RQ configuration
# - You can hardcode RQ_TEXT for now, or load it later from NOTION_RQ_DB_ID.
RQ_TEXT = os.getenv("RQ_TEXT", "").strip()
if RQ_TEXT:
    print("✅ RQ_TEXT loaded from env.txt (optional).")
else:
    print("ℹ️ RQ_TEXT not provided in env.txt (you can set it later in the notebook).")

# Year range for backward filling candidates (adjust for your stage gate)
YEAR_MIN = int(os.getenv("YEAR_MIN", "1990"))
YEAR_MAX = int(os.getenv("YEAR_MAX", "2100"))  # "2100" effectively means no upper bound
print(f"📅 Year range configured: YEAR_MIN={YEAR_MIN}, YEAR_MAX={YEAR_MAX}")

# Core selection and expansion parameters
TOPK_CORE = int(os.getenv("TOPK_CORE", "30"))
MAX_HOPS = int(os.getenv("MAX_HOPS", "2"))
CANDIDATE_LIMIT_PER_CORE = int(os.getenv("CANDIDATE_LIMIT_PER_CORE", "200"))

# Composite score weights (tune these over time)
W_CITATION = float(os.getenv("W_CITATION", "0.55"))
W_RQ = float(os.getenv("W_RQ", "0.40"))
W_OA_BONUS = float(os.getenv("W_OA_BONUS", "0.05"))
W_RECENCY_PENALTY = float(os.getenv("W_RECENCY_PENALTY", "0.10"))  # penalty term (applied as subtraction)

# Execution flags
DRY_RUN = os.getenv("DRY_RUN", "true").lower() in ("1", "true", "yes", "y")
print(f"🧪 DRY_RUN={DRY_RUN}")

from datetime import datetime

# ------------------------------------------------------------
# Run timestamp (used for artifacts / logs isolation)
# ------------------------------------------------------------
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"🕒 RUN_TS = {RUN_TS}")

PROJECT_ROOT = pathlib.Path(os.getenv("PROJECT_ROOT", ".")).resolve()

# Shared cache (cross-run)
CACHE_DIR = PROJECT_ROOT / "cache"

# Run-scoped artifacts (day20/<timestamp>/)
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "day20" / RUN_TS

# Optional: run-scoped logs
LOG_DIR = PROJECT_ROOT / "logs" / "day20" / RUN_TS

for d in [CACHE_DIR, ARTIFACTS_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"📦 Cache dir:     {CACHE_DIR}")
print(f"🧾 Artifacts dir: {ARTIFACTS_DIR}")
print(f"🪵 Logs dir:      {LOG_DIR}")

# ============================================================
# Notion configuration
# ============================================================
# Notes for this project:
# - NOTION_LIT_DB_ID  : Literature / Papers DB (primary)
# - NOTION_RQ_DB_ID   : Research Questions DB (for RQ management)
# - NOTION_PAPERS_DB_ID does NOT exist in this setup

NOTION_TOKEN = require_env("NOTION_TOKEN")
NOTION_VERSION = require_env("NOTION_VERSION")  # e.g., "2022-06-28"

NOTION_LIT_DB_ID = require_env("NOTION_LIT_DB_ID")   # 문献一覧 (primary)
NOTION_RQ_DB_ID = require_env("NOTION_RQ_DB_ID")     # RQ一覧

NOTION_HEADERS = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Notion-Version": NOTION_VERSION,
    "Content-Type": "application/json",
}

print("✅ Notion config loaded:")
print(f"   - NOTION_LIT_DB_ID: {NOTION_LIT_DB_ID}")
print(f"   - NOTION_RQ_DB_ID:  {NOTION_RQ_DB_ID}")

# Quick auth check (safe: does not reveal token)
try:
    r = requests.get("https://api.notion.com/v1/users/me", headers=NOTION_HEADERS, timeout=30)
    if r.status_code == 200:
        print("✅ Notion auth OK")
    else:
        print("⚠️ Notion auth check failed:", r.status_code, r.text[:200])
except Exception as e:
    print("⚠️ Notion auth check error:", type(e).__name__, str(e))


# ============================================================
# Google Drive configuration
# ============================================================
# NOTE:
# For uploading PDFs, you need a scope that allows file creation.
# If you keep drive.readonly, uploads will fail.
DRIVE_SCOPES = ["https://www.googleapis.com/auth/drive"]

# OAuth client secret handling
# Priority:
# 1) GOOGLE_OAUTH_CLIENT_SECRET_JSON in env.txt (path)
# 2) Default local filename (checked in working directory)
DEFAULT_GOOGLE_CLIENT_SECRET = (
    "client_secret_750875982200-85rnsoqhr2af2b13peueev0bm60q22sh.apps.googleusercontent.com.json"
)

GOOGLE_OAUTH_CLIENT_SECRET_JSON = os.getenv("GOOGLE_OAUTH_CLIENT_SECRET_JSON")
client_secret_path = pathlib.Path(GOOGLE_OAUTH_CLIENT_SECRET_JSON) if GOOGLE_OAUTH_CLIENT_SECRET_JSON else pathlib.Path(DEFAULT_GOOGLE_CLIENT_SECRET)

if not client_secret_path.exists():
    raise ValueError(
        "Google OAuth client secret JSON not found.\n"
        "Either:\n"
        "  - set GOOGLE_OAUTH_CLIENT_SECRET_JSON in env.txt, or\n"
        f"  - place the file at: {DEFAULT_GOOGLE_CLIENT_SECRET}"
    )

print(f"✅ Google OAuth client secret located: {client_secret_path}")

# Token cache path (configurable via env.txt)
GOOGLE_TOKEN_JSON = os.getenv("GOOGLE_TOKEN_JSON", "google_token.json")
token_cache_path = pathlib.Path(GOOGLE_TOKEN_JSON)

# Destination Drive folder where PDFs will be saved
DRIVE_FOLDER_ID = require_env("DRIVE_FOLDER_ID")

print("✅ Drive config loaded:")
print(f"   - DRIVE_FOLDER_ID: {DRIVE_FOLDER_ID}")
print(f"   - Token cache:     {token_cache_path}")


# ============================================================
# OpenAlex configuration (metadata + citation graph)
# ============================================================
OPENALEX_BASE_URL = os.getenv("OPENALEX_BASE_URL", "https://api.openalex.org")
OPENALEX_MAILTO = os.getenv("OPENALEX_MAILTO", "").strip()  # recommended by OpenAlex for polite usage
OPENALEX_TIMEOUT = int(os.getenv("OPENALEX_TIMEOUT", "30"))

print("✅ OpenAlex config loaded:")
print(f"   - OPENALEX_BASE_URL: {OPENALEX_BASE_URL}")
print(f"   - OPENALEX_MAILTO:   {OPENALEX_MAILTO if OPENALEX_MAILTO else '(not set)'}")
print(f"   - OPENALEX_TIMEOUT:  {OPENALEX_TIMEOUT}s")


# ============================================================
# Optional: OpenAI key (only if used in later cells)
# ============================================================
# This notebook (020) can run without OpenAI, but keeping this block makes it
# easy to reuse summarization/refinement or RQ scoring upgrades downstream.
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY:
    print("🔑 OPENAI_API_KEY loaded successfully (optional).")
else:
    print("ℹ️ OPENAI_API_KEY not set (optional).")


🔧 Environment variables loaded from env.txt
ℹ️ RQ_TEXT not provided in env.txt (you can set it later in the notebook).
📅 Year range configured: YEAR_MIN=1990, YEAR_MAX=2100
🧪 DRY_RUN=True
🕒 RUN_TS = 20260113_061442
📦 Cache dir:     /Users/yuetoya/Desktop/researchOS100-private/notebooks/cache
🧾 Artifacts dir: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442
🪵 Logs dir:      /Users/yuetoya/Desktop/researchOS100-private/notebooks/logs/day20/20260113_061442
✅ Notion config loaded:
   - NOTION_LIT_DB_ID: 2a98e0e4d16280cbb6cbdcd1ebedee54
   - NOTION_RQ_DB_ID:  2a98e0e4d16280b7bad2cf6635a3ef17
✅ Notion auth OK
✅ Google OAuth client secret located: client_secret_750875982200-85rnsoqhr2af2b13peueev0bm60q22sh.apps.googleusercontent.com.json
✅ Drive config loaded:
   - DRIVE_FOLDER_ID: 1SygzpVjCuk-_8oHk9XQOponn7T3ZOsgh
   - Token cache:     google_token.json
✅ OpenAlex config loaded:
   - OPENALEX_BASE_URL: https://api.openalex.org
   - OPENALEX_MAILTO:   (no

In [8]:
# ============================================================
# 2. Loading the existing corpus and building deduplication indices
# ============================================================

import json
import re
import time
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd


# ------------------------------------------------------------
# Utilities: normalization + stable keys
# ------------------------------------------------------------

def normalize_whitespace(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

def normalize_doi(doi: Optional[str]) -> Optional[str]:
    if not doi:
        return None
    doi = normalize_whitespace(doi).lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "").replace("doi:", "")
    doi = doi.strip()
    return doi or None

def normalize_openalex_id(openalex_id: Optional[str]) -> Optional[str]:
    """
    Accept either full URL form (https://openalex.org/Wxxxx) or Wxxxx.
    Return canonical URL form for consistent dedupe.
    """
    if not openalex_id:
        return None
    s = normalize_whitespace(openalex_id)
    if s.startswith("http"):
        return s
    if re.match(r"^W\d+$", s):
        return f"https://openalex.org/{s}"
    return s

def normalize_arxiv_id(arxiv_id: Optional[str]) -> Optional[str]:
    """
    Normalize common arXiv formats:
      - 'arXiv:2101.01234v2' -> '2101.01234'
      - '2101.01234' -> '2101.01234'
    """
    if not arxiv_id:
        return None
    s = normalize_whitespace(arxiv_id)
    s = s.replace("arXiv:", "").strip()
    # drop version suffix like v2
    s = re.sub(r"v\d+$", "", s)
    return s or None

def title_key(title: Optional[str]) -> Optional[str]:
    """
    Aggressive title key for fuzzy-ish dedupe when IDs are missing.
    - lowercase
    - strip punctuation
    - collapse spaces
    """
    if not title:
        return None
    s = normalize_whitespace(title).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = normalize_whitespace(s)
    return s or None


# ------------------------------------------------------------
# Notion DB fetch helpers (minimal, safe defaults)
# ------------------------------------------------------------

def notion_query_database_all(database_id: str, headers: Dict[str, str], page_size: int = 100) -> List[Dict[str, Any]]:
    """
    Pull all pages from a Notion database via pagination.
    Returns raw Notion page objects.
    """
    url = f"https://api.notion.com/v1/databases/{database_id}/query"
    payload = {"page_size": page_size}
    results: List[Dict[str, Any]] = []
    next_cursor = None

    while True:
        if next_cursor:
            payload["start_cursor"] = next_cursor

        r = requests.post(url, headers=headers, json=payload, timeout=30)
        if r.status_code != 200:
            raise RuntimeError(f"Notion query failed ({r.status_code}): {r.text[:500]}")

        data = r.json()
        results.extend(data.get("results", []))
        next_cursor = data.get("next_cursor")

        if not data.get("has_more"):
            break

        time.sleep(0.35)  # gentle pacing

    return results


def _get_rich_text(prop: Dict[str, Any]) -> str:
    if not prop:
        return ""
    rich = prop.get("rich_text") or prop.get("title") or []
    texts = [x.get("plain_text", "") for x in rich]
    return normalize_whitespace("".join(texts))

def _get_select(prop: Dict[str, Any]) -> str:
    if not prop:
        return ""
    sel = prop.get("select")
    return (sel or {}).get("name", "") if isinstance(sel, dict) else ""

def _get_number(prop: Dict[str, Any]) -> Optional[float]:
    if not prop:
        return None
    return prop.get("number")

def _get_url(prop: Dict[str, Any]) -> str:
    if not prop:
        return ""
    return prop.get("url") or ""


def extract_paper_fields_from_notion_page(page: Dict[str, Any]) -> Dict[str, Any]:
    """
    Map your Notion Literature DB schema into a flat dict.
    Adjust property names here to match your actual Notion DB fields.
    """
    props = page.get("properties", {})

    # ---- IMPORTANT ----
    # Replace these keys with your exact Notion property names.
    # Common examples:
    #   Title: "Name" or "Title"
    #   Year: "Year"
    #   DOI: "DOI"
    #   OpenAlex: "OpenAlex ID" or "openalex_id"
    #   arXiv: "arXiv ID"
    #   Cited-by count: "Cited By"
    #
    title = _get_rich_text(props.get("Title") or props.get("Name") or {})
    year = _get_number(props.get("Year") or {})  # might be float
    doi = _get_rich_text(props.get("DOI") or {})
    openalex_id = _get_rich_text(props.get("OpenAlex ID") or props.get("openalex_id") or {})
    arxiv_id = _get_rich_text(props.get("arXiv ID") or props.get("arxiv_id") or {})
    cited_by = _get_number(props.get("Cited By") or props.get("cited_by_count") or {})
    abstract = _get_rich_text(props.get("Abstract") or props.get("abstract") or {})

    return {
        "notion_page_id": page.get("id"),
        "title": title,
        "year": int(year) if year is not None else None,
        "doi": normalize_doi(doi),
        "openalex_id": normalize_openalex_id(openalex_id),
        "arxiv_id": normalize_arxiv_id(arxiv_id),
        "cited_by_count": int(cited_by) if cited_by is not None else None,
        "abstract": abstract,
    }


# ------------------------------------------------------------
# Load existing corpus from Notion (primary source)
# ------------------------------------------------------------

print("📥 Loading existing corpus from Notion Literature DB...")

raw_pages = notion_query_database_all(
    database_id=NOTION_LIT_DB_ID,
    headers=NOTION_HEADERS,
    page_size=100,
)

print(f"✅ Pulled {len(raw_pages):,} pages from Notion.")

records = [extract_paper_fields_from_notion_page(p) for p in raw_pages]
corpus_df = pd.DataFrame.from_records(records)

# ============================================================
# Enrich cited_by_count from OpenAlex (fallback when Notion lacks it)
# ============================================================

import requests
import time
import json
from urllib.parse import quote
import pandas as pd


# -----------------------------
# OpenAlex request helper
# -----------------------------
SESSION = requests.Session()

def openalex_get(url: str, params: dict | None = None, timeout: int = 30, max_retries: int = 5):
    """
    Small robust GET with backoff.
    """
    params = params or {}
    if OPENALEX_MAILTO:
        params["mailto"] = OPENALEX_MAILTO

    last_err = None
    for i in range(max_retries):
        try:
            r = SESSION.get(url, params=params, timeout=timeout)
            if r.status_code == 429:
                # rate limited
                time.sleep(1.5 + 0.8 * i)
                continue
            if r.status_code >= 400:
                last_err = RuntimeError(f"OpenAlex error {r.status_code}: {r.text[:200]}")
                time.sleep(0.8 + 0.4 * i)
                continue
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(0.8 + 0.4 * i)

    raise RuntimeError(f"OpenAlex GET failed after retries. Last error: {type(last_err).__name__}: {last_err}")


# -----------------------------
# Resolve OpenAlex work by ID/DOI/title (in that order)
# -----------------------------
def resolve_work_from_openalex(openalex_id: str | None, doi: str | None, title: str | None) -> dict | None:
    """
    Returns OpenAlex work JSON if resolved, else None.
    Priority:
      1) OpenAlex Work ID (best)
      2) DOI (stable)
      3) Title search (fallback; may mismatch)
    """
    # 1) OpenAlex ID direct
    if openalex_id and str(openalex_id).strip():
        oa = normalize_openalex_id(openalex_id)
        # oa might already be a full URL
        work_url = oa if oa.startswith("http") else f"{OPENALEX_BASE_URL}/works/{oa}"
        try:
            return openalex_get(work_url, timeout=OPENALEX_TIMEOUT)
        except Exception:
            pass

    # 2) DOI filter
    if doi and str(doi).strip():
        d = normalize_doi(doi)
        if d:
            url = f"{OPENALEX_BASE_URL}/works"
            params = {"filter": f"doi:{d}", "per-page": 1}
            data = openalex_get(url, params=params, timeout=OPENALEX_TIMEOUT)
            results = data.get("results", [])
            if results:
                return results[0]

    # 3) Title search (fallback)
    if title and str(title).strip():
        # Use search endpoint; keep it conservative
        url = f"{OPENALEX_BASE_URL}/works"
        params = {"search": title, "per-page": 5}
        data = openalex_get(url, params=params, timeout=OPENALEX_TIMEOUT)
        results = data.get("results", [])
        if not results:
            return None

        # Pick best by naive title similarity key match (cheap + deterministic)
        tkey = title_key(title)
        for w in results:
            if title_key(w.get("title")) == tkey:
                return w

        # Otherwise: just return top result (risky; mark as low-confidence)
        return results[0]

    return None


# -----------------------------
# Main enrichment function
# -----------------------------
def enrich_cited_by_count(
    df: pd.DataFrame,
    cache_path: str | None = None,
    force: bool = False,
) -> pd.DataFrame:
    """
    Adds/updates:
      - cited_by_count (int)
      - cited_by_source (openalex_id|doi|title|none)
      - cited_by_confidence (high|medium|low)
    Uses a local cache to avoid repeated API calls.
    """
    out = df.copy()

    # Prepare cache
    cache = {}
    if cache_path:
        cache_file = pathlib.Path(cache_path)
        if cache_file.exists():
            try:
                cache = json.loads(cache_file.read_text())
                print(f"✅ Loaded OpenAlex cache: {cache_file} ({len(cache):,} items)")
            except Exception:
                print("⚠️ Failed to load cache; starting fresh.")

    def cache_key(row) -> str:
        # Prefer stable IDs for cache keying
        if row.get("openalex_norm"):
            return f"oa:{row['openalex_norm']}"
        if row.get("doi_norm"):
            return f"doi:{row['doi_norm']}"
        if row.get("title_norm"):
            return f"title:{row['title_norm']}"
        return f"row:{row.get('notion_page_id','unknown')}"

    filled = 0
    skipped = 0

    cited_by_counts = []
    sources = []
    confs = []

    for _, r in out.iterrows():
        current = r.get("cited_by_count", None)

        # If already present (and not NaN) and not forcing, keep it
        if (current is not None) and (pd.notna(current)) and (not force):
            cited_by_counts.append(int(current))
            sources.append("existing")
            confs.append("high")
            skipped += 1
            continue

        key = cache_key(r)

        if key in cache and not force:
            cached = cache[key]
            cited_by_counts.append(int(cached.get("cited_by_count", 0)))
            sources.append(cached.get("source", "cache"))
            confs.append(cached.get("confidence", "high"))
            filled += 1
            continue

        # Resolve via OpenAlex
        work = resolve_work_from_openalex(
            openalex_id=r.get("openalex_id"),
            doi=r.get("doi"),
            title=r.get("title"),
        )

        if work:
            cbc = work.get("cited_by_count", 0) or 0

            # determine source/confidence
            if r.get("openalex_id") and str(r.get("openalex_id")).strip():
                source = "openalex_id"
                conf = "high"
            elif r.get("doi") and str(r.get("doi")).strip():
                source = "doi"
                conf = "high"
            else:
                source = "title"
                conf = "low"  # title search is less reliable

            cited_by_counts.append(int(cbc))
            sources.append(source)
            confs.append(conf)

            cache[key] = {"cited_by_count": int(cbc), "source": source, "confidence": conf}
            filled += 1
        else:
            cited_by_counts.append(0)
            sources.append("none")
            confs.append("low")
            cache[key] = {"cited_by_count": 0, "source": "none", "confidence": "low"}

        # gentle pacing
        time.sleep(0.12)

    out["cited_by_count"] = cited_by_counts
    out["cited_by_source"] = sources
    out["cited_by_confidence"] = confs

    if cache_path:
        cache_file = pathlib.Path(cache_path)
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        cache_file.write_text(json.dumps(cache, ensure_ascii=False, indent=2))
        print(f"💾 Saved OpenAlex cache: {cache_file} ({len(cache):,} items)")

    print(f"✅ Enrichment done: filled={filled:,}, skipped(existing)={skipped:,}")
    return out


# ------------------------------------------------------------
# Run enrichment (recommended right after loading corpus_df)
# ------------------------------------------------------------
CACHE_PATH = str(CACHE_DIR / "openalex_cited_by_cache.json")

corpus_df["cited_by_count"] = pd.to_numeric(corpus_df.get("cited_by_count", None), errors="coerce")
corpus_df = enrich_cited_by_count(corpus_df, cache_path=CACHE_PATH, force=False)

display(corpus_df[["title", "doi", "openalex_id", "cited_by_count", "cited_by_source", "cited_by_confidence"]].head(10))


# Basic cleanup
corpus_df["title_norm"] = corpus_df["title"].apply(title_key)
corpus_df["doi_norm"] = corpus_df["doi"].apply(normalize_doi)
corpus_df["openalex_norm"] = corpus_df["openalex_id"].apply(normalize_openalex_id)
corpus_df["arxiv_norm"] = corpus_df["arxiv_id"].apply(normalize_arxiv_id)

print("✅ Corpus dataframe created.")
display(corpus_df.head(5))


# ------------------------------------------------------------
# Build deduplication indices (fast membership checks)
# ------------------------------------------------------------

existing_openalex_ids = set(corpus_df["openalex_norm"].dropna().astype(str).tolist())
existing_dois = set(corpus_df["doi_norm"].dropna().astype(str).tolist())
existing_arxiv_ids = set(corpus_df["arxiv_norm"].dropna().astype(str).tolist())
existing_title_keys = set(corpus_df["title_norm"].dropna().astype(str).tolist())

print("🧩 Deduplication indices built:")
print(f"   - OpenAlex IDs: {len(existing_openalex_ids):,}")
print(f"   - DOIs:        {len(existing_dois):,}")
print(f"   - arXiv IDs:   {len(existing_arxiv_ids):,}")
print(f"   - Title keys:  {len(existing_title_keys):,}")


# ------------------------------------------------------------
# Optional: quick corpus health checks
# ------------------------------------------------------------

def count_duplicates(series: pd.Series) -> int:
    s = series.dropna().astype(str)
    return int(s.duplicated().sum())

dup_openalex = count_duplicates(corpus_df["openalex_norm"])
dup_doi = count_duplicates(corpus_df["doi_norm"])
dup_arxiv = count_duplicates(corpus_df["arxiv_norm"])
dup_title = count_duplicates(corpus_df["title_norm"])

print("🔎 Duplicate signals in the current corpus (non-fatal):")
print(f"   - duplicated OpenAlex IDs: {dup_openalex}")
print(f"   - duplicated DOIs:         {dup_doi}")
print(f"   - duplicated arXiv IDs:    {dup_arxiv}")
print(f"   - duplicated title keys:   {dup_title}")

# If duplicates are high, you may want to inspect them:
# display(corpus_df[corpus_df["doi_norm"].duplicated(keep=False)].sort_values("doi_norm").head(50))

# ------------------------------------------------------------
# Helper: decide if a candidate paper is already in the corpus
# ------------------------------------------------------------

def is_already_in_corpus(
    openalex_id: Optional[str] = None,
    doi: Optional[str] = None,
    arxiv_id: Optional[str] = None,
    title: Optional[str] = None,
) -> bool:
    """
    Returns True if the paper matches an existing record by any strong identifier.
    Falls back to a normalized title key when IDs are missing.
    """
    oa = normalize_openalex_id(openalex_id)
    if oa and oa in existing_openalex_ids:
        return True

    d = normalize_doi(doi)
    if d and d in existing_dois:
        return True

    ax = normalize_arxiv_id(arxiv_id)
    if ax and ax in existing_arxiv_ids:
        return True

    tk = title_key(title)
    if tk and tk in existing_title_keys:
        return True

    return False

print("✅ is_already_in_corpus() is ready.")



📥 Loading existing corpus from Notion Literature DB...
✅ Pulled 50 pages from Notion.
💾 Saved OpenAlex cache: /Users/yuetoya/Desktop/researchOS100-private/notebooks/cache/openalex_cited_by_cache.json (50 items)
✅ Enrichment done: filled=50, skipped(existing)=0


,title,doi,openalex_id,cited_by_count,cited_by_source,cited_by_confidence
0,Venture Capital Networks and Cross-Border Star...,None,None,2693,title,low
1,From Startup to Scaleup: Public Policies for E...,None,None,4,title,low
2,"Formal institutions, culture, and venture capi...",None,None,443,title,low
3,"On Resource Complementarity Among Startups, Ac...",None,None,2,title,low
4,Liberalizing Home-Based Business,None,None,2,title,low
5,The Effects of Government-Sponsored Venture Ca...,None,None,297,title,low
6,Entrepreneurial ecosystems and growth oriented...,None,None,953,title,low
7,Institutional differences and the development ...,None,None,215,title,low
8,"Venture capital, entrepreneurship and economic...",None,None,647,title,low
9,The Government as Venture Capitalist: The Long...,None,None,1134,title,low


✅ Corpus dataframe created.


,notion_page_id,title,year,doi,openalex_id,arxiv_id,cited_by_count,abstract,cited_by_source,cited_by_confidence,title_norm,doi_norm,openalex_norm,arxiv_norm
0,2132b6ad-fd39-4f9f-a433-45a2fbdbd855,Venture Capital Networks and Cross-Border Star...,None,None,None,None,2693,,title,low,venture capital networks and cross border star...,None,None,None
1,244b98a4-43d0-4fa3-bf7b-4955665da00e,From Startup to Scaleup: Public Policies for E...,None,None,None,None,4,,title,low,from startup to scaleup public policies for em...,None,None,None
2,2a98e0e4-d162-810b-9aa3-d575608e429e,"Formal institutions, culture, and venture capi...",None,None,None,None,443,,title,low,formal institutions culture and venture capita...,None,None,None
3,2a98e0e4-d162-810e-a210-d558c3e44495,"On Resource Complementarity Among Startups, Ac...",None,None,None,None,2,,title,low,on resource complementarity among startups acc...,None,None,None
4,2a98e0e4-d162-8117-81d3-c0a8f0c17d89,Liberalizing Home-Based Business,None,None,None,None,2,,title,low,liberalizing home based business,None,None,None


🧩 Deduplication indices built:
   - OpenAlex IDs: 0
   - DOIs:        0
   - arXiv IDs:   0
   - Title keys:  50
🔎 Duplicate signals in the current corpus (non-fatal):
   - duplicated OpenAlex IDs: 0
   - duplicated DOIs:         0
   - duplicated arXiv IDs:    0
   - duplicated title keys:   0
✅ is_already_in_corpus() is ready.


In [11]:
# ============================================================
# 3. Selection of high-impact, high-RQ-alignment core papers
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Assumptions / inputs available at this point
# ------------------------------------------------------------
# - corpus_df: dataframe of existing papers loaded from Notion
# - YEAR_MIN / YEAR_MAX: year range config
# - TOPK_CORE: number of core papers to select
# - RQ_TEXT: optional; if empty, we fall back to keyword-only scoring
#
# Next cells will define a better compute_rq_score().
# For now, we use a lightweight heuristic so this cell can run end-to-end.
# ------------------------------------------------------------


# ------------------------------------------------------------
# Lightweight RQ alignment score (temporary heuristic)
# - Replace later with your "real" model in Section 4.
# ------------------------------------------------------------
DEFAULT_RQ_KEYWORDS = [
    # TODO: customize for your domain; keep this list short and editable
    "venture capital", "vc", "limited partner", "lp",
    "entrepreneurship", "startup", "innovation policy",
    "government venture", "public venture", "industrial policy",
]

def compute_rq_score_lightweight(title: str, abstract: str, rq_text: str = "") -> float:
    """
    Minimal RQ alignment score:
    - If rq_text is provided, we treat its tokens as soft keywords.
    - Otherwise, use DEFAULT_RQ_KEYWORDS.
    Score is a simple normalized hit-rate over title+abstract.
    """
    blob = f"{title or ''} {abstract or ''}".lower()

    if rq_text and rq_text.strip():
        # naive tokenization from rq_text
        toks = [t.strip().lower() for t in re.split(r"[\n,;]+", rq_text) if t.strip()]
        keywords = toks[:25] if toks else DEFAULT_RQ_KEYWORDS
    else:
        keywords = DEFAULT_RQ_KEYWORDS

    hits = sum(1 for kw in keywords if kw.lower() in blob)
    score = hits / max(1, len(keywords))
    return float(score)


# ------------------------------------------------------------
# Prepare candidate pool from the existing corpus
# ------------------------------------------------------------

work_df = corpus_df.copy()

# Ensure numeric fields
work_df["year"] = pd.to_numeric(work_df["year"], errors="coerce")
work_df["cited_by_count"] = pd.to_numeric(work_df["cited_by_count"], errors="coerce").fillna(0)

# Filter by year range (optional)
work_df = work_df[(work_df["year"].isna()) | ((work_df["year"] >= YEAR_MIN) & (work_df["year"] <= YEAR_MAX))].copy()
print(f"📚 Candidate pool after year filter: {len(work_df):,} papers")

# Compute RQ score (lightweight)
work_df["rq_score"] = work_df.apply(
    lambda r: compute_rq_score_lightweight(r.get("title", ""), r.get("abstract", ""), RQ_TEXT),
    axis=1
)

# Normalize citation counts (robust to skew)
# Use log1p so very large citation counts do not dominate.
work_df["citation_log"] = np.log1p(work_df["cited_by_count"].astype(float))

# Min-max normalize (avoid division by zero)
cmin, cmax = work_df["citation_log"].min(), work_df["citation_log"].max()
if cmax > cmin:
    work_df["citation_norm"] = (work_df["citation_log"] - cmin) / (cmax - cmin)
else:
    work_df["citation_norm"] = 0.0

# Clamp rq_score to [0,1] just in case
work_df["rq_score"] = work_df["rq_score"].clip(0.0, 1.0)

# Core score combines impact + alignment
# Note: weights here are just for core selection; you can reuse global weights if you prefer.
CORE_W_CITATION = float(os.getenv("CORE_W_CITATION", "0.6"))
CORE_W_RQ = float(os.getenv("CORE_W_RQ", "0.4"))

work_df["core_score"] = CORE_W_CITATION * work_df["citation_norm"] + CORE_W_RQ * work_df["rq_score"]

# Explainability columns
work_df["core_score_explain"] = work_df.apply(
    lambda r: f"core={r['core_score']:.3f} (cite_norm={r['citation_norm']:.3f}, rq={r['rq_score']:.3f})",
    axis=1
)

# Sort and select top-K
core_df = (
    work_df.sort_values(["core_score", "cited_by_count"], ascending=[False, False])
           .head(TOPK_CORE)
           .copy()
)

core_df["is_core"] = True

print(f"✅ Selected {len(core_df):,} core papers (TOPK_CORE={TOPK_CORE}).")
display(core_df[[
    "title", "year", "cited_by_count", "rq_score", "core_score", "core_score_explain",
    "doi", "openalex_id", "arxiv_id", "notion_page_id"
]].head(20))


# ------------------------------------------------------------
# Persist core selection artifacts (optional but recommended)
# ------------------------------------------------------------
core_out_path = ARTIFACTS_DIR / "day20_core_papers.csv"
core_df.to_csv(core_out_path, index=False)
print(f"💾 Saved core papers: {core_out_path}")

# ------------------------------------------------------------
# Optional: sanity checks / diagnostics
# ------------------------------------------------------------

print("📊 Core paper diagnostics")
print(f" - year range in core: {core_df['year'].min()} .. {core_df['year'].max()}")
print(f" - median cited_by_count: {int(core_df['cited_by_count'].median())}")
print(f" - median rq_score: {core_df['rq_score'].median():.3f}")

# If you want to quickly see low-RQ but high-citation items:
low_rq = core_df.sort_values("rq_score", ascending=True).head(10)
print("\n🔻 Lowest RQ-score items among the core selection:")
display(low_rq[["title", "year", "cited_by_count", "rq_score", "core_score"]])


📚 Candidate pool after year filter: 50 papers
✅ Selected 30 core papers (TOPK_CORE=30).


,title,year,cited_by_count,rq_score,core_score,core_score_explain,doi,openalex_id,arxiv_id,notion_page_id
0,Venture Capital Networks and Cross-Border Star...,NaN,2693,0.2,0.680000,"core=0.680 (cite_norm=1.000, rq=0.200)",None,None,None,2132b6ad-fd39-4f9f-a433-45a2fbdbd855
38,Seeding the Way: Governments as Limited Partne...,NaN,2356,0.2,0.669849,"core=0.670 (cite_norm=0.983, rq=0.200)",None,None,None,4a5fe6a2-d640-4218-84c8-61260b7955c4
45,Spatial heterogeneity in venture capital and n...,NaN,1641,0.1,0.602391,"core=0.602 (cite_norm=0.937, rq=0.100)",None,None,None,da255a45-0fe2-42a0-bf86-ee9af1097021
19,The Barriers to Embedding Entrepreneurship Edu...,NaN,1180,0.1,0.577358,"core=0.577 (cite_norm=0.896, rq=0.100)",None,None,None,2dc8e0e4-d162-8196-9fe5-c97ef51ef4a4
9,The Government as Venture Capitalist: The Long...,NaN,1134,0.1,0.574340,"core=0.574 (cite_norm=0.891, rq=0.100)",None,None,None,2a98e0e4-d162-81af-bf07-db4467b2c888
8,"Venture capital, entrepreneurship and economic...",NaN,647,0.2,0.571764,"core=0.572 (cite_norm=0.820, rq=0.200)",None,None,None,2a98e0e4-d162-8194-a50e-d3efd1ed5d9f
6,Entrepreneurial ecosystems and growth oriented...,NaN,953,0.1,0.561143,"core=0.561 (cite_norm=0.869, rq=0.100)",None,None,None,2a98e0e4-d162-8174-8b95-cf725a4fca42
12,The determinants of venture capital funding: e...,NaN,932,0.1,0.559453,"core=0.559 (cite_norm=0.866, rq=0.100)",None,None,None,2a98e0e4-d162-81e2-8131-e269353c4dc3
13,"Corporate Venture Capital, Value Creation, and...",NaN,653,0.1,0.532464,"core=0.532 (cite_norm=0.821, rq=0.100)",None,None,None,2a98e0e4-d162-81e4-b518-d74871550e8b
2,"Formal institutions, culture, and venture capi...",NaN,443,0.1,0.503045,"core=0.503 (cite_norm=0.772, rq=0.100)",None,None,None,2a98e0e4-d162-810b-9aa3-d575608e429e


💾 Saved core papers: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/day20_core_papers.csv
📊 Core paper diagnostics
 - year range in core: nan .. nan
 - median cited_by_count: 253
 - median rq_score: 0.100

🔻 Lowest RQ-score items among the core selection:


,title,year,cited_by_count,rq_score,core_score
27,The Role of Policy and Regulation in Promoting...,NaN,34,0.0,0.270068
29,Symbiotic synergy: How Arbuscular Mycorrhizal ...,NaN,38,0.0,0.278288
32,"Climate change, ESG criteria and recent regula...",NaN,52,0.0,0.301588
21,"Mindfulness, self-efficacy, and self-regulatio...",NaN,58,0.0,0.309734
30,Frontier AI Regulation: Managing Emerging Risk...,NaN,68,0.0,0.321627
35,Managing the Race to the Moon: Global Policy a...,NaN,87,0.0,0.340103
14,Institutional Influences on the Worldwide Expa...,NaN,291,0.1,0.471212
23,The evolution of k-shell in syndication networ...,NaN,7,0.1,0.197957
34,The impact and effectiveness of China’s entrep...,NaN,8,0.1,0.206904
15,Demand pull versus resource push training appr...,NaN,10,0.1,0.222147


In [14]:
# ============================================================
# 4. Computation of a lightweight RQ alignment score
#   - Load High-priority RQs from NOTION_RQ_DB_ID (Japanese)
#   - Translate them to English
#   - Convert to a lightweight keyword set
#   - Score each paper by keyword hits in title/abstract (+ optional OpenAlex concepts)
# ============================================================

import re
import time
import json
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd


# ------------------------------------------------------------
# 4.1 Notion helpers for the RQ database
# ------------------------------------------------------------

def notion_query_database_all_with_filter(
    database_id: str,
    headers: Dict[str, str],
    page_size: int = 100,
    notion_filter: Optional[Dict[str, Any]] = None,
) -> List[Dict[str, Any]]:
    """
    Query all pages from a Notion database with an optional filter.
    Returns raw Notion page objects.
    """
    url = f"https://api.notion.com/v1/databases/{database_id}/query"
    results: List[Dict[str, Any]] = []
    next_cursor = None

    while True:
        payload: Dict[str, Any] = {"page_size": page_size}
        if notion_filter:
            payload["filter"] = notion_filter
        if next_cursor:
            payload["start_cursor"] = next_cursor

        r = requests.post(url, headers=headers, json=payload, timeout=30)
        if r.status_code != 200:
            raise RuntimeError(f"Notion query failed ({r.status_code}): {r.text[:500]}")

        data = r.json()
        results.extend(data.get("results", []))
        next_cursor = data.get("next_cursor")

        if not data.get("has_more"):
            break

        time.sleep(0.35)

    return results


def extract_rq_from_notion_page(page: Dict[str, Any]) -> Dict[str, Any]:
    """
    Expected properties in NOTION_RQ_DB:
      - Name: Japanese RQ text (title property)
      - Priority: select property with values like High / Medium / Low
    Adjust property names here if your DB differs.
    """
    props = page.get("properties", {})

    rq_jp = _get_rich_text(props.get("Name") or props.get("Title") or {})
    priority = _get_select(props.get("Priority") or {})

    return {
        "notion_page_id": page.get("id"),
        "rq_jp": rq_jp,
        "priority": priority,
    }


# ------------------------------------------------------------
# 4.2 Load High-priority RQs (Japanese) from NOTION_RQ_DB_ID
# ------------------------------------------------------------

print("📥 Loading High-priority RQs from Notion (NOTION_RQ_DB_ID)...")

high_filter = {
    "property": "Priority",
    "select": {"equals": "High"}
}

rq_pages = notion_query_database_all_with_filter(
    database_id=NOTION_RQ_DB_ID,
    headers=NOTION_HEADERS,
    page_size=100,
    notion_filter=high_filter,
)

rq_records = [extract_rq_from_notion_page(p) for p in rq_pages]
rq_df = pd.DataFrame(rq_records)

rq_df = rq_df.dropna(subset=["rq_jp"])
rq_df["rq_jp"] = rq_df["rq_jp"].astype(str).str.strip()
rq_df = rq_df[rq_df["rq_jp"] != ""]

print(f"✅ Loaded {len(rq_df):,} High-priority RQs.")
display(rq_df.head(20))


# ------------------------------------------------------------
# 4.3 Translate JP RQs -> EN (OpenAI optional)
# ------------------------------------------------------------
# If OPENAI_API_KEY is not set, we fall back to a very naive approach:
# use the JP text as-is as "keywords" (works poorly for English papers).
#
# Recommended: set OPENAI_API_KEY and run translation once per RQ,
# then cache results to ARTIFACTS_DIR for reproducibility.

TRANSLATION_CACHE_PATH = ARTIFACTS_DIR / "rq_translation_cache.json"

def load_translation_cache(path: pathlib.Path) -> Dict[str, str]:
    if path.exists():
        try:
            return json.loads(path.read_text())
        except Exception:
            return {}
    return {}

def save_translation_cache(path: pathlib.Path, cache: Dict[str, str]) -> None:
    path.write_text(json.dumps(cache, ensure_ascii=False, indent=2))

translation_cache = load_translation_cache(TRANSLATION_CACHE_PATH)
print(f"🗂️ Translation cache loaded: {len(translation_cache):,} items")


def translate_jp_to_en_openai(text_jp: str) -> str:
    """
    Translate Japanese research-question text into concise English.
    Uses OpenAI if available; otherwise returns the original Japanese string.
    """
    if not OPENAI_API_KEY:
        return text_jp

    # Lazy import so this cell doesn't break if openai isn't installed
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    prompt = (
        "Translate the following Japanese research question into concise academic English. "
        "Return ONLY the English translation.\n\n"
        f"Japanese:\n{text_jp}\n"
    )

    resp = client.chat.completions.create(
        model=os.getenv("OPENAI_TRANSLATION_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a precise research assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
    )

    return resp.choices[0].message.content.strip()


rq_en_list = []
for rq_jp in rq_df["rq_jp"].tolist():
    if rq_jp in translation_cache:
        rq_en = translation_cache[rq_jp]
    else:
        rq_en = translate_jp_to_en_openai(rq_jp)
        translation_cache[rq_jp] = rq_en
        # gentle pacing if API is used
        time.sleep(0.2)

    rq_en_list.append(rq_en)

rq_df["rq_en"] = rq_en_list
save_translation_cache(TRANSLATION_CACHE_PATH, translation_cache)

print("✅ JP->EN translation completed (cached).")
display(rq_df[["priority", "rq_jp", "rq_en"]].head(20))


# ------------------------------------------------------------
# 4.4 Build high-quality keyword/phrase sets using ChatGPT
# ------------------------------------------------------------

import json
import time
from typing import List, Dict

RQ_TERM_CACHE_PATH = ARTIFACTS_DIR / "rq_terms_cache.json"

def load_json_cache(path: pathlib.Path) -> dict:
    if path.exists():
        try:
            return json.loads(path.read_text())
        except Exception:
            return {}
    return {}

def save_json_cache(path: pathlib.Path, obj: dict) -> None:
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2))

rq_terms_cache = load_json_cache(RQ_TERM_CACHE_PATH)
print(f"🗂️ RQ terms cache loaded: {len(rq_terms_cache):,} items")

def generate_terms_with_chatgpt(rq_en: str) -> Dict[str, List[str]]:
    """
    Returns a dict like:
      {
        "phrases": ["government venture capital", "public LP program", ...],
        "keywords": ["gvc", "lp", "swf", ...],
        "exclude": ["most", "effective", ...]  # optional
      }
    """
    if not OPENAI_API_KEY:
        # Fallback: keep the RQ sentence itself as a "phrase"
        return {"phrases": [rq_en], "keywords": [], "exclude": []}

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    system = (
        "You are a research assistant designing search terms for academic literature retrieval. "
        "Focus on domain-specific terminology, not generic words."
    )

    user = f"""
Given this research question in English:

RQ: {rq_en}

Generate a compact set of search terms for finding relevant academic papers.

Requirements:
- Output MUST be valid JSON (no markdown).
- Provide:
  1) "phrases": 8–15 multi-word phrases (2–6 words) that capture the core concepts.
  2) "keywords": 10–25 single-word terms, including common abbreviations (e.g., VC, LP, SWF, CVC, GVC) and synonyms.
  3) "exclude": up to 10 generic terms to avoid (e.g., "most", "effective") if they appear.
- Avoid overly generic words (e.g., "public", "effective", "role", "system") unless paired into a specific phrase.
- Prefer academically used phrasing (e.g., "government venture capital", "limited partner program", "sovereign wealth fund", "innovation hub", "entrepreneurial ecosystem").
"""

    resp = client.chat.completions.create(
        model=os.getenv("OPENAI_TERMS_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.2,
    )

    raw = resp.choices[0].message.content.strip()
    return json.loads(raw)

def normalize_term_list(xs: List[str]) -> List[str]:
    out = []
    seen = set()
    for x in xs or []:
        t = re.sub(r"\s+", " ", str(x).strip().lower())
        t = t.strip(" ,;.")
        if not t:
            continue
        if t not in seen:
            out.append(t)
            seen.add(t)
    return out

# Generate / cache terms per RQ
rq_df["phrases"] = None
rq_df["keywords"] = None
rq_df["exclude"] = None

for i, row in rq_df.iterrows():
    rq_en = row["rq_en"]
    if rq_en in rq_terms_cache:
        terms = rq_terms_cache[rq_en]
    else:
        terms = generate_terms_with_chatgpt(rq_en)
        rq_terms_cache[rq_en] = terms
        time.sleep(0.2)

    phrases = normalize_term_list(terms.get("phrases", []))
    keywords = normalize_term_list(terms.get("keywords", []))
    exclude = normalize_term_list(terms.get("exclude", []))

    rq_df.at[i, "phrases"] = phrases
    rq_df.at[i, "keywords"] = keywords
    rq_df.at[i, "exclude"] = exclude

save_json_cache(RQ_TERM_CACHE_PATH, rq_terms_cache)
print(f"💾 Saved RQ terms cache: {RQ_TERM_CACHE_PATH}")

display(rq_df[["rq_en", "phrases", "keywords"]].head(10))


# ------------------------------------------------------------
# 4.5 Build global RQ term sets (phrases + keywords)
# ------------------------------------------------------------

ALL_PHRASES = []
ALL_KEYWORDS = []
ALL_EXCLUDE = []

for _, row in rq_df.iterrows():
    ALL_PHRASES.extend(row["phrases"] or [])
    ALL_KEYWORDS.extend(row["keywords"] or [])
    ALL_EXCLUDE.extend(row["exclude"] or [])

ALL_PHRASES = normalize_term_list(ALL_PHRASES)
ALL_KEYWORDS = normalize_term_list(ALL_KEYWORDS)
ALL_EXCLUDE = set(normalize_term_list(ALL_EXCLUDE))

# Remove excluded items if they appear
ALL_KEYWORDS = [k for k in ALL_KEYWORDS if k not in ALL_EXCLUDE]
ALL_PHRASES = [p for p in ALL_PHRASES if p not in ALL_EXCLUDE]

print(f"✅ Global phrase set:  {len(ALL_PHRASES):,}")
print(f"✅ Global keyword set: {len(ALL_KEYWORDS):,}")
print("Sample phrases:", ALL_PHRASES[:20])
print("Sample keywords:", ALL_KEYWORDS[:30])

(ARTIFACTS_DIR / "rq_phrases_en.json").write_text(json.dumps(ALL_PHRASES, ensure_ascii=False, indent=2))
(ARTIFACTS_DIR / "rq_keywords_en.json").write_text(json.dumps(ALL_KEYWORDS, ensure_ascii=False, indent=2))
print("💾 Saved rq_phrases_en.json and rq_keywords_en.json")

# ------------------------------------------------------------
# 4.6 Compute RQ alignment score for papers (phrase-weighted)
# ------------------------------------------------------------

import pandas as pd

def compute_rq_score(title: str, abstract: str, phrases: list[str], keywords: list[str]) -> float:
    """
    Weighted matching:
      - multi-word phrases are strong signals
      - single keywords are weaker signals
    Returns a score in [0, 1].
    """
    blob = f"{title or ''} {abstract or ''}".lower()

    if not phrases and not keywords:
        return 0.0

    phrase_hits = sum(1 for p in phrases if p and p in blob)
    keyword_hits = sum(1 for k in keywords if k and k in blob)

    phrase_score = phrase_hits / max(1, len(phrases))
    keyword_score = keyword_hits / max(1, len(keywords))

    # Heavier weight on phrases to reduce generic keyword noise
    score = 0.75 * phrase_score + 0.25 * keyword_score
    return float(max(0.0, min(1.0, score)))


def add_rq_scores(df: pd.DataFrame, phrases: list[str], keywords: list[str]) -> pd.DataFrame:
    out = df.copy()
    out["rq_score"] = out.apply(
        lambda r: compute_rq_score(
            r.get("title", ""),
            r.get("abstract", ""),
            phrases,
            keywords,
        ),
        axis=1
    )

    # Optional: keep some debug columns for quick inspection
    def _debug_hits(title, abstract):
        blob = f"{title or ''} {abstract or ''}".lower()
        ph = [p for p in phrases if p in blob][:5]
        kw = [k for k in keywords if k in blob][:10]
        return ph, kw

    out[["rq_phrase_hits_top5", "rq_keyword_hits_top10"]] = out.apply(
        lambda r: pd.Series(_debug_hits(r.get("title",""), r.get("abstract",""))),
        axis=1
    )

    return out


# Apply scoring
# If you already have work_df (from Section 3), use that instead of corpus_df.
target_df = corpus_df.copy()
target_df = add_rq_scores(target_df, ALL_PHRASES, ALL_KEYWORDS)

print("✅ RQ scores computed (phrase-weighted).")
display(
    target_df[["title", "rq_score", "rq_phrase_hits_top5", "rq_keyword_hits_top10", "doi", "openalex_id"]]
    .sort_values("rq_score", ascending=False)
    .head(25)
)

# Persist scored corpus (run-scoped artifacts)
rqscore_path = ARTIFACTS_DIR / "corpus_with_rq_scores.csv"
target_df.to_csv(rqscore_path, index=False)
print(f"💾 Saved scored corpus: {rqscore_path}")


📥 Loading High-priority RQs from Notion (NOTION_RQ_DB_ID)...
✅ Loaded 6 High-priority RQs.


,notion_page_id,rq_jp,priority
0,2a98e0e4-d162-8106-a2c6-ed44dd206ef8,どの公的介入が民間ベンチャーキャピタルの呼び込みに最も効果的か？,High
1,2a98e0e4-d162-8111-b44c-fa02ef55612b,政府の間接的VC（LP出資）モデルは直接VCより効果的か？,High
2,2aa8e0e4-d162-8147-9a97-fb835a4ca517,LP投資者として行動する政府系ファンド（SWF）は、国内スタートアップ・エコシステムの形成に...,High
3,2aa8e0e4-d162-8158-8f02-f232db3f0d1b,官民連携型のイノベーションハブがエコシステムの中核機能を果たすためには、どのような制度的要因...,High
4,2aa8e0e4-d162-8171-8c54-c590acbb3373,スタートアップ投資バブルを長期的なイノベーションの軌道に変換するには、どのような神話や物語構...,High
5,2aa8e0e4-d162-81ba-bb49-cd5ff39e8acb,外国系VCとコーポレートVCは、新興エコシステムにおけるスタートアップの成果にどのような違い...,High


🗂️ Translation cache loaded: 6 items
✅ JP->EN translation completed (cached).


,priority,rq_jp,rq_en
0,High,どの公的介入が民間ベンチャーキャピタルの呼び込みに最も効果的か？,Which public interventions are most effective ...
1,High,政府の間接的VC（LP出資）モデルは直接VCより効果的か？,Is the government's indirect VC (LP investment...
2,High,LP投資者として行動する政府系ファンド（SWF）は、国内スタートアップ・エコシステムの形成に...,How do sovereign wealth funds (SWFs) acting as...
3,High,官民連携型のイノベーションハブがエコシステムの中核機能を果たすためには、どのような制度的要因...,"What institutional factors (legal systems, cul..."
4,High,スタートアップ投資バブルを長期的なイノベーションの軌道に変換するには、どのような神話や物語構...,What myths or narrative structures are necessa...
5,High,外国系VCとコーポレートVCは、新興エコシステムにおけるスタートアップの成果にどのような違い...,How do foreign venture capitalists and corpora...


🗂️ RQ terms cache loaded: 6 items
💾 Saved RQ terms cache: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/rq_terms_cache.json


,rq_en,phrases,keywords
0,Which public interventions are most effective ...,"[government venture capital programs, public-p...","[vc, lp, swf, cvc, gvc, investment, funding, e..."
1,Is the government's indirect VC (LP investment...,"[government venture capital model, indirect ve...","[vc, lp, swf, cvc, gvc, investment, funding, e..."
2,How do sovereign wealth funds (SWFs) acting as...,"[sovereign wealth funds as lp investors, influ...","[swf, lp, vc, cvc, gvc, entrepreneurship, inno..."
3,"What institutional factors (legal systems, cul...","[institutional factors for innovation, public-...","[innovation, partnership, ecosystem, governanc..."
4,What myths or narrative structures are necessa...,"[startup investment bubble, long-term innovati...","[startup, innovation, venture capital, myth, n..."
5,How do foreign venture capitalists and corpora...,"[foreign venture capitalists impact, corporate...","[vc, cvc, gvc, startup, ecosystem, investment,..."


✅ Global phrase set:  86
✅ Global keyword set: 65
Sample phrases: ['government venture capital programs', 'public-private partnership models', 'venture capital attraction strategies', 'entrepreneurial ecosystem development', 'innovation funding mechanisms', 'limited partner investment incentives', 'sovereign wealth fund allocation', 'corporate venture capital initiatives', 'startup financing interventions', 'investment climate enhancement', 'seed funding programs', 'venture capital market dynamics', 'angel investor engagement', 'regional development agencies', 'economic growth through venture capital', 'government venture capital model', 'indirect venture capital investment', 'limited partner investment strategy', 'direct venture capital approach', 'venture capital effectiveness comparison']
Sample keywords: ['vc', 'lp', 'swf', 'cvc', 'gvc', 'investment', 'funding', 'entrepreneurship', 'innovation', 'capital', 'startup', 'policy', 'incentives', 'financing', 'ecosystem', 'intervention',

,title,rq_score,rq_phrase_hits_top5,rq_keyword_hits_top10,doi,openalex_id
36,Government-backed venture capital investments ...,0.031798,[government-backed venture capital],"[investment, capital, government, venture, net...",None,None
49,What is the role of Government Venture Capital...,0.026923,[],"[entrepreneurship, innovation, capital, govern...",None,None
7,Institutional differences and the development ...,0.023077,[],"[capital, development, venture, institutional,...",None,None
8,"Venture capital, entrepreneurship and economic...",0.023077,[],"[entrepreneurship, capital, venture, venture c...",None,None
13,"Corporate Venture Capital, Value Creation, and...",0.019231,[],"[innovation, capital, venture, venture capital...",None,None
48,Direct And Indirect Government Venture Capital...,0.019231,[],"[investment, capital, government, venture, ven...",None,None
47,Government venture capital and cross-border in...,0.019231,[],"[investment, capital, government, venture, ven...",None,None
46,Governmental and independent venture capital i...,0.019231,[],"[investment, capital, government, venture, ven...",None,None
38,Seeding the Way: Governments as Limited Partne...,0.019231,[],"[capital, government, limited partner, venture...",None,None
37,Optimizing Sustainable Entrepreneurial Ecosyst...,0.019231,[],"[financing, ecosystem, government, incubator, ...",None,None


💾 Saved scored corpus: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/corpus_with_rq_scores.csv


In [16]:
# ============================================================
# 5. Backward expansion via citation links
#   (controlled, purpose-driven version for Day20)
#
#   Strategy:
#     - Expand ONLY backward via references (hop=1)
#     - Strong global and per-core caps to avoid explosion
#     - Skip anything already in the existing corpus
# ============================================================

import time
import re
import json
from typing import Dict, Any, List, Optional, Tuple
import pandas as pd


# ------------------------------------------------------------
# 5.1 Parameters (SAFE DEFAULTS for Day20)
# ------------------------------------------------------------

MAX_HOPS = 1                       # backward only (direct references)
REF_LIMIT_PER_CORE = 50            # cap per core paper
MAX_CANDIDATES = 8000              # global hard stop
SLEEP_SEC = 0.10                   # gentle pacing

print("⚙️ Backward expansion parameters:")
print(f"   - MAX_HOPS={MAX_HOPS}")
print(f"   - REF_LIMIT_PER_CORE={REF_LIMIT_PER_CORE}")
print(f"   - MAX_CANDIDATES={MAX_CANDIDATES}")


# ------------------------------------------------------------
# 5.2 OpenAlex ID / API helpers (robust)
# ------------------------------------------------------------

def openalex_work_id_only(openalex_id: str) -> str:
    """
    Extract bare W-id from:
      - https://openalex.org/Wxxxx
      - https://api.openalex.org/works/Wxxxx
      - Wxxxx
    """
    if not openalex_id:
        return ""
    s = str(openalex_id).strip().rstrip("/")
    return s.split("/")[-1]

def to_openalex_api_work_url(openalex_id: str) -> str:
    """
    Convert any OpenAlex ID form into API URL.
    """
    wid = openalex_work_id_only(openalex_id)
    if not wid:
        return ""
    return f"{OPENALEX_BASE_URL}/works/{wid}"

def fetch_work_by_openalex_id(openalex_id: str) -> Optional[Dict[str, Any]]:
    url = to_openalex_api_work_url(openalex_id)
    if not url:
        return None
    try:
        return openalex_get(url, timeout=OPENALEX_TIMEOUT)
    except Exception:
        return None

def extract_reference_ids(work_json: Dict[str, Any]) -> List[str]:
    """
    OpenAlex references are returned as a list of work IDs.
    """
    return work_json.get("referenced_works") or []


# ------------------------------------------------------------
# 5.3 Resolve core papers to OpenAlex works (W-id based)
# ------------------------------------------------------------

print("🔗 Resolving core papers to OpenAlex works...")

core_rows = []
for _, row in core_df.iterrows():
    work = resolve_work_from_openalex(
        openalex_id=row.get("openalex_id"),
        doi=row.get("doi"),
        title=row.get("title"),
    )
    if not work:
        continue

    core_rows.append({
        "core_notion_page_id": row.get("notion_page_id"),
        "core_title": row.get("title"),
        "core_openalex_wid": openalex_work_id_only(work.get("id")),
        "core_year": work.get("publication_year"),
        "core_cited_by_count": work.get("cited_by_count", 0),
    })

core_oa_df = pd.DataFrame(core_rows).dropna(subset=["core_openalex_wid"])
print(f"✅ Resolved {len(core_oa_df):,}/{len(core_df):,} core papers.")
display(core_oa_df.head(10))

core_oa_path = ARTIFACTS_DIR / "core_openalex_resolved.csv"
core_oa_df.to_csv(core_oa_path, index=False)
print(f"💾 Saved: {core_oa_path}")


# ------------------------------------------------------------
# 5.4 Backward expansion (references only, hop=1)
# ------------------------------------------------------------

candidate_map: Dict[str, Dict[str, Any]] = {}

def add_candidate(candidate_wid: str, core_wid: str):
    if not candidate_wid:
        return
    cid = openalex_work_id_only(candidate_wid)

    # Skip if already in existing corpus
    if f"https://openalex.org/{cid}" in existing_openalex_ids:
        return

    if cid not in candidate_map:
        candidate_map[cid] = {
            "candidate_openalex_wid": cid,
            "min_hop": 1,
            "source_cores": {core_wid},
        }
    else:
        candidate_map[cid]["source_cores"].add(core_wid)


print("🚶 Starting controlled backward expansion (references only)...")

for _, core in core_oa_df.iterrows():
    core_wid = core["core_openalex_wid"]

    # Global stop
    if len(candidate_map) >= MAX_CANDIDATES:
        print(f"🧯 Reached MAX_CANDIDATES={MAX_CANDIDATES:,}. Stopping expansion.")
        break

    w = fetch_work_by_openalex_id(core_wid)
    if not w:
        continue

    ref_ids = extract_reference_ids(w)
    if not ref_ids:
        continue

    # Cap per core
    ref_ids = ref_ids[:REF_LIMIT_PER_CORE]

    for rid in ref_ids:
        add_candidate(rid, core_wid)

        if len(candidate_map) >= MAX_CANDIDATES:
            print(f"🧯 Reached MAX_CANDIDATES={MAX_CANDIDATES:,}. Stopping expansion.")
            break

    print(
        f"   - Core {core_wid}: "
        f"refs used={len(ref_ids):,}, "
        f"candidates so far={len(candidate_map):,}"
    )

    time.sleep(SLEEP_SEC)


# ------------------------------------------------------------
# 5.5 Materialize candidate pool (safe even if empty)
# ------------------------------------------------------------

rows = []
for cid, meta in candidate_map.items():
    rows.append({
        "candidate_openalex_wid": cid,
        "candidate_openalex_id": f"https://openalex.org/{cid}",
        "min_hop": meta["min_hop"],
        "source_cores": ",".join(sorted(meta["source_cores"])),
    })

if rows:
    candidate_pool_df = (
        pd.DataFrame(rows)
        .sort_values(["min_hop", "candidate_openalex_wid"])
        .reset_index(drop=True)
    )
else:
    candidate_pool_df = pd.DataFrame(
        columns=["candidate_openalex_wid", "candidate_openalex_id", "min_hop", "source_cores"]
    )

print(f"✅ Candidate pool built: {len(candidate_pool_df):,} candidates.")
display(candidate_pool_df.head(20))

candidate_pool_path = ARTIFACTS_DIR / "candidate_pool_raw.csv"
candidate_pool_df.to_csv(candidate_pool_path, index=False)
print(f"💾 Saved: {candidate_pool_path}")


⚙️ Backward expansion parameters:
   - MAX_HOPS=1
   - REF_LIMIT_PER_CORE=50
   - MAX_CANDIDATES=8000
🔗 Resolving core papers to OpenAlex works...
ERROR! Session/line number was not unique in database. History logging moved to new session 258
✅ Resolved 30/30 core papers.


,core_notion_page_id,core_title,core_openalex_wid,core_year,core_cited_by_count
0,2132b6ad-fd39-4f9f-a433-45a2fbdbd855,Venture Capital Networks and Cross-Border Star...,W3123914693,2001,2693
1,4a5fe6a2-d640-4218-84c8-61260b7955c4,Seeding the Way: Governments as Limited Partne...,W2153509701,2008,2356
2,da255a45-0fe2-42a0-bf86-ee9af1097021,Spatial heterogeneity in venture capital and n...,W2146006314,1995,1641
3,2dc8e0e4-d162-8196-9fe5-c97ef51ef4a4,The Barriers to Embedding Entrepreneurship Edu...,W2032540219,2014,1180
4,2a98e0e4-d162-81af-bf07-db4467b2c888,The Government as Venture Capitalist: The Long...,W3122760487,1999,1134
5,2a98e0e4-d162-8194-a50e-d3efd1ed5d9f,"Venture capital, entrepreneurship and economic...",W1990980528,2010,647
6,2a98e0e4-d162-8174-8b95-cf725a4fca42,Entrepreneurial ecosystems and growth oriented...,W2186324016,2014,953
7,2a98e0e4-d162-81e2-8131-e269353c4dc3,The determinants of venture capital funding: e...,W3121491467,2000,932
8,2a98e0e4-d162-81e4-b518-d74871550e8b,"Corporate Venture Capital, Value Creation, and...",W3124418789,2014,653
9,2a98e0e4-d162-810b-9aa3-d575608e429e,"Formal institutions, culture, and venture capi...",W3122486910,2011,443


💾 Saved: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/core_openalex_resolved.csv
🚶 Starting controlled backward expansion (references only)...
   - Core W3123914693: refs used=50, candidates so far=50
   - Core W2153509701: refs used=50, candidates so far=100
   - Core W2146006314: refs used=23, candidates so far=122
   - Core W2032540219: refs used=50, candidates so far=172
   - Core W3122760487: refs used=30, candidates so far=202
   - Core W1990980528: refs used=34, candidates so far=236
   - Core W2186324016: refs used=50, candidates so far=286
   - Core W3121491467: refs used=50, candidates so far=334
   - Core W3124418789: refs used=50, candidates so far=382
   - Core W3122486910: refs used=50, candidates so far=429
   - Core W1972989461: refs used=50, candidates so far=466
   - Core W2119345321: refs used=50, candidates so far=509
   - Core W3123296080: refs used=50, candidates so far=551
   - Core W3122536485: refs used=50, candidates s

,candidate_openalex_wid,candidate_openalex_id,min_hop,source_cores
0,W109618454,https://openalex.org/W109618454,1,W2031124818
1,W113044168,https://openalex.org/W113044168,1,W4387937766
2,W117623715,https://openalex.org/W117623715,1,W2146006314
3,W124740690,https://openalex.org/W124740690,1,W2031124818
4,W127852763,https://openalex.org/W127852763,1,W3121491467
5,W1459540051,https://openalex.org/W1459540051,1,W2153509701
6,W1480130531,https://openalex.org/W1480130531,1,W3124418789
7,W1480315348,https://openalex.org/W1480315348,1,W2186324016
8,W1482157032,https://openalex.org/W1482157032,1,"W2009200822,W2031124818,W3122486910"
9,W1489436643,https://openalex.org/W1489436643,1,W3124418789


💾 Saved: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/candidate_pool_raw.csv


In [17]:
# ============================================================
# 6. Metadata enrichment and candidate deduplication
#
# Goal:
#   - Enrich candidate_pool_raw.csv (OpenAlex works) with metadata needed for ranking:
#       title, year, doi, cited_by_count, abstract, concepts, venues, etc.
#   - Deduplicate candidates again using strong identifiers (OpenAlex/DOI/arXiv/title key)
#   - Optionally compute/update RQ score on enriched text (title+abstract)
#
# Inputs:
#   - candidate_pool_df (from Section 5) OR candidate_pool_raw.csv artifact
#   - existing_* indices from Section 2 (existing corpus)
#
# Outputs:
#   - candidate_pool_enriched.csv (run-scoped artifact)
#   - candidate_pool_enriched_deduped.csv (optional)
# ============================================================

import time
import json
from typing import Dict, Any, List, Optional
import pandas as pd


# ------------------------------------------------------------
# 6.1 Load candidate pool (in-memory or from artifact)
# ------------------------------------------------------------

candidate_pool_path = ARTIFACTS_DIR / "candidate_pool_raw.csv"
if "candidate_pool_df" in globals() and isinstance(candidate_pool_df, pd.DataFrame) and len(candidate_pool_df) > 0:
    candidate_pool = candidate_pool_df.copy()
    print(f"📥 Using in-memory candidate_pool_df: {len(candidate_pool):,} rows")
elif candidate_pool_path.exists():
    candidate_pool = pd.read_csv(candidate_pool_path)
    print(f"📥 Loaded candidate pool from artifact: {candidate_pool_path} ({len(candidate_pool):,} rows)")
else:
    raise FileNotFoundError("candidate_pool_raw.csv not found and candidate_pool_df not in memory.")


# ------------------------------------------------------------
# 6.2 Helpers: OpenAlex work fetch + parsing
# ------------------------------------------------------------

def parse_openalex_work(work: Dict[str, Any]) -> Dict[str, Any]:
    """
    Flatten key fields from an OpenAlex 'work' object.
    Keep it lightweight: only what we need for ranking + ingestion.
    """
    # Concepts: keep top N by score
    concepts = work.get("concepts") or []
    concepts_sorted = sorted(concepts, key=lambda x: x.get("score", 0), reverse=True)
    top_concepts = [c.get("display_name") for c in concepts_sorted[:8] if c.get("display_name")]

    host_venue = work.get("host_venue") or {}
    primary_loc = work.get("primary_location") or {}
    source = (host_venue.get("display_name") or primary_loc.get("source", {}) or {}).get("display_name")

    oa_status = None
    oa_url = None
    open_access = work.get("open_access") or {}
    oa_status = open_access.get("oa_status")
    oa_url = (primary_loc.get("pdf_url") or primary_loc.get("landing_page_url") or open_access.get("oa_url"))

    return {
        "openalex_id": work.get("id"),
        "openalex_wid": openalex_work_id_only(work.get("id")),
        "title": work.get("title"),
        "publication_year": work.get("publication_year"),
        "publication_date": work.get("publication_date"),
        "doi": (work.get("doi") or "").replace("https://doi.org/", "") if work.get("doi") else None,
        "cited_by_count": work.get("cited_by_count", 0),
        "abstract_inverted_index": work.get("abstract_inverted_index"),
        "concepts_top": "; ".join(top_concepts) if top_concepts else "",
        "venue": source,
        "type": work.get("type"),
        "oa_status": oa_status,
        "oa_url": oa_url,
    }


def inverted_index_to_text(inv: Optional[Dict[str, List[int]]]) -> str:
    """
    OpenAlex sometimes provides abstracts as an inverted index:
      { "word": [pos1,pos2,...], ... }
    Convert to plain text.
    """
    if not inv or not isinstance(inv, dict):
        return ""
    positions = []
    for word, pos_list in inv.items():
        for p in pos_list:
            positions.append((p, word))
    positions.sort(key=lambda x: x[0])
    return " ".join([w for _, w in positions])


def fetch_openalex_work_by_wid(wid: str) -> Optional[Dict[str, Any]]:
    """
    Fetch by W-id (e.g., W3123914693).
    """
    if not wid:
        return None
    url = f"{OPENALEX_BASE_URL}/works/{wid}"
    try:
        return openalex_get(url, timeout=OPENALEX_TIMEOUT)
    except Exception:
        return None


# ------------------------------------------------------------
# 6.3 Enrich candidates (with caching)
# ------------------------------------------------------------

ENRICH_CACHE_PATH = ARTIFACTS_DIR / "openalex_work_cache.json"

def load_work_cache(path: pathlib.Path) -> Dict[str, Any]:
    if path.exists():
        try:
            return json.loads(path.read_text())
        except Exception:
            return {}
    return {}

def save_work_cache(path: pathlib.Path, cache: Dict[str, Any]) -> None:
    path.write_text(json.dumps(cache, ensure_ascii=False))

work_cache = load_work_cache(ENRICH_CACHE_PATH)
print(f"🗂️ Work cache loaded: {len(work_cache):,} items")


enriched_rows = []
misses = 0

for i, row in candidate_pool.iterrows():
    wid = row.get("candidate_openalex_wid") or openalex_work_id_only(row.get("candidate_openalex_id", ""))
    if not wid:
        misses += 1
        continue

    if wid in work_cache:
        work = work_cache[wid]
    else:
        work = fetch_openalex_work_by_wid(wid)
        if work:
            work_cache[wid] = work
        time.sleep(0.08)

    if not work:
        misses += 1
        continue

    meta = parse_openalex_work(work)
    abstract_text = inverted_index_to_text(meta.get("abstract_inverted_index"))
    meta["abstract"] = abstract_text

    # Merge provenance columns from candidate pool
    meta["min_hop"] = row.get("min_hop", 1)
    meta["source_cores"] = row.get("source_cores", "")

    enriched_rows.append(meta)

    # Periodic checkpoint
    if (i + 1) % 200 == 0:
        print(f"   - processed {i+1:,}/{len(candidate_pool):,} | enriched={len(enriched_rows):,} | cache={len(work_cache):,}")
        save_work_cache(ENRICH_CACHE_PATH, work_cache)

# Save cache at end
save_work_cache(ENRICH_CACHE_PATH, work_cache)
print(f"💾 Saved work cache: {ENRICH_CACHE_PATH} ({len(work_cache):,} items)")
print(f"✅ Enrichment complete: enriched={len(enriched_rows):,}, misses={misses:,}")


candidate_enriched_df = pd.DataFrame(enriched_rows)
print(f"📦 Enriched candidate dataframe: {len(candidate_enriched_df):,} rows")
display(candidate_enriched_df.head(10))

enriched_path = ARTIFACTS_DIR / "candidate_pool_enriched.csv"
candidate_enriched_df.to_csv(enriched_path, index=False)
print(f"💾 Saved: {enriched_path}")


# ------------------------------------------------------------
# 6.4 Candidate deduplication (strong identifiers first)
# ------------------------------------------------------------

candidate_enriched_df["doi_norm"] = candidate_enriched_df["doi"].apply(normalize_doi)
candidate_enriched_df["openalex_norm"] = candidate_enriched_df["openalex_id"].apply(normalize_openalex_id)
candidate_enriched_df["title_norm"] = candidate_enriched_df["title"].apply(title_key)

# Remove items already in corpus
mask_in_corpus = (
    candidate_enriched_df["openalex_norm"].isin(existing_openalex_ids) |
    candidate_enriched_df["doi_norm"].isin(existing_dois) |
    candidate_enriched_df["title_norm"].isin(existing_title_keys)
)

dedup_df = candidate_enriched_df[~mask_in_corpus].copy()
print(f"✅ After removing items already in corpus: {len(dedup_df):,} candidates remain")

# Within-candidate duplicates (prefer DOI, then OpenAlex, then title)
dedup_df = dedup_df.sort_values(
    ["doi_norm", "openalex_norm", "cited_by_count"],
    ascending=[True, True, False]
)

dedup_df = dedup_df.drop_duplicates(subset=["doi_norm"], keep="first")
dedup_df = dedup_df.drop_duplicates(subset=["openalex_norm"], keep="first")
dedup_df = dedup_df.drop_duplicates(subset=["title_norm"], keep="first")

print(f"✅ After within-candidate dedupe: {len(dedup_df):,} candidates remain")
display(dedup_df[["title", "publication_year", "cited_by_count", "doi", "openalex_id", "oa_status"]].head(20))

dedup_path = ARTIFACTS_DIR / "candidate_pool_enriched_deduped.csv"
dedup_df.to_csv(dedup_path, index=False)
print(f"💾 Saved: {dedup_path}")


# ------------------------------------------------------------
# 6.5 Optional: recompute RQ score using enriched title+abstract (phrase-weighted)
# ------------------------------------------------------------

if "ALL_PHRASES" in globals() and "ALL_KEYWORDS" in globals():
    dedup_df["rq_score"] = dedup_df.apply(
        lambda r: compute_rq_score(r.get("title", ""), r.get("abstract", ""), ALL_PHRASES, ALL_KEYWORDS),
        axis=1
    )
    scored_path = ARTIFACTS_DIR / "candidate_pool_enriched_scored.csv"
    dedup_df.to_csv(scored_path, index=False)
    print(f"💾 Saved: {scored_path}")

    display(dedup_df[["title", "publication_year", "cited_by_count", "rq_score", "oa_status"]]
            .sort_values(["rq_score", "cited_by_count"], ascending=[False, False])
            .head(30))
else:
    print("ℹ️ Skipping RQ re-scoring (ALL_PHRASES / ALL_KEYWORDS not found).")


📥 Using in-memory candidate_pool_df: 1,125 rows
🗂️ Work cache loaded: 0 items
   - processed 200/1,125 | enriched=198 | cache=198
   - processed 400/1,125 | enriched=397 | cache=397
   - processed 600/1,125 | enriched=595 | cache=595
   - processed 1,000/1,125 | enriched=942 | cache=942
💾 Saved work cache: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/openalex_work_cache.json (983 items)
✅ Enrichment complete: enriched=983, misses=142
📦 Enriched candidate dataframe: 983 rows


,openalex_id,openalex_wid,title,publication_year,publication_date,doi,cited_by_count,abstract_inverted_index,concepts_top,venue,type,oa_status,oa_url,abstract,min_hop,source_cores
0,https://openalex.org/W109618454,W109618454,Private Equity in Latin America,2002.0,2002-11-30,10.3905/jpe.2002.320032,12,"{'Mexico': [0], 'is': [1, 83], 'carefully': [2...",Private equity; Latin Americans; Venture capit...,The Journal of Private Equity,article,closed,https://doi.org/10.3905/jpe.2002.320032,Mexico is carefully reviewed in this article a...,1,W2031124818
1,https://openalex.org/W113044168,W113044168,Why the Lean Start-Up Changes Everything,2013.0,2013-01-01,None,1362,"{'In': [0, 164], 'the': [12], 'past': [2], 'fe...",Business process reengineering; Business; Lean...,Munich Personal RePEc Archive (Ludwig Maximili...,article,green,http://hdl.handle.net/10596/6143,"In past few years, a new methodology for launc...",1,W4387937766
2,https://openalex.org/W124740690,W124740690,Organizational and Economic Explanations of Au...,1998.0,1998-06-22,None,132,"{'Corporate': [0], 'governance': [1, 87, 224, ...",Audit committee; Accounting; Corporate governa...,Journal of managerial issues,article,closed,https://www.questia.com/library/journal/1G1-21...,Corporate governance plays an important role i...,1,W2031124818
3,https://openalex.org/W127852763,W127852763,The Rise and Fall of Venture Capital,1994.0,1994-01-01,None,172,"{'Small': [0], 'firms': [1, 21, 53, 82, 136, 1...",Business; Venture capital; Capital (architectu...,None,article,closed,https://www.thebhc.org/sites/default/files/beh...,Small firms and new business creation have bec...,1,W3121491467
4,https://openalex.org/W1459540051,W1459540051,The Role of Risk in Targeting Payments for Env...,2005.0,2005-01-01,10.2139/ssrn.836144,13,None,Payment; Deforestation (computer science); Nat...,SSRN Electronic Journal,article,green,https://doi.org/10.2139/ssrn.836144,,1,W2153509701
5,https://openalex.org/W1480130531,W1480130531,The value of patents as indicators of inventiv...,1987.0,1987-10-15,10.1017/cbo9780511559938.006,307,"{'In': [0], 'this': [1], 'paper': [2], 'we': [...",Value (mathematics); Economics; Stock (firearm...,Cambridge University Press eBooks,book-chapter,closed,https://doi.org/10.1017/cbo9780511559938.006,In this paper we present an overview of a seri...,1,W3124418789
6,https://openalex.org/W1480315348,W1480315348,The Role and Importance of Gazelles and Other ...,2012.0,2012-10-19,None,17,None,Business; Industrial organization; Marketing,Research Portal (King's College London),article,closed,https://research.manchester.ac.uk/en/publicati...,,1,W2186324016
7,https://openalex.org/W1482157032,W1482157032,An institutional view of China's venture capit...,2003.0,2003-03-01,10.1016/s0883-9026(02)00079-4,533,None,Venture capital; Social venture capital; China...,Journal of Business Venturing,article,closed,https://doi.org/10.1016/s0883-9026(02)00079-4,,1,"W2009200822,W2031124818,W3122486910"
8,https://openalex.org/W1489436643,W1489436643,"Dynamic Competition, Innovation and Strategic ...",2008.0,2008-01-01,10.2139/ssrn.1161239,61,None,Competition (biology); Business; Industrial or...,SSRN Electronic Journal,article,green,https://doi.org/10.2139/ssrn.1161239,,1,W3124418789
9,https://openalex.org/W1489715817,W1489715817,The venture capital cycle,1999.0,1999-09-24,None,1663,"{'The': [118], 'venture': [1, 19, 57, 69, 84, ...",Venture capital; Social venture capital; Inves...,None,book,closed,http://www.loc.gov/catdir/toc/fy034/99013957.html,venture captial industry in the United States ...,1,W3122760487


💾 Saved: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/candidate_pool_enriched.csv
✅ After removing items already in corpus: 975 candidates remain
✅ After within-candidate dedupe: 902 candidates remain


,title,publication_year,cited_by_count,doi,openalex_id,oa_status
67,Wiley Interdisciplinary Reviews: Climate Change,2018.0,1603,10.1002/(issn)1757-7799,https://openalex.org/W1927194981,hybrid
251,"FOREIGN ENTRY, CULTURAL BARRIERS, AND LEARNING",1996.0,1741,10.1002/(sici)1097-0266(199602)17:2<151::aid-s...,https://openalex.org/W2050766092,closed
379,ENTREPRENEURSHIP IN MULTINATIONAL CORPORATIONS...,1997.0,1071,10.1002/(sici)1097-0266(199703)18:3<207::aid-s...,https://openalex.org/W2112033364,closed
258,The impact of stocks and flows of organization...,1999.0,1316,10.1002/(sici)1097-0266(199910)20:10<953::aid-...,https://openalex.org/W2055034160,closed
418,Don't go it alone: alliance network compositio...,2000.0,2705,10.1002/(sici)1097-0266(200003)21:3<267::aid-s...,https://openalex.org/W2127027722,closed
630,The Statistical Analysis of Failure Time Data,2002.0,3067,10.1002/9781118032985,https://openalex.org/W2977606681,closed
968,Startup Communities: Building an Entrepreneuri...,2012.0,459,10.1002/9781119204459,https://openalex.org/W614658731,closed
466,Measuring corporate environmental performance:...,2010.0,432,10.1002/bse.676,https://openalex.org/W2147174553,closed
138,The global strategy of emerging multinationals...,2012.0,440,10.1002/gsj.1030,https://openalex.org/W1996751723,closed
124,Venture capitalist participation and the post‐...,1995.0,266,10.1002/mde.4090160603,https://openalex.org/W1991333212,closed


💾 Saved: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/candidate_pool_enriched_deduped.csv
💾 Saved: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/candidate_pool_enriched_scored.csv


,title,publication_year,cited_by_count,rq_score,oa_status
790,TOWARD A GLOBAL MODEL OF VENTURE CAPITAL?,2004.0,263,0.061538,closed
66,Boulevard of broken dreams : why public effort...,2012.0,353,0.057692,closed
281,The impact of the institutional environment on...,2002.0,95,0.053846,closed
711,The legislative road to Silicon Valley,2006.0,361,0.050000,closed
392,Corporate Venture Capital as a Window on New T...,2008.0,334,0.050000,closed
584,Behind the Scenes: Intermediary Organizations ...,2017.0,258,0.050000,closed
446,Trajectories of Industrial Districts: Impact o...,2000.0,33,0.050000,closed
267,"Corporate venture capitalists: Autonomy, obsta...",1988.0,316,0.046154,hybrid
776,"Do Accelerators Work? If So, How?",2020.0,224,0.046154,closed
115,High-Tech Entrepreneurship in Europe: A Heuris...,2014.0,40,0.046154,green


In [18]:
# ============================================================
# 7. Priority scoring and ranking of backfill candidates
#
# Goal:
#   Turn the enriched + deduped candidate pool into a ranked "backfill list"
#   that favors:
#     (1) High RQ alignment
#     (2) High impact (cited_by_count)
#     (3) Older / foundational works (year)
#     (4) Strong provenance (cited by many core papers)
#
# Inputs:
#   - candidate_pool_enriched_scored.csv (recommended) OR dedup_df in memory
#
# Outputs:
#   - backfill_ranked.csv              (full ranking)
#   - backfill_topN.csv                (review list, e.g., Top 200)
#   - backfill_topN_fetch_queue.csv    (minimal columns for PDF fetch / Notion ingest)
# ============================================================

import math
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 7.1 Load scored candidates
# ------------------------------------------------------------

scored_path = ARTIFACTS_DIR / "candidate_pool_enriched_scored.csv"

if "dedup_df" in globals() and isinstance(dedup_df, pd.DataFrame) and len(dedup_df) > 0:
    scored_df = dedup_df.copy()
    print(f"📥 Using in-memory dedup_df: {len(scored_df):,} rows")
elif scored_path.exists():
    scored_df = pd.read_csv(scored_path)
    print(f"📥 Loaded scored candidates: {scored_path} ({len(scored_df):,} rows)")
else:
    raise FileNotFoundError("No scored candidate dataframe found.")


# Ensure numeric
scored_df["publication_year"] = pd.to_numeric(scored_df["publication_year"], errors="coerce")
scored_df["cited_by_count"] = pd.to_numeric(scored_df["cited_by_count"], errors="coerce").fillna(0)
scored_df["rq_score"] = pd.to_numeric(scored_df["rq_score"], errors="coerce").fillna(0.0)

# Optional: count how many core papers point to this candidate
def count_source_cores(x: str) -> int:
    if not isinstance(x, str) or not x.strip():
        return 0
    return len([p for p in x.split(",") if p.strip()])

scored_df["n_source_cores"] = scored_df["source_cores"].apply(count_source_cores)


# ------------------------------------------------------------
# 7.2 Feature engineering (normalized signals)
# ------------------------------------------------------------

# Impact: log-scale to reduce skew
scored_df["impact_log"] = np.log1p(scored_df["cited_by_count"])

# Normalize impact (min-max)
imin, imax = scored_df["impact_log"].min(), scored_df["impact_log"].max()
scored_df["impact_norm"] = 0.0 if imax <= imin else (scored_df["impact_log"] - imin) / (imax - imin)

# Oldness / foundational bonus:
# Convert year -> "older is better" in [0,1] using a soft range.
# You can tune YEAR_ANCHOR_START/END to match your field.
YEAR_ANCHOR_START = int(os.getenv("YEAR_ANCHOR_START", "1970"))  # very old
YEAR_ANCHOR_END = int(os.getenv("YEAR_ANCHOR_END", "2015"))      # recent-ish

def oldness_score(year: float) -> float:
    if np.isnan(year):
        return 0.25  # unknown year gets small bonus, not zero
    y = int(year)
    # Older than start -> 1.0 ; newer than end -> 0.0
    if y <= YEAR_ANCHOR_START:
        return 1.0
    if y >= YEAR_ANCHOR_END:
        return 0.0
    return float((YEAR_ANCHOR_END - y) / (YEAR_ANCHOR_END - YEAR_ANCHOR_START))

scored_df["oldness"] = scored_df["publication_year"].apply(oldness_score)

# Provenance: being referenced by multiple core papers
# Use log1p and normalize
scored_df["core_ref_log"] = np.log1p(scored_df["n_source_cores"])
cmin, cmax = scored_df["core_ref_log"].min(), scored_df["core_ref_log"].max()
scored_df["core_ref_norm"] = 0.0 if cmax <= cmin else (scored_df["core_ref_log"] - cmin) / (cmax - cmin)

# RQ alignment is already in [0,1], but it may be small depending on keyword set.
# Optionally rescale to emphasize separation.
RQ_RESCALE_POWER = float(os.getenv("RQ_RESCALE_POWER", "0.7"))
scored_df["rq_norm"] = np.power(scored_df["rq_score"].clip(0, 1), RQ_RESCALE_POWER)


# ------------------------------------------------------------
# 7.3 Priority score (weighted sum)
# ------------------------------------------------------------
# Recommended default weights for Day20:
#   - RQ:          0.45
#   - Impact:      0.35
#   - Oldness:     0.10
#   - Provenance:  0.10
#
W_RQ = float(os.getenv("W_RQ", "0.45"))
W_IMPACT = float(os.getenv("W_IMPACT", "0.35"))
W_OLD = float(os.getenv("W_OLD", "0.10"))
W_CORE = float(os.getenv("W_CORE", "0.10"))

w_sum = W_RQ + W_IMPACT + W_OLD + W_CORE
if abs(w_sum - 1.0) > 1e-6:
    print(f"⚠️ Weights sum to {w_sum:.3f}; renormalizing to 1.0")
    W_RQ, W_IMPACT, W_OLD, W_CORE = [w / w_sum for w in [W_RQ, W_IMPACT, W_OLD, W_CORE]]

scored_df["priority_score"] = (
    W_RQ * scored_df["rq_norm"] +
    W_IMPACT * scored_df["impact_norm"] +
    W_OLD * scored_df["oldness"] +
    W_CORE * scored_df["core_ref_norm"]
)

# Add an explain string (useful for human review)
scored_df["priority_explain"] = scored_df.apply(
    lambda r: (
        f"prio={r['priority_score']:.3f} | "
        f"rq={r['rq_norm']:.3f}, "
        f"impact={r['impact_norm']:.3f}, "
        f"old={r['oldness']:.3f}, "
        f"core={r['core_ref_norm']:.3f} "
        f"(n_core={int(r['n_source_cores'])})"
    ),
    axis=1
)

print("✅ Priority scores computed.")
display(scored_df[[
    "title", "publication_year", "cited_by_count", "rq_score",
    "n_source_cores", "priority_score", "priority_explain",
    "doi", "openalex_id", "oa_status", "oa_url"
]].sort_values("priority_score", ascending=False).head(30))


# ------------------------------------------------------------
# 7.4 Ranking outputs
# ------------------------------------------------------------

ranked_df = scored_df.sort_values(
    ["priority_score", "rq_score", "cited_by_count"],
    ascending=[False, False, False]
).reset_index(drop=True)

ranked_path = ARTIFACTS_DIR / "backfill_ranked.csv"
ranked_df.to_csv(ranked_path, index=False)
print(f"💾 Saved full ranking: {ranked_path}")

# Human review size (tune)
TOP_N = int(os.getenv("TOP_N_BACKFILL", "200"))
top_df = ranked_df.head(TOP_N).copy()

top_path = ARTIFACTS_DIR / f"backfill_top{TOP_N}.csv"
top_df.to_csv(top_path, index=False)
print(f"💾 Saved top list: {top_path}")

# Minimal fetch queue for downstream PDF ingestion / Notion upsert
fetch_cols = [
    "openalex_id", "openalex_wid", "doi", "title",
    "publication_year", "cited_by_count",
    "rq_score", "priority_score",
    "oa_status", "oa_url",
    "source_cores"
]

fetch_queue = top_df[[c for c in fetch_cols if c in top_df.columns]].copy()

fetch_queue_path = ARTIFACTS_DIR / f"backfill_fetch_queue_top{TOP_N}.csv"
fetch_queue.to_csv(fetch_queue_path, index=False)
print(f"💾 Saved fetch queue: {fetch_queue_path}")

display(fetch_queue.head(20))


📥 Using in-memory dedup_df: 902 rows
✅ Priority scores computed.


,title,publication_year,cited_by_count,rq_score,n_source_cores,priority_score,priority_explain,doi,openalex_id,oa_status,oa_url
577,"Theory of the firm: Managerial behavior, agenc...",1976.0,68693,0.007692,2,0.488397,"prio=0.488 | rq=0.033, impact=1.000, old=0.867...",10.1016/0304-405x(76)90026-x,https://openalex.org/W2752617332,closed,https://doi.org/10.1016/0304-405x(76)90026-x
174,The Nature of the Firm,1937.0,22988,0.034615,1,0.455967,"prio=0.456 | rq=0.095, impact=0.895, old=1.000...",10.1111/j.1468-0335.1937.tb00002.x,https://openalex.org/W2015930340,bronze,https://doi.org/10.1111/j.1468-0335.1937.tb000...
878,Self-efficacy: Toward a unifying theory of beh...,1977.0,40665,0.011538,1,0.436596,"prio=0.437 | rq=0.044, impact=0.950, old=0.844...",10.1037/0033-295x.84.2.191,https://openalex.org/W4292808503,green,https://doi.org/10.1037/0033-295x.84.2.191
766,Firm Resources and Sustained Competitive Advan...,1991.0,42825,0.003846,2,0.433501,"prio=0.434 | rq=0.020, impact=0.955, old=0.533...",10.1177/014920639101700108,https://openalex.org/W3124536214,closed,https://doi.org/10.1177/014920639101700108
756,Corporate financing and investment decisions w...,1984.0,18628,0.011538,2,0.431793,"prio=0.432 | rq=0.044, impact=0.875, old=0.689...",10.1016/0304-405x(84)90023-0,https://openalex.org/W3124114405,closed,https://doi.org/10.1016/0304-405x(84)90023-0
879,The moderator–mediator variable distinction in...,1986.0,68870,0.003846,1,0.423622,"prio=0.424 | rq=0.020, impact=1.000, old=0.644...",10.1037/0022-3514.51.6.1173,https://openalex.org/W4292811746,closed,https://doi.org/10.1037/0022-3514.51.6.1173
698,Law and Finance,1998.0,17725,0.007692,3,0.420310,"prio=0.420 | rq=0.033, impact=0.870, old=0.378...",10.1086/250042,https://openalex.org/W3121759882,closed,https://doi.org/10.1086/250042
431,EFFICIENT CAPITAL MARKETS: A REVIEW OF THEORY ...,1970.0,15571,0.007692,1,0.415098,"prio=0.415 | rq=0.033, impact=0.858, old=1.000...",10.1111/j.1540-6261.1970.tb00518.x,https://openalex.org/W2131773668,closed,https://doi.org/10.1111/j.1540-6261.1970.tb005...
738,The Iron Cage Revisited: Institutional Isomorp...,1983.0,33665,0.007692,1,0.412041,"prio=0.412 | rq=0.033, impact=0.931, old=0.711...",10.2307/2095101,https://openalex.org/W3123282572,closed,https://doi.org/10.2307/2095101
365,"Risk, Return, and Equilibrium: Empirical Tests",1973.0,14835,0.011538,1,0.411703,"prio=0.412 | rq=0.044, impact=0.853, old=0.933...",10.1086/260061,https://openalex.org/W2104795328,closed,https://doi.org/10.1086/260061


💾 Saved full ranking: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/backfill_ranked.csv
💾 Saved top list: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/backfill_top200.csv
💾 Saved fetch queue: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/backfill_fetch_queue_top200.csv


,openalex_id,openalex_wid,doi,title,publication_year,cited_by_count,rq_score,priority_score,oa_status,oa_url,source_cores
0,https://openalex.org/W2752617332,W2752617332,10.1016/0304-405x(76)90026-x,"Theory of the firm: Managerial behavior, agenc...",1976.0,68693,0.007692,0.488397,closed,https://doi.org/10.1016/0304-405x(76)90026-x,"W2119345321,W3121491467"
1,https://openalex.org/W2015930340,W2015930340,10.1111/j.1468-0335.1937.tb00002.x,The Nature of the Firm,1937.0,22988,0.034615,0.455967,bronze,https://doi.org/10.1111/j.1468-0335.1937.tb000...,W2032540219
2,https://openalex.org/W4292808503,W4292808503,10.1037/0033-295x.84.2.191,Self-efficacy: Toward a unifying theory of beh...,1977.0,40665,0.011538,0.436596,green,https://doi.org/10.1037/0033-295x.84.2.191,W4393229081
3,https://openalex.org/W3124536214,W3124536214,10.1177/014920639101700108,Firm Resources and Sustained Competitive Advan...,1991.0,42825,0.003846,0.433501,closed,https://doi.org/10.1177/014920639101700108,"W2032540219,W4387937766"
4,https://openalex.org/W3124114405,W3124114405,10.1016/0304-405x(84)90023-0,Corporate financing and investment decisions w...,1984.0,18628,0.011538,0.431793,closed,https://doi.org/10.1016/0304-405x(84)90023-0,"W3121491467,W3122760487"
5,https://openalex.org/W4292811746,W4292811746,10.1037/0022-3514.51.6.1173,The moderator–mediator variable distinction in...,1986.0,68870,0.003846,0.423622,closed,https://doi.org/10.1037/0022-3514.51.6.1173,W4393229081
6,https://openalex.org/W3121759882,W3121759882,10.1086/250042,Law and Finance,1998.0,17725,0.007692,0.420310,closed,https://doi.org/10.1086/250042,"W2009200822,W2031124818,W3121491467"
7,https://openalex.org/W2131773668,W2131773668,10.1111/j.1540-6261.1970.tb00518.x,EFFICIENT CAPITAL MARKETS: A REVIEW OF THEORY ...,1970.0,15571,0.007692,0.415098,closed,https://doi.org/10.1111/j.1540-6261.1970.tb005...,W3123914693
8,https://openalex.org/W3123282572,W3123282572,10.2307/2095101,The Iron Cage Revisited: Institutional Isomorp...,1983.0,33665,0.007692,0.412041,closed,https://doi.org/10.2307/2095101,W2009200822
9,https://openalex.org/W2104795328,W2104795328,10.1086/260061,"Risk, Return, and Equilibrium: Empirical Tests",1973.0,14835,0.011538,0.411703,closed,https://doi.org/10.1086/260061,W3123914693


In [24]:
# ============================================================
# 8. Open-access PDF resolution and download attempts (REVISED)
#
# Goal:
#   For the top-ranked backfill candidates, try to resolve and download an OA PDF.
#   We will:
#     1) Build an ordered list of URL candidates per paper (OpenAlex pdf_url > OA landing > known landings > DOI)
#     2) (Optional) Expand certain landing pages (DOAJ/Wiley) into direct PDF links by scraping HTML once
#     3) Download safely via streaming with validation (%PDF magic bytes + min size)
#     4) Write structured per-paper diagnostics (attempt logs, chosen_url, sha256)
#     5) Save a "download results" CSV for downstream Drive/Notion ingestion
#
# Inputs:
#   - backfill_fetch_queue_top{TOP_N}.csv (from Section 7)
#   - openalex_work_cache.json (from Section 6; should contain full OpenAlex work JSON)
#
# Outputs:
#   - pdf_download_results.csv
#   - pdf_download_attempts.jsonl
#   - downloaded PDFs under: ARTIFACTS_DIR / "pdfs"
#
# Notes:
#   - This does NOT bypass paywalls. Failures are recorded.
#   - Success rate will be low if your queue is dominated by "closed" classics.
#   - To raise success rate, we filter to "likely OA" items by default (configurable).
# ============================================================

from __future__ import annotations

import os
import re
import time
import json
import hashlib
import pathlib
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, List, Tuple

import requests
import pandas as pd

# Optional (used only for landing->pdf expansion)
try:
    from bs4 import BeautifulSoup
    _BS4_AVAILABLE = True
except Exception:
    BeautifulSoup = None
    _BS4_AVAILABLE = False


# ------------------------------------------------------------
# 8.1 Config knobs (override via env.txt)
# ------------------------------------------------------------
PDF_TIMEOUT_SEC = float(os.getenv("PDF_TIMEOUT_SEC", "60"))
PDF_MAX_REDIRECTS = int(os.getenv("PDF_MAX_REDIRECTS", "10"))

# Lower default MIN bytes to reduce false failures on small PDFs.
# Still protected by %PDF magic bytes validation.
PDF_MIN_BYTES = int(os.getenv("PDF_MIN_BYTES", "15000"))  # 15KB default (was 50KB)
PDF_MAX_BYTES = int(os.getenv("PDF_MAX_BYTES", str(200 * 1024 * 1024)))  # 200MB safeguard
PDF_SLEEP_BETWEEN_ATTEMPTS_SEC = float(os.getenv("PDF_SLEEP_BETWEEN_ATTEMPTS_SEC", "0.8"))

# Candidate list controls
MAX_URLS_PER_PAPER = int(os.getenv("MAX_URLS_PER_PAPER", "8"))
GLOBAL_SLEEP_SEC = float(os.getenv("GLOBAL_SLEEP_SEC", "0.15"))

# Filtering to improve success rate
FILTER_LIKELY_OA = os.getenv("FILTER_LIKELY_OA", "true").lower() in ("1", "true", "yes")

# Landing->PDF expansion
ENABLE_LANDING_PDF_EXPANSION = os.getenv("ENABLE_LANDING_PDF_EXPANSION", "true").lower() in ("1", "true", "yes")
LANDING_EXPANSION_DOMAINS = tuple(
    d.strip() for d in os.getenv("LANDING_EXPANSION_DOMAINS", "doaj.org,onlinelibrary.wiley.com").split(",") if d.strip()
)

PDF_HEADERS = {
    "Accept": "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8",
    "User-Agent": os.getenv(
        "PDF_USER_AGENT",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122 Safari/537.36",
    ),
    "Accept-Language": "en-US,en;q=0.9,ja;q=0.8",
    "Connection": "keep-alive",
}

PDF_PROXIES = {}
if os.getenv("HTTP_PROXY"):
    PDF_PROXIES["http"] = os.getenv("HTTP_PROXY")
if os.getenv("HTTPS_PROXY"):
    PDF_PROXIES["https"] = os.getenv("HTTPS_PROXY")

PDF_DIR = ARTIFACTS_DIR / "pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

print("=== PDF Download Config ===")
print("PDF_DIR:", PDF_DIR)
print("PDF_TIMEOUT_SEC:", PDF_TIMEOUT_SEC)
print("PDF_MIN_BYTES:", PDF_MIN_BYTES)
print("PDF_MAX_BYTES:", PDF_MAX_BYTES)
print("MAX_URLS_PER_PAPER:", MAX_URLS_PER_PAPER)
print("FILTER_LIKELY_OA:", FILTER_LIKELY_OA)
print("ENABLE_LANDING_PDF_EXPANSION:", ENABLE_LANDING_PDF_EXPANSION, "| bs4:", _BS4_AVAILABLE)
print("LANDING_EXPANSION_DOMAINS:", LANDING_EXPANSION_DOMAINS)
print("===========================\n")


# ------------------------------------------------------------
# 8.2 Attempt data model (structured diagnostics)
# ------------------------------------------------------------
@dataclass
class DownloadAttempt:
    url: str
    final_url: Optional[str] = None
    status_code: Optional[int] = None
    content_type: Optional[str] = None
    content_length: Optional[int] = None
    bytes: int = 0
    error: str = ""
    elapsed_sec: Optional[float] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


# ------------------------------------------------------------
# 8.3 Helpers: hashing, filename, validation
# ------------------------------------------------------------
def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sanitize_filename(name: str, max_len: int = 160) -> str:
    name = (name or "").strip()
    name = re.sub(r"[\\/:*?\"<>|]+", "_", name)
    name = re.sub(r"\s+", " ", name).strip()
    if len(name) > max_len:
        name = name[:max_len].rstrip()
    return name or "paper"

def sniff_pdf_magic(local_path: str) -> bool:
    try:
        with open(local_path, "rb") as f:
            head = f.read(5)
        return head == b"%PDF-"
    except Exception:
        return False

def safe_int(x: Optional[str]) -> Optional[int]:
    try:
        if x is None:
            return None
        return int(x)
    except Exception:
        return None

def normalize_url(u: Any) -> str:
    u = str(u or "").strip()
    return u

def is_probably_pdf_url(u: str) -> bool:
    u = (u or "").lower()
    return (".pdf" in u) or ("pdf" in u and "download" in u)


# ------------------------------------------------------------
# 8.4 Streaming downloader (single URL)
# ------------------------------------------------------------
SESSION = requests.Session()

def download_pdf_streaming(
    url: str,
    out_path: str,
    session: Optional[requests.Session] = None,
    timeout_sec: float = PDF_TIMEOUT_SEC,
    min_bytes: int = PDF_MIN_BYTES,
    max_bytes: int = PDF_MAX_BYTES,
) -> Tuple[bool, DownloadAttempt]:
    """
    Attempt to download a single URL. Writes to out_path if successful and valid.
    Returns: (ok, attempt)
    """
    sess = session or SESSION
    attempt = DownloadAttempt(url=url)

    t0 = time.time()
    try:
        resp = sess.get(
            url,
            stream=True,
            timeout=timeout_sec,
            allow_redirects=True,
            headers=PDF_HEADERS,
            proxies=PDF_PROXIES or None,
        )
        attempt.final_url = resp.url
        attempt.status_code = resp.status_code
        attempt.content_type = resp.headers.get("Content-Type")
        attempt.content_length = safe_int(resp.headers.get("Content-Length"))

        if resp.status_code != 200:
            attempt.error = f"http_status_{resp.status_code}"
            attempt.elapsed_sec = round(time.time() - t0, 3)
            return False, attempt

        tmp_path = out_path + ".part"
        total = 0
        with open(tmp_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 256):
                if not chunk:
                    continue
                f.write(chunk)
                total += len(chunk)
                if total > max_bytes:
                    attempt.error = f"too_large>{max_bytes}"
                    break

        attempt.bytes = total
        attempt.elapsed_sec = round(time.time() - t0, 3)

        if attempt.error.startswith("too_large"):
            pathlib.Path(tmp_path).unlink(missing_ok=True)
            return False, attempt

        if total < min_bytes:
            attempt.error = f"too_small<{min_bytes}"
            pathlib.Path(tmp_path).unlink(missing_ok=True)
            return False, attempt

        if not sniff_pdf_magic(tmp_path):
            attempt.error = "not_pdf_magic"
            pathlib.Path(tmp_path).unlink(missing_ok=True)
            return False, attempt

        pathlib.Path(tmp_path).rename(out_path)
        return True, attempt

    except requests.exceptions.RequestException as e:
        resp = getattr(e, "response", None)
        if resp is not None:
            attempt.final_url = getattr(resp, "url", None)
            attempt.status_code = getattr(resp, "status_code", None)
            attempt.content_type = (getattr(resp, "headers", {}) or {}).get("Content-Type")
            attempt.content_length = safe_int((getattr(resp, "headers", {}) or {}).get("Content-Length"))
        attempt.error = f"{type(e).__name__}: {str(e)[:160]}"
        attempt.elapsed_sec = round(time.time() - t0, 3)
        return False, attempt

    except Exception as e:
        attempt.error = f"{type(e).__name__}: {str(e)[:160]}"
        attempt.elapsed_sec = round(time.time() - t0, 3)
        return False, attempt


# ------------------------------------------------------------
# 8.5 Work cache loading + utilities
# ------------------------------------------------------------
WORK_CACHE_PATH = ARTIFACTS_DIR / "openalex_work_cache.json"

def load_work_cache(path: pathlib.Path) -> Dict[str, Any]:
    if path.exists():
        try:
            return json.loads(path.read_text())
        except Exception:
            return {}
    return {}

WORK_CACHE = load_work_cache(WORK_CACHE_PATH)
print(f"🗂️ Loaded work cache for URL resolution: {len(WORK_CACHE):,} works")

def _safe_get(d: Any, keys: List[str]) -> Any:
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return None
        cur = cur.get(k)
    return cur

def _add_url(urls: List[str], u: Any):
    if not u:
        return
    u = normalize_url(u)
    if not u:
        return
    urls.append(u)


# ------------------------------------------------------------
# 8.6 Landing page -> PDF expansion (DOAJ / Wiley; conservative)
# ------------------------------------------------------------
def try_expand_landing_to_pdf(url: str, timeout: int = 20) -> List[str]:
    """
    Fetch landing HTML and try to extract likely PDF links.
    Returns list of PDF-ish URLs (may be empty).
    """
    url = normalize_url(url)
    if not url:
        return []

    if not ENABLE_LANDING_PDF_EXPANSION:
        return []
    if not _BS4_AVAILABLE:
        return []
    if not any(d in url for d in LANDING_EXPANSION_DOMAINS):
        return []

    try:
        r = SESSION.get(url, timeout=timeout, headers=PDF_HEADERS, allow_redirects=True)
        if r.status_code != 200:
            return []
        ct = (r.headers.get("Content-Type") or "").lower()
        if "html" not in ct:
            return []

        soup = BeautifulSoup(r.text, "html.parser")
        links = []

        for a in soup.select("a[href]"):
            href = a.get("href") or ""
            href = href.strip()
            if not href:
                continue
            if ".pdf" not in href.lower():
                continue

            # absolutize
            if href.startswith("/"):
                m = re.match(r"^(https?://[^/]+)", r.url)
                if m:
                    href = m.group(1) + href

            links.append(href)

        # de-dup preserve order
        out, seen = [], set()
        for u in links:
            u = normalize_url(u)
            if u and u not in seen:
                seen.add(u)
                out.append(u)

        return out[:5]

    except Exception:
        return []


# ------------------------------------------------------------
# 8.7 Build ordered URL candidates per paper (OpenAlex-first)
# ------------------------------------------------------------
def build_pdf_url_candidates(row: pd.Series) -> List[str]:
    """
    Ordered candidates (best-first):
      1) OpenAlex best_oa_location.pdf_url / primary_location.pdf_url (direct PDFs)
      2) OpenAlex open_access.oa_url (repository landing)
      3) locations[].pdf_url / landing_page_url (if any)
      4) row.oa_url
      5) DOI landing (last)
    Then, optionally expand certain landing pages into direct PDF URLs.
    """
    urls: List[str] = []

    wid = str(row.get("openalex_wid") or "").strip()
    if not wid:
        oid = normalize_url(row.get("openalex_id"))
        wid = openalex_work_id_only(oid)

    w = WORK_CACHE.get(wid) if wid else None

    if isinstance(w, dict):
        # direct PDFs
        _add_url(urls, _safe_get(w, ["best_oa_location", "pdf_url"]))
        _add_url(urls, _safe_get(w, ["primary_location", "pdf_url"]))

        # OA landing
        _add_url(urls, _safe_get(w, ["open_access", "oa_url"]))

        # landing pages (may be expanded later)
        _add_url(urls, _safe_get(w, ["best_oa_location", "landing_page_url"]))
        _add_url(urls, _safe_get(w, ["primary_location", "landing_page_url"]))

        # locations list (sometimes has repo links)
        locs = w.get("locations") or []
        if isinstance(locs, list):
            for loc in locs[:10]:
                if not isinstance(loc, dict):
                    continue
                _add_url(urls, loc.get("pdf_url"))
                _add_url(urls, loc.get("landing_page_url"))

    # row-level OA URL
    _add_url(urls, row.get("oa_url"))

    # DOI last resort
    doi = normalize_url(row.get("doi"))
    if doi:
        if doi.startswith("10."):
            _add_url(urls, f"https://doi.org/{doi}")
        elif doi.startswith("http"):
            _add_url(urls, doi)

    # de-dup preserve order
    seen = set()
    out: List[str] = []
    for u in urls:
        u = normalize_url(u)
        if not u or u in seen:
            continue
        seen.add(u)
        out.append(u)

    # optionally expand landings to direct PDF links
    expanded = []
    for u in list(out)[:MAX_URLS_PER_PAPER]:
        expanded.extend(try_expand_landing_to_pdf(u))

    # put expanded PDFs first
    for u in expanded:
        u = normalize_url(u)
        if u and u not in out:
            out.insert(0, u)

    # final cap
    return out[:MAX_URLS_PER_PAPER]


# ------------------------------------------------------------
# 8.8 Likely-OA filter (raises success rate)
# ------------------------------------------------------------
def is_likely_oa(row: pd.Series) -> bool:
    wid = str(row.get("openalex_wid") or "").strip()
    w = WORK_CACHE.get(wid)
    if not isinstance(w, dict):
        return False
    oa = w.get("open_access") or {}
    if oa.get("is_oa") is True:
        return True
    if oa.get("any_repository_has_fulltext") is True:
        return True
    if w.get("has_fulltext") is True:
        return True
    # If OpenAlex provides a direct pdf_url anywhere, treat as likely OA
    if _safe_get(w, ["best_oa_location", "pdf_url"]) or _safe_get(w, ["primary_location", "pdf_url"]):
        return True
    locs = w.get("locations") or []
    if isinstance(locs, list):
        for loc in locs[:5]:
            if isinstance(loc, dict) and loc.get("pdf_url"):
                return True
    return False


# ------------------------------------------------------------
# 8.9 Multi-candidate fetch with diagnostics (per paper)
# ------------------------------------------------------------
def fetch_pdf_from_candidates(
    paper_id: str,
    url_candidates: List[str],
    out_dir: pathlib.Path = PDF_DIR,
    filename_hint: Optional[str] = None,
    max_attempts: int = 6,
    sleep_sec: float = PDF_SLEEP_BETWEEN_ATTEMPTS_SEC,
) -> Dict[str, Any]:
    """
    Try multiple URL candidates to fetch a PDF. Returns a structured dict.
    """
    seen = set()
    deduped: List[str] = []
    for u in url_candidates or []:
        u = normalize_url(u)
        if not u or u in seen:
            continue
        seen.add(u)
        deduped.append(u)

    deduped = deduped[:max_attempts]
    out_dir.mkdir(parents=True, exist_ok=True)

    base = sanitize_filename(filename_hint) if filename_hint else sanitize_filename(paper_id.replace("https://openalex.org/", ""))
    out_path = str(out_dir / f"{base}.pdf")

    attempts: List[Dict[str, Any]] = []
    last_err = None

    for u in deduped:
        ok, att = download_pdf_streaming(url=u, out_path=out_path, session=SESSION)
        attempts.append(att.to_dict())

        if ok:
            sha = sha256_file(out_path)
            size = pathlib.Path(out_path).stat().st_size
            return {
                "ok": True,
                "paper_id": paper_id,
                "chosen_url": u,
                "final_url": att.final_url,
                "local_path": out_path,
                "bytes": size,
                "sha256": sha,
                "attempts": attempts,
                "final_error_code": None,
                "final_error_message": None,
            }

        last_err = att.error
        time.sleep(sleep_sec)

    any_403 = any((a.get("status_code") == 403) for a in attempts)
    any_401 = any((a.get("status_code") == 401) for a in attempts)
    any_404 = any((a.get("status_code") == 404) for a in attempts)
    any_not_pdf = any((a.get("error") in ("not_pdf_magic",)) for a in attempts)

    if any_403 or any_401:
        final_code = "paywall_suspected"
    elif any_404:
        final_code = "not_found"
    elif any_not_pdf:
        final_code = "not_pdf"
    else:
        final_code = "fetch_failed"

    return {
        "ok": False,
        "paper_id": paper_id,
        "chosen_url": None,
        "final_url": None,
        "local_path": None,
        "bytes": 0,
        "sha256": None,
        "attempts": attempts,
        "final_error_code": final_code,
        "final_error_message": last_err or "no_successful_download",
    }


# ------------------------------------------------------------
# 8.10 Orchestrate downloads for Top-N queue
# ------------------------------------------------------------
TOP_N_PDF = int(os.getenv("TOP_N_PDF", "30"))
MAX_PER_RUN = min(TOP_N_PDF, 500)

# Choose the newest fetch queue file (best effort)
fetch_queue_files = sorted(list(ARTIFACTS_DIR.glob("backfill_fetch_queue_top*.csv")))
if not fetch_queue_files:
    raise FileNotFoundError("No backfill_fetch_queue_top*.csv found. Run Section 7 first.")
fetch_queue_path = fetch_queue_files[-1]

targets_df = pd.read_csv(fetch_queue_path).head(MAX_PER_RUN).copy()
print(f"📥 Loaded fetch queue: {fetch_queue_path.name} | targets={len(targets_df):,}")

# Optional filter to improve success rate
if FILTER_LIKELY_OA:
    targets_df["likely_oa"] = targets_df.apply(is_likely_oa, axis=1)
    print("🔎 likely_oa count:", int(targets_df["likely_oa"].sum()), "/", len(targets_df))
    targets_df = targets_df[targets_df["likely_oa"]].copy()
    print("📌 After FILTER_LIKELY_OA:", len(targets_df))

# If filtering made it empty, fall back to original top rows
if len(targets_df) == 0:
    print("⚠️ FILTER_LIKELY_OA resulted in 0 targets. Falling back to unfiltered top rows.")
    targets_df = pd.read_csv(fetch_queue_path).head(MAX_PER_RUN).copy()

results: List[Dict[str, Any]] = []
attempts_jsonl_path = ARTIFACTS_DIR / "pdf_download_attempts.jsonl"

ok_count = 0
fail_count = 0

with attempts_jsonl_path.open("w", encoding="utf-8") as f:
    for i, row in targets_df.reset_index(drop=True).iterrows():
        paper_id = normalize_url(row.get("openalex_id")) or normalize_url(row.get("doi")) or f"row{i}"
        title = normalize_url(row.get("title"))[:140]

        url_candidates = build_pdf_url_candidates(row)

        filename_hint = f"{openalex_work_id_only(paper_id)}_{title}" if title else openalex_work_id_only(paper_id)

        res = fetch_pdf_from_candidates(
            paper_id=paper_id,
            url_candidates=url_candidates,
            out_dir=PDF_DIR,
            filename_hint=filename_hint,
            max_attempts=min(6, len(url_candidates)),
        )

        results.append({
            "paper_id": paper_id,
            "title": row.get("title"),
            "doi": row.get("doi"),
            "publication_year": row.get("publication_year"),
            "cited_by_count": row.get("cited_by_count"),
            "rq_score": row.get("rq_score"),
            "priority_score": row.get("priority_score"),
            "oa_status": row.get("oa_status"),
            "oa_url": row.get("oa_url"),
            "url_candidates": json.dumps(url_candidates, ensure_ascii=False),
            "ok": res["ok"],
            "chosen_url": res["chosen_url"],
            "final_url": res.get("final_url"),
            "local_path": res["local_path"],
            "bytes": res["bytes"],
            "sha256": res["sha256"],
            "final_error_code": res["final_error_code"],
            "final_error_message": res["final_error_message"],
        })

        # Write full attempt logs as JSONL (1 line per paper)
        f.write(json.dumps({
            "paper_id": paper_id,
            "url_candidates": url_candidates,
            "result": res,
        }, ensure_ascii=False) + "\n")

        if res["ok"]:
            ok_count += 1
        else:
            fail_count += 1

        if (i + 1) % 10 == 0:
            print(f"   - processed {i+1:,}/{len(targets_df):,} | ok={ok_count:,} fail={fail_count:,}")

        time.sleep(GLOBAL_SLEEP_SEC)  # global politeness pacing

print(f"✅ PDF attempt run done: ok={ok_count:,}, fail={fail_count:,}")
print(f"🧾 Attempts log (JSONL): {attempts_jsonl_path}")

results_df = pd.DataFrame(results)
display(results_df.head(20))

results_path = ARTIFACTS_DIR / "pdf_download_results.csv"
results_df.to_csv(results_path, index=False)
print(f"💾 Saved: {results_path}")


# ------------------------------------------------------------
# 8.11 Quick diagnostics
# ------------------------------------------------------------
if len(results_df) > 0:
    print("📊 Success rate:", round(100 * results_df["ok"].mean(), 1), "%")
    print("Top failure codes:")
    display(results_df[~results_df["ok"]]["final_error_code"].value_counts().head(10))

    print("✅ Sample successful downloads:")
    display(results_df[results_df["ok"]][["title", "doi", "bytes", "local_path", "chosen_url"]].head(10))

    print("🔍 Sample URL candidates (first 5 papers):")
    for j in range(min(5, len(results_df))):
        print("-" * 80)
        print("title:", results_df.loc[j, "title"])
        try:
            cands = json.loads(results_df.loc[j, "url_candidates"])
        except Exception:
            cands = []
        print("cands:", cands[:8])

else:
    print("⚠️ No results were produced (empty targets).")

print("✅ Section 8 ready: Open-access PDF resolution + download attempts")


=== PDF Download Config ===
PDF_DIR: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/pdfs
PDF_TIMEOUT_SEC: 60.0
PDF_MIN_BYTES: 15000
PDF_MAX_BYTES: 209715200
MAX_URLS_PER_PAPER: 8
FILTER_LIKELY_OA: True
ENABLE_LANDING_PDF_EXPANSION: True | bs4: True
LANDING_EXPANSION_DOMAINS: ('doaj.org', 'onlinelibrary.wiley.com')

🗂️ Loaded work cache for URL resolution: 983 works
📥 Loaded fetch queue: backfill_fetch_queue_top200.csv | targets=30
🔎 likely_oa count: 8 / 30
📌 After FILTER_LIKELY_OA: 8
✅ PDF attempt run done: ok=1, fail=7
🧾 Attempts log (JSONL): /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/pdf_download_attempts.jsonl


,paper_id,title,doi,publication_year,cited_by_count,rq_score,priority_score,oa_status,oa_url,url_candidates,ok,chosen_url,final_url,local_path,bytes,sha256,final_error_code,final_error_message
0,https://openalex.org/W2015930340,The Nature of the Firm,10.1111/j.1468-0335.1937.tb00002.x,1937.0,22988,0.034615,0.455967,bronze,https://doi.org/10.1111/j.1468-0335.1937.tb000...,"[""https://onlinelibrary.wiley.com/doi/full/10....",False,None,None,None,0,None,paywall_suspected,http_status_403
1,https://openalex.org/W4292808503,Self-efficacy: Toward a unifying theory of beh...,10.1037/0033-295x.84.2.191,1977.0,40665,0.011538,0.436596,green,https://doi.org/10.1037/0033-295x.84.2.191,"[""https://doaj.org/article/ccb9da1c287e4fc8898...",False,None,None,None,0,None,not_pdf,too_small<15000
2,https://openalex.org/W2162012454,Investor protection and corporate governance,10.1016/s0304-405x(00)00065-9,2000.0,6134,0.007692,0.400629,green,https://doi.org/10.1016/s0304-405x(00)00065-9,"[""http://nrs.harvard.edu/urn-3:HUL.InstRepos:2...",False,None,None,None,0,None,not_pdf,too_small<15000
3,https://openalex.org/W3121865524,Venture capital and the structure of capital m...,10.1016/s0304-405x(97)00045-7,1998.0,1395,0.030769,0.396508,hybrid,https://doi.org/10.1016/s0304-405x(97)00045-7,"[""https://doi.org/10.1016/s0304-405x(97)00045-7""]",False,None,None,None,0,None,fetch_failed,too_small<15000
4,https://openalex.org/W2150291618,The central role of the propensity score in ob...,10.1093/biomet/70.1.41,1983.0,29863,0.000000,0.393117,bronze,https://academic.oup.com/biomet/article-pdf/70...,"[""https://academic.oup.com/biomet/article-pdf/...",False,None,None,None,0,None,paywall_suspected,not_pdf_magic
5,https://openalex.org/W2141951329,Building Theories from Case Study Research,10.5465/amr.1989.4308385,1989.0,23023,0.003846,0.380246,gold,https://doi.org/10.5465/amr.1989.4308385,"[""http://hdl.handle.net/20.500.12010/27586"", ""...",False,None,None,None,0,None,paywall_suspected,http_status_403
6,https://openalex.org/W1546523058,Credit Rationing in Markets with Imperfect Inf...,10.7916/d8v12ft1,1981.0,12858,0.003846,0.378509,green,https://www.jstor.org/stable/pdfplus/1802787.pdf,"[""https://doi.org/10.7916/d8v12ft1"", ""https://...",False,None,None,None,0,None,paywall_suspected,too_small<15000
7,https://openalex.org/W2140358882,Corporate Financing and Investment Decisions W...,10.3386/w1396,1984.0,6788,0.015385,0.365485,gold,https://doi.org/10.3386/w1396,"[""https://doi.org/10.3386/w1396"", ""http://hdl....",True,https://doi.org/10.3386/w1396,https://www.nber.org/system/files/working_pape...,/Users/yuetoya/Desktop/researchOS100-private/n...,606715,550c9143b5808aa99380e534f64cb479cb3d8e5a8db59c...,None,None


💾 Saved: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/pdf_download_results.csv
📊 Success rate: 12.5 %
Top failure codes:


final_error_code
paywall_suspected    4
not_pdf              2
fetch_failed         1
Name: count, dtype: int64

✅ Sample successful downloads:


,title,doi,bytes,local_path,chosen_url
7,Corporate Financing and Investment Decisions W...,10.3386/w1396,606715,/Users/yuetoya/Desktop/researchOS100-private/n...,https://doi.org/10.3386/w1396


🔍 Sample URL candidates (first 5 papers):
--------------------------------------------------------------------------------
title: The Nature of the Firm
cands: ['https://onlinelibrary.wiley.com/doi/full/10.1111/j.1468-0335.1937.tb00002.x', 'https://doi.org/10.1111/j.1468-0335.1937.tb00002.x']
--------------------------------------------------------------------------------
title: Self-efficacy: Toward a unifying theory of behavioral change.
cands: ['https://doaj.org/article/ccb9da1c287e4fc889803ed90e077385', 'https://doi.org/10.1037/0033-295x.84.2.191']
--------------------------------------------------------------------------------
title: Investor protection and corporate governance
cands: ['http://nrs.harvard.edu/urn-3:HUL.InstRepos:29408126', 'https://doi.org/10.1016/s0304-405x(00)00065-9']
--------------------------------------------------------------------------------
title: Venture capital and the structure of capital markets: banks versus stock markets
cands: ['https://doi.org/10

In [45]:
# ============================================================
# 9. Persistence to Drive and upsert into the Notion database
# (Revised: index-safe, NaN-safe, idempotent, and restart-friendly)
#
# Key fixes vs previous version:
# - Never writes download results back by positional index (prevents "nan rows" bug)
# - Filters junk rows early (missing openalex_wid/title/oa_url)
# - Normalizes required columns (year/authors/landing_url/paper_id)
# - Downloads only from valid URLs and writes back by stable key (openalex_wid or doi)
# - Persistence loop skips rows without a local PDF (gated)
# ============================================================

import os
import re
import io
import json
import time
import difflib
import pathlib
from datetime import datetime
from typing import Any, Optional, List, Dict

import requests
import pandas as pd
from googleapiclient.http import MediaFileUpload


# ------------------------------------------------------------
# 9.A Small utilities
# ------------------------------------------------------------
def _safe_str(x: Any) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x).strip()
    return "" if s.lower() in ("nan", "none") else s

def is_valid_url(x: Any) -> bool:
    s = _safe_str(x)
    return s.startswith("http://") or s.startswith("https://")

def _safe_filename(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s or "")).strip()
    s = re.sub(r"[^a-zA-Z0-9\-\._\(\) ]", "_", s)
    s = s.strip().replace(" ", "_")
    return s[:160] if len(s) > 160 else s


# ============================================================
# Pre-Cell for Section 9: Download PDFs locally (oa_url -> local_pdf_path)
# ============================================================

# -----------------------------
# Config
# -----------------------------
PDF_DOWNLOAD_DIR = pathlib.Path(os.getenv("PDF_DOWNLOAD_DIR", str(ARTIFACTS_DIR / "pdfs"))).resolve()
PDF_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

PDF_DL_TIMEOUT = int(os.getenv("PDF_DL_TIMEOUT", "60"))
PDF_DL_MAX_RETRIES = int(os.getenv("PDF_DL_MAX_RETRIES", "4"))
PDF_DL_SLEEP = float(os.getenv("PDF_DL_SLEEP", "0.3"))
PDF_DL_MAX_MB = int(os.getenv("PDF_DL_MAX_MB", "80"))  # safety cap
DL_LIMIT = int(os.getenv("DL_LIMIT", "30"))            # set smaller while testing

UA = os.getenv("PDF_DL_USER_AGENT", "researchOS/1.0 (+pdf-fetch)")
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": UA})


def _is_pdf_response(resp: requests.Response) -> bool:
    ctype = (resp.headers.get("Content-Type") or "").lower()
    return ("application/pdf" in ctype) or (resp.url.lower().endswith(".pdf"))

def _content_length_ok(resp: requests.Response, max_mb: int) -> bool:
    cl = resp.headers.get("Content-Length")
    if not cl:
        return True
    try:
        n = int(cl)
        return n <= max_mb * 1024 * 1024
    except Exception:
        return True

def download_pdf(url: str, out_path: pathlib.Path, timeout: int = 60, max_retries: int = 4) -> Dict[str, Any]:
    """
    Download a PDF via GET (stream).
    Returns dict:
      status: downloaded | skipped_exists | failed
      reason, final_url, bytes
    """
    if out_path.exists() and out_path.stat().st_size > 10_000:
        return {"status": "skipped_exists", "reason": None, "final_url": url, "bytes": out_path.stat().st_size}

    last_err = None
    for i in range(max_retries):
        try:
            r = SESSION.get(url, stream=True, allow_redirects=True, timeout=timeout)

            if r.status_code >= 400:
                last_err = f"http_{r.status_code}"
                time.sleep(0.8 + 0.4 * i)
                continue

            if not _content_length_ok(r, PDF_DL_MAX_MB):
                return {"status": "failed", "reason": "too_large_by_content_length", "final_url": r.url, "bytes": 0}

            if not _is_pdf_response(r):
                return {
                    "status": "failed",
                    "reason": f"not_pdf_content_type:{r.headers.get('Content-Type')}",
                    "final_url": r.url,
                    "bytes": 0,
                }

            out_path.parent.mkdir(parents=True, exist_ok=True)
            nbytes = 0
            max_bytes = PDF_DL_MAX_MB * 1024 * 1024

            with open(out_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 256):
                    if not chunk:
                        continue
                    f.write(chunk)
                    nbytes += len(chunk)
                    if nbytes > max_bytes:
                        try:
                            out_path.unlink(missing_ok=True)
                        except Exception:
                            pass
                        return {"status": "failed", "reason": "too_large_stream", "final_url": r.url, "bytes": nbytes}

            if out_path.exists() and out_path.stat().st_size > 10_000:
                return {"status": "downloaded", "reason": None, "final_url": r.url, "bytes": out_path.stat().st_size}

            return {
                "status": "failed",
                "reason": "downloaded_but_too_small",
                "final_url": r.url,
                "bytes": out_path.stat().st_size if out_path.exists() else 0,
            }

        except Exception as e:
            last_err = f"{type(e).__name__}: {str(e)[:160]}"
            time.sleep(0.8 + 0.4 * i)

    return {"status": "failed", "reason": last_err or "unknown", "final_url": url, "bytes": 0}


# ------------------------------------------------------------
# 9.B Normalize / clean targets_df (prevents NaN rows and missing cols)
# ------------------------------------------------------------
targets_df = targets_df.copy()

# Drop junk rows early (common source of "nan" paper_id)
# Keep only rows with at least a title or openalex id.
keep_mask = True
if "openalex_wid" in targets_df.columns:
    keep_mask = keep_mask & targets_df["openalex_wid"].notna()
if "title" in targets_df.columns:
    keep_mask = keep_mask & targets_df["title"].notna()
targets_df = targets_df[keep_mask].copy()
targets_df = targets_df.reset_index(drop=True)

# year
if "year" not in targets_df.columns or targets_df["year"].isna().all():
    if "publication_year" in targets_df.columns:
        targets_df["year"] = pd.to_numeric(targets_df["publication_year"], errors="coerce")

# authors (best-effort)
if "authors" not in targets_df.columns or targets_df["authors"].astype(str).str.strip().eq("").all():
    for cand in ["author_names", "authors_str", "authors_list"]:
        if cand in targets_df.columns:
            targets_df["authors"] = targets_df[cand]
            break
if "authors" not in targets_df.columns:
    targets_df["authors"] = ""

# venue
if "venue" not in targets_df.columns:
    targets_df["venue"] = ""

# landing_url (for Notion matching)
if "landing_url" not in targets_df.columns:
    for cand in ["url", "openalex_id", "oa_url"]:
        if cand in targets_df.columns:
            targets_df["landing_url"] = targets_df[cand]
            break
if "landing_url" not in targets_df.columns:
    targets_df["landing_url"] = ""

# paper_id (stable)
if "paper_id" not in targets_df.columns or targets_df["paper_id"].astype(str).str.strip().eq("").all():
    if "openalex_wid" in targets_df.columns:
        targets_df["paper_id"] = targets_df["openalex_wid"].apply(lambda x: f"openalex:{x}" if _safe_str(x) else "")
    elif "doi" in targets_df.columns:
        targets_df["paper_id"] = targets_df["doi"].apply(lambda x: f"doi:{x}" if _safe_str(x) else "")
    else:
        targets_df["paper_id"] = ""

# Ensure columns exist (do not overwrite if already present)
for c in ["local_pdf_path", "pdf_download_status", "pdf_download_reason", "pdf_final_url"]:
    if c not in targets_df.columns:
        targets_df[c] = ""

# Use openalex_wid as the write-back key if present, else doi
KEY_COL = "openalex_wid" if "openalex_wid" in targets_df.columns else ("doi" if "doi" in targets_df.columns else None)
if KEY_COL is None:
    raise ValueError("targets_df must have openalex_wid or doi to safely write back download results.")


def build_local_pdf_path(row: pd.Series) -> pathlib.Path:
    wid = _safe_str(row.get("openalex_wid"))
    if wid:
        fname = f"{wid}.pdf"
    else:
        fname = _safe_filename(_safe_str(row.get("title")) or "unknown") + ".pdf"
    return PDF_DOWNLOAD_DIR / fname

# Build local paths (for all targets)
targets_df["local_pdf_path"] = targets_df.apply(lambda r: str(build_local_pdf_path(r)), axis=1)

# Mark valid URL rows
if "oa_url" not in targets_df.columns:
    targets_df["oa_url"] = ""
targets_df["_oa_url_valid"] = targets_df["oa_url"].apply(is_valid_url)

# Only attempt downloads for valid oa_url
dl_df = targets_df[targets_df["_oa_url_valid"]].copy()
dl_df = dl_df.reset_index(drop=True)

print("📥 Downloading PDFs locally from oa_url ...")
print(f"[INFO] targets_df={len(targets_df)} | dl_df(valid oa_url)={len(dl_df)}")

ok, fail = 0, 0
run_n = min(len(dl_df), DL_LIMIT)

for j in range(run_n):
    row = dl_df.iloc[j]
    key = row.get(KEY_COL)
    url = _safe_str(row.get("oa_url"))
    out_path = pathlib.Path(_safe_str(row.get("local_pdf_path")))

    if not url:
        # write back by key
        m = (targets_df[KEY_COL] == key)
        targets_df.loc[m, "pdf_download_status"] = "failed"
        targets_df.loc[m, "pdf_download_reason"] = "missing_oa_url"
        fail += 1
        continue

    res = download_pdf(url, out_path, timeout=PDF_DL_TIMEOUT, max_retries=PDF_DL_MAX_RETRIES)

    # write back by key (NOT by position!)
    m = (targets_df[KEY_COL] == key)
    targets_df.loc[m, "pdf_download_status"] = res["status"]
    targets_df.loc[m, "pdf_download_reason"] = res.get("reason") or ""
    targets_df.loc[m, "pdf_final_url"] = res.get("final_url") or ""

    if res["status"] in ("downloaded", "skipped_exists"):
        ok += 1
    else:
        fail += 1

    print(f"[{j+1}/{run_n}] {key} | {res['status']} | reason={res.get('reason')}")
    time.sleep(PDF_DL_SLEEP)

print(f"✅ Local PDF download finished: ok={ok}, fail={fail}, dir={PDF_DOWNLOAD_DIR}")

display_cols = ["openalex_wid", "title", "oa_status", "oa_url", "pdf_download_status", "pdf_download_reason", "local_pdf_path"]
display(targets_df[[c for c in display_cols if c in targets_df.columns]].head(30))

# Save a log artifact
dl_log_path = ARTIFACTS_DIR / "pdf_download_log.csv"
targets_df.to_csv(dl_log_path, index=False)
print(f"💾 Saved: {dl_log_path}")


# ------------------------------------------------------------
# Pre-Cell: Build Google Drive service (drive_service)
# ------------------------------------------------------------
import pathlib
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

def build_drive_service(client_secret_path: str, token_cache_path: str, scopes: list[str]):
    creds = None
    token_path = pathlib.Path(token_cache_path)

    if token_path.exists():
        creds = Credentials.from_authorized_user_file(str(token_path), scopes)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(str(client_secret_path), scopes)
            creds = flow.run_local_server(port=0)
        token_path.write_text(creds.to_json())

    return build("drive", "v3", credentials=creds, cache_discovery=False)

drive_service = build_drive_service(
    client_secret_path=str(client_secret_path),
    token_cache_path=str(token_cache_path),
    scopes=DRIVE_SCOPES,
)

print("✅ drive_service initialized.")
print("📁 DRIVE_FOLDER_ID:", DRIVE_FOLDER_ID)


# ------------------------------------------------------------
# 9.0 Config knobs (override via env.txt)
# ------------------------------------------------------------
NOTION_DB_ID = NOTION_LIT_DB_ID

DRIVE_LIST_PAGE_SIZE = int(os.getenv("DRIVE_LIST_PAGE_SIZE", "200"))
DRIVE_LIST_MAX = int(os.getenv("DRIVE_LIST_MAX", "5000"))

K_PREFILTER = int(os.getenv("K_PREFILTER", "15"))
PREFILTER_CUTOFF = float(os.getenv("PREFILTER_CUTOFF", "0.60"))

LLM_DUPLICATE_ENABLED = os.getenv("LLM_DUPLICATE_ENABLED", "true").lower() in ("1", "true", "yes", "y")
LLM_DUPLICATE_MODEL = os.getenv("LLM_DUPLICATE_MODEL", "gpt-4o-mini")
LLM_MAX_CANDIDATES_SENT = int(os.getenv("LLM_MAX_CANDIDATES_SENT", "10"))
LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.0"))
LLM_CONFIDENCE_THRESHOLD = float(os.getenv("LLM_CONFIDENCE_THRESHOLD", "0.85"))

DRIVE_UPLOAD_CHUNK_MB = int(os.getenv("DRIVE_UPLOAD_CHUNK_MB", "10"))
DRIVE_ALWAYS_UPLOAD_IF_NO_LLM = os.getenv("DRIVE_ALWAYS_UPLOAD_IF_NO_LLM", "true").lower() in ("1", "true", "yes", "y")

NOTION_TIMEOUT_SEC = float(os.getenv("NOTION_TIMEOUT_SEC", "30"))
NOTION_LLM_ENRICH = os.getenv("NOTION_LLM_ENRICH", "true").lower() in ("1", "true", "yes", "y")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_TEMPERATURE = float(os.getenv("OPENAI_TEMPERATURE", "0.2"))

PDF_TEXT_MAX_CHARS = int(os.getenv("PDF_TEXT_MAX_CHARS", "120000"))
PDF_TEXT_MAX_PAGES = int(os.getenv("PDF_TEXT_MAX_PAGES", "20"))

PREFERRED_MATCH_KEYS = [x.strip() for x in os.getenv("PREFERRED_MATCH_KEYS", "paper_id,doi,landing_url").split(",") if x.strip()]

print("=== Persistence Config ===")
print("Drive: LIST_MAX", DRIVE_LIST_MAX, "| K_PREFILTER", K_PREFILTER, "| LLM_DUPLICATE_ENABLED", LLM_DUPLICATE_ENABLED)
print("Notion: NOTION_DB_ID", NOTION_DB_ID, "| NOTION_LLM_ENRICH", NOTION_LLM_ENRICH)
print("Match keys:", PREFERRED_MATCH_KEYS)
print("=========================\n")


# ------------------------------------------------------------
# 9.1 Google Drive: list PDFs (paged)
# ------------------------------------------------------------
def drive_list_pdfs_in_folder(service, folder_id: str, max_items: int = 5000) -> pd.DataFrame:
    files: List[Dict[str, Any]] = []
    page_token = None
    fetched = 0
    q = f"'{folder_id}' in parents and trashed=false and mimeType='application/pdf'"

    while True:
        resp = service.files().list(
            q=q,
            pageSize=DRIVE_LIST_PAGE_SIZE,
            pageToken=page_token,
            fields="nextPageToken,files(id,name,mimeType,md5Checksum,size,modifiedTime,createdTime,webViewLink)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()

        batch = resp.get("files", [])
        files.extend(batch)
        fetched += len(batch)

        page_token = resp.get("nextPageToken")
        if not page_token or fetched >= max_items:
            break

    df = pd.DataFrame(files)
    if len(df) == 0:
        return df
    if "size" in df.columns:
        df["size"] = pd.to_numeric(df["size"], errors="coerce").fillna(0).astype(int)
    return df

drive_files_df = drive_list_pdfs_in_folder(drive_service, DRIVE_FOLDER_ID, max_items=DRIVE_LIST_MAX)
print(f"✅ Drive PDFs listed: {len(drive_files_df):,}")
display(drive_files_df.head(10))


# ------------------------------------------------------------
# 9.2 Drive dedupe helpers
# ------------------------------------------------------------
def _clean_filename_text(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s or "")).strip()
    s = s.replace("/", "-").replace("\\", "-").replace(":", "-").replace("|", "-")
    return s

def format_author_citation(authors: str) -> str:
    a = _safe_str(authors)
    if not a:
        return "Unknown"
    first = a.split(",")[0].strip()
    parts = first.split()
    if len(parts) >= 2:
        last = parts[-1].strip()
        first_initial = parts[0].strip()[0].upper()
        return f"{last}, {first_initial}."
    return first

def format_drive_filename(row: pd.Series, max_len: int = 180) -> str:
    author_part = format_author_citation(row.get("authors", ""))
    y = row.get("year")
    year_str = str(int(y)) if pd.notna(y) else "n.d."
    title = _clean_filename_text(_safe_str(row.get("title")))
    venue = _clean_filename_text(_safe_str(row.get("venue")))

    base = f"{author_part} ({year_str}). {title}."
    if venue:
        base += f" {venue}"
    base = base.strip()
    base = base[:max_len].rstrip(" .")
    return f"{base}.pdf"

def normalize_for_match(s: str) -> str:
    s = _safe_str(s).lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s\(\)\.\-]", "", s)
    return s

def build_candidate_signature(row: pd.Series) -> str:
    author = _safe_str(row.get("authors"))
    y = row.get("year")
    year_str = str(int(y)) if pd.notna(y) else ""
    title = _safe_str(row.get("title"))
    venue = _safe_str(row.get("venue"))
    sig = f"{author} {year_str} {title} {venue}"
    return normalize_for_match(sig)

def build_drive_signature(filename: str) -> str:
    return normalize_for_match(filename)

if len(drive_files_df):
    drive_files_df["sig"] = drive_files_df["name"].apply(build_drive_signature)
else:
    drive_files_df["sig"] = pd.Series(dtype=str)

targets_df["sig"] = targets_df.apply(build_candidate_signature, axis=1)

def prefilter_candidates_for_row(row: pd.Series, drive_df: pd.DataFrame, k: int = 15, cutoff: float = 0.6) -> pd.DataFrame:
    if len(drive_df) == 0:
        return drive_df.iloc[0:0]
    sig = row.get("sig") or ""
    choices = drive_df["sig"].tolist()
    matched = difflib.get_close_matches(sig, choices, n=k, cutoff=cutoff)
    if not matched:
        return drive_df.iloc[0:0]
    out = drive_df[drive_df["sig"].isin(matched)].copy()
    y = row.get("year")
    if pd.notna(y):
        y_str = str(int(y))
        out["year_hint"] = out["name"].astype(str).str.contains(y_str, regex=False).astype(int)
        out = out.sort_values(["year_hint", "modifiedTime"], ascending=[False, False])
    return out.head(k)


# ------------------------------------------------------------
# 9.3 Optional LLM duplicate judgement (ChatGPT/OpenAI)
# ------------------------------------------------------------
def build_duplicate_prompt(paper_row: pd.Series, drive_candidates: pd.DataFrame) -> str:
    paper = {
        "title": _safe_str(paper_row.get("title")),
        "authors": _safe_str(paper_row.get("authors")),
        "year": int(paper_row["year"]) if pd.notna(paper_row.get("year")) else None,
        "venue": _safe_str(paper_row.get("venue")),
        "doi": _safe_str(paper_row.get("doi")),
    }
    cand_list = []
    for _, r in drive_candidates.iterrows():
        cand_list.append({
            "id": r.get("id"),
            "name": r.get("name"),
            "size": r.get("size"),
            "modifiedTime": r.get("modifiedTime"),
        })

    return f"""
You are a strict deduplication assistant.

Decide whether the same scholarly work already exists in the Drive folder.
Only mark duplicate if it is clearly the SAME paper (same title/year; authors consistent).
If uncertain, return is_duplicate=false.

Return ONLY valid JSON:
{{
  "is_duplicate": boolean,
  "best_match_file_id": string|null,
  "confidence": number (0..1),
  "reason": string
}}

Target paper:
{json.dumps(paper, ensure_ascii=False)}

Existing Drive candidates:
{json.dumps(cand_list, ensure_ascii=False)}
""".strip()

def call_openai_json(prompt: str) -> Dict[str, Any]:
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY is not set")
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    r = client.chat.completions.create(
        model=LLM_DUPLICATE_MODEL,
        temperature=LLM_TEMPERATURE,
        messages=[
            {"role": "system", "content": "You return strict JSON only."},
            {"role": "user", "content": prompt},
        ],
    )
    text = (r.choices[0].message.content or "").strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not m:
            raise ValueError(f"LLM did not return JSON. Got: {text[:200]}")
        return json.loads(m.group(0))

def judge_duplicate_with_llm(paper_row: pd.Series, drive_df: pd.DataFrame) -> Dict[str, Any]:
    cand_df = prefilter_candidates_for_row(paper_row, drive_df, k=K_PREFILTER, cutoff=PREFILTER_CUTOFF)
    if len(cand_df) == 0:
        return {"is_duplicate": False, "best_match_file_id": None, "confidence": 0.0, "reason": "no_prefilter_hits"}
    cand_df = cand_df.head(LLM_MAX_CANDIDATES_SENT)
    out = call_openai_json(build_duplicate_prompt(paper_row, cand_df))
    return {
        "is_duplicate": bool(out.get("is_duplicate", False)),
        "best_match_file_id": out.get("best_match_file_id", None),
        "confidence": float(out.get("confidence", 0.0) or 0.0),
        "reason": str(out.get("reason", ""))[:200],
    }


# ------------------------------------------------------------
# 9.4 Drive upload (with dedupe)
# ------------------------------------------------------------
def drive_get_file_metadata(service, file_id: str) -> Dict[str, Any]:
    return service.files().get(
        fileId=file_id,
        fields="id,name,webViewLink,md5Checksum,size,modifiedTime",
        supportsAllDrives=True,
    ).execute()

def drive_upload_pdf_file(service, folder_id: str, local_path: str, filename: str) -> Dict[str, Any]:
    file_metadata = {"name": filename, "parents": [folder_id]}
    media = MediaFileUpload(
        local_path,
        mimetype="application/pdf",
        resumable=True,
        chunksize=DRIVE_UPLOAD_CHUNK_MB * 1024 * 1024,
    )
    created = service.files().create(
        body=file_metadata,
        media_body=media,
        fields="id,name,webViewLink,md5Checksum",
        supportsAllDrives=True,
    ).execute()
    return created

def drive_upload_pdf_with_dedupe(
    service,
    folder_id: str,
    local_path: str,
    paper_row: pd.Series,
    drive_df: pd.DataFrame,
    llm_enabled: bool = True,
    confidence_threshold: float = 0.85,
) -> Dict[str, Any]:
    p = pathlib.Path(_safe_str(local_path))
    if not p.exists():
        return {"status": "failed", "drive_file_id": None, "drive_link": None, "filename": None, "reason": "local_pdf_not_found"}

    filename = format_drive_filename(paper_row)

    # LLM duplicate check
    if llm_enabled and LLM_DUPLICATE_ENABLED and len(drive_df) > 0 and OPENAI_API_KEY:
        try:
            j = judge_duplicate_with_llm(paper_row, drive_df)
            if j["is_duplicate"] and j["best_match_file_id"] and j["confidence"] >= confidence_threshold:
                meta = drive_get_file_metadata(service, j["best_match_file_id"])
                return {
                    "status": "skipped_duplicate",
                    "drive_file_id": meta.get("id"),
                    "drive_link": meta.get("webViewLink"),
                    "filename": meta.get("name"),
                    "reason": f"Duplicate by LLM (conf={j['confidence']:.2f}): {j['reason']}",
                }
        except Exception as e:
            if not DRIVE_ALWAYS_UPLOAD_IF_NO_LLM:
                return {
                    "status": "failed",
                    "drive_file_id": None,
                    "drive_link": None,
                    "filename": filename,
                    "reason": f"llm_failed: {type(e).__name__}: {str(e)[:180]}",
                }

    # Upload
    try:
        created = drive_upload_pdf_file(service, folder_id, str(p), filename)

        # update drive listing
        new_row = {
            "id": created.get("id"),
            "name": created.get("name"),
            "mimeType": "application/pdf",
            "md5Checksum": created.get("md5Checksum"),
            "size": p.stat().st_size,
            "modifiedTime": datetime.utcnow().isoformat() + "Z",
            "createdTime": datetime.utcnow().isoformat() + "Z",
            "webViewLink": created.get("webViewLink"),
        }
        global drive_files_df
        drive_files_df = pd.concat([drive_files_df, pd.DataFrame([new_row])], ignore_index=True)
        drive_files_df["sig"] = drive_files_df["name"].apply(build_drive_signature)

        return {
            "status": "uploaded",
            "drive_file_id": created.get("id"),
            "drive_link": created.get("webViewLink"),
            "filename": created.get("name"),
            "reason": None,
        }
    except Exception as e:
        return {"status": "failed", "drive_file_id": None, "drive_link": None, "filename": filename, "reason": f"{type(e).__name__}: {str(e)[:200]}"}


# ------------------------------------------------------------
# 9.5 Notion helpers (schema-aware upsert)
# ------------------------------------------------------------
def notion_get_database_schema(db_id: str) -> Dict[str, Any]:
    url = f"https://api.notion.com/v1/databases/{db_id}"
    r = requests.get(url, headers=NOTION_HEADERS, timeout=NOTION_TIMEOUT_SEC)
    if r.status_code != 200:
        raise RuntimeError(f"Notion DB read failed: {r.status_code} {r.text[:200]}")
    return r.json()

def notion_query_database(db_id: str, filter_obj: dict) -> dict:
    url = f"https://api.notion.com/v1/databases/{db_id}/query"
    payload = {"filter": filter_obj, "page_size": 10}
    r = requests.post(url, headers=NOTION_HEADERS, json=payload, timeout=NOTION_TIMEOUT_SEC)
    if r.status_code != 200:
        raise RuntimeError(f"Notion query failed: {r.status_code} {r.text[:200]}")
    return r.json()

def notion_create_page(db_id: str, properties: dict) -> dict:
    url = "https://api.notion.com/v1/pages"
    payload = {"parent": {"database_id": db_id}, "properties": properties}
    r = requests.post(url, headers=NOTION_HEADERS, json=payload, timeout=NOTION_TIMEOUT_SEC)
    if r.status_code != 200:
        raise RuntimeError(f"Notion create failed: {r.status_code} {r.text[:200]}")
    return r.json()

def notion_update_page(page_id: str, properties: dict) -> dict:
    url = f"https://api.notion.com/v1/pages/{page_id}"
    payload = {"properties": properties}
    r = requests.patch(url, headers=NOTION_HEADERS, json=payload, timeout=NOTION_TIMEOUT_SEC)
    if r.status_code != 200:
        raise RuntimeError(f"Notion update failed: {r.status_code} {r.text[:200]}")
    return r.json()

def _to_text(x: Any) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    if isinstance(x, str):
        return x
    if isinstance(x, list):
        items = [str(v).strip() for v in x if v is not None and str(v).strip()]
        return "\n".join([f"- {i}" for i in items]) if items else ""
    if isinstance(x, dict):
        return json.dumps(x, ensure_ascii=False)
    return str(x)

def rt(s: Any):
    s2 = _to_text(s).strip()
    return {"rich_text": [{"type": "text", "text": {"content": s2}}]} if s2 else {"rich_text": []}

def title_prop(s: Any):
    s2 = _to_text(s).strip() or "Untitled Paper"
    return {"title": [{"type": "text", "text": {"content": s2}}]}

def ms(options):
    options = options or []
    cleaned = [{"name": str(x).strip()} for x in options if str(x).strip()]
    return {"multi_select": cleaned}

def url_prop(s: Optional[str]):
    s = _safe_str(s)
    return {"url": s if s else None}

db_schema = notion_get_database_schema(NOTION_DB_ID)
DB_PROPERTIES = db_schema.get("properties", {}) or {}
DB_PROP_NAMES = set(DB_PROPERTIES.keys())

PROPERTY_NAME_CANDIDATES = {
    "Name": ["Name", "Title", "Paper Title"],
    "Authors & Year": ["Authors & Year", "Authors", "Author(s)", "Authors Year"],
    "Source": ["Source", "Venue", "Journal"],
    "DOI": ["DOI", "Doi"],
    "Landing URL": ["Landing URL", "URL", "Link"],
    "PDF Link": ["PDF Link", "PDF", "PDF URL", "Drive Link"],
    "Paper ID": ["Paper ID", "paper_id", "ID", "Key"],
    "Notes": ["Notes", "Note", "Memo"],
    "Tags": ["Tags", "Tag", "Keywords"],
    "RQ Score": ["RQ Score", "rq_score"],
    "Cited By": ["Cited By", "cited_by_count"],
    "Core Idea": ["Core Idea", "Summary"],
    "Methods": ["Methods", "Method"],
    "Findings": ["Findings", "Key Findings"],
}

def _pick_prop(candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in DB_PROP_NAMES:
            return c
    return None

RESOLVED = {k: _pick_prop(v) for k, v in PROPERTY_NAME_CANDIDATES.items()}
print("✅ Notion property mapping resolved (non-null only):")
for k, v in RESOLVED.items():
    if v:
        print(f"  - {k:12s} -> {v}")

def build_props_safe(fields: dict) -> dict:
    props = {}
    if RESOLVED["Name"]:
        props[RESOLVED["Name"]] = title_prop(fields.get("title"))
    if RESOLVED["Authors & Year"]:
        props[RESOLVED["Authors & Year"]] = rt(fields.get("authors_year"))
    if RESOLVED["Source"]:
        props[RESOLVED["Source"]] = rt(fields.get("venue"))
    if RESOLVED["DOI"]:
        props[RESOLVED["DOI"]] = rt(fields.get("doi"))
    if RESOLVED["Landing URL"]:
        props[RESOLVED["Landing URL"]] = url_prop(fields.get("landing_url"))
    if RESOLVED["PDF Link"]:
        props[RESOLVED["PDF Link"]] = url_prop(fields.get("pdf_link"))
    if RESOLVED["Paper ID"]:
        props[RESOLVED["Paper ID"]] = rt(fields.get("paper_id"))
    if RESOLVED["Notes"]:
        props[RESOLVED["Notes"]] = rt(fields.get("notes"))
    if RESOLVED["Tags"]:
        props[RESOLVED["Tags"]] = ms(fields.get("tags", []))
    if RESOLVED["RQ Score"] and fields.get("rq_score") is not None:
        props[RESOLVED["RQ Score"]] = rt(str(fields.get("rq_score")))
    if RESOLVED["Cited By"] and fields.get("cited_by_count") is not None:
        props[RESOLVED["Cited By"]] = rt(str(int(fields.get("cited_by_count"))))
    if RESOLVED.get("Core Idea") and fields.get("core_idea"):
        props[RESOLVED["Core Idea"]] = rt(fields.get("core_idea"))
    if RESOLVED.get("Methods") and fields.get("methods"):
        props[RESOLVED["Methods"]] = rt(fields.get("methods"))
    if RESOLVED.get("Findings") and fields.get("findings"):
        props[RESOLVED["Findings"]] = rt(fields.get("findings"))
    return props

def _filter_rich_text_contains(prop_name: str, val: str) -> dict:
    return {"property": prop_name, "rich_text": {"contains": val}}

def _filter_url_equals(prop_name: str, val: str) -> dict:
    return {"property": prop_name, "url": {"equals": val}}

def find_existing_page_id(fields: dict) -> Optional[str]:
    checks: List[dict] = []

    if "paper_id" in PREFERRED_MATCH_KEYS and RESOLVED["Paper ID"]:
        v = _safe_str(fields.get("paper_id"))
        if v:
            checks.append(_filter_rich_text_contains(RESOLVED["Paper ID"], v))

    if "doi" in PREFERRED_MATCH_KEYS and RESOLVED["DOI"]:
        v = _safe_str(fields.get("doi"))
        if v:
            checks.append(_filter_rich_text_contains(RESOLVED["DOI"], v))

    if "landing_url" in PREFERRED_MATCH_KEYS and RESOLVED["Landing URL"]:
        v = _safe_str(fields.get("landing_url"))
        if v:
            checks.append(_filter_url_equals(RESOLVED["Landing URL"], v))

    for fobj in checks:
        res = notion_query_database(NOTION_DB_ID, fobj)
        results = res.get("results", [])
        if results:
            return results[0]["id"]
    return None


# ------------------------------------------------------------
# 9.6 PDF text extraction (LOCAL) + optional LLM enrichment
# ------------------------------------------------------------
def _try_import_pymupdf():
    try:
        import fitz  # PyMuPDF
        return fitz
    except Exception:
        return None

def _try_import_pdfplumber():
    try:
        import pdfplumber
        return pdfplumber
    except Exception:
        return None

def _clean_text(s: str) -> str:
    s = s or ""
    s = s.replace("\x00", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def extract_pdf_text_local(pdf_path: str, max_chars: int = PDF_TEXT_MAX_CHARS, max_pages: int = PDF_TEXT_MAX_PAGES) -> Dict[str, Any]:
    p = pathlib.Path(_safe_str(pdf_path))
    if not p.exists():
        return {"ok": False, "text": "", "method": None, "pages_extracted": 0, "char_count": 0, "reason": "pdf_not_found"}

    fitz = _try_import_pymupdf()
    if fitz:
        try:
            doc = fitz.open(str(p))
            chunks = []
            n_pages = min(len(doc), max_pages)
            for i in range(n_pages):
                chunks.append(doc.load_page(i).get_text("text") or "")
                if sum(len(c) for c in chunks) >= max_chars:
                    break
            text = _clean_text("\n".join(chunks))[:max_chars]
            return {"ok": bool(text.strip()), "text": text, "method": "pymupdf", "pages_extracted": n_pages, "char_count": len(text), "reason": None if text.strip() else "empty_text"}
        except Exception:
            pass

    pdfplumber = _try_import_pdfplumber()
    if pdfplumber:
        try:
            chunks = []
            with pdfplumber.open(str(p)) as pdf:
                n_pages = min(len(pdf.pages), max_pages)
                for i in range(n_pages):
                    chunks.append(pdf.pages[i].extract_text() or "")
                    if sum(len(c) for c in chunks) >= max_chars:
                        break
            text = _clean_text("\n".join(chunks))[:max_chars]
            return {"ok": bool(text.strip()), "text": text, "method": "pdfplumber", "pages_extracted": n_pages, "char_count": len(text), "reason": None if text.strip() else "empty_text"}
        except Exception as e:
            return {"ok": False, "text": "", "method": "pdfplumber", "pages_extracted": 0, "char_count": 0, "reason": f"{type(e).__name__}: {str(e)[:160]}"}

    return {"ok": False, "text": "", "method": None, "pages_extracted": 0, "char_count": 0, "reason": "no_pdf_text_lib_available"}

def openai_summarize_to_fields(title_hint: str, text: str) -> Dict[str, Any]:
    if not OPENAI_API_KEY:
        return {"core_idea": "", "methods": "", "findings": "", "tags": []}

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    prompt = f"""
You are a research assistant. Create database-ready notes from the paper text.

Return JSON ONLY with keys:
- core_idea: Japanese, 2–4 sentences
- methods: Japanese bullets (use "- " per line)
- findings: Japanese bullets (use "- " per line)
- tags: English array (2–6 items, Title Case)

Title hint: {title_hint}

Paper text:
{text[:PDF_TEXT_MAX_CHARS]}
""".strip()

    r = client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=OPENAI_TEMPERATURE,
        messages=[
            {"role": "system", "content": "Return strict JSON only."},
            {"role": "user", "content": prompt},
        ],
    )
    out = (r.choices[0].message.content or "").strip()
    try:
        return json.loads(out)
    except Exception:
        m = re.search(r"\{.*\}", out, flags=re.DOTALL)
        return json.loads(m.group(0)) if m else {"core_idea": "", "methods": "", "findings": "", "tags": []}


# ------------------------------------------------------------
# 9.7 Notion upsert orchestrator (with optional LLM enrichment)
# ------------------------------------------------------------
def build_fields_from_row(row: pd.Series, drive_link: Optional[str]) -> dict:
    y = row.get("year")
    if pd.isna(y) or y is None:
        y = row.get("publication_year")
    year_str = str(int(y)) if pd.notna(y) else "n.d."

    authors = _safe_str(row.get("authors"))
    if not authors:
        for cand in ["author_names", "authors_str"]:
            v = row.get(cand)
            if _safe_str(v):
                authors = _safe_str(v)
                break

    authors_year = f"{authors} ({year_str})" if authors else f"(Unknown) ({year_str})"

    return {
        "paper_id": _safe_str(row.get("paper_id")) or _safe_str(row.get("openalex_wid")) or _safe_str(row.get("openalex_id")),
        "title": _safe_str(row.get("title")) or "Untitled Paper",
        "authors_year": authors_year,
        "venue": _safe_str(row.get("venue")),
        "doi": _safe_str(row.get("doi")),
        "landing_url": _safe_str(row.get("landing_url")) or _safe_str(row.get("oa_url")),
        "pdf_link": drive_link,
        "rq_score": row.get("rq_score", None),
        "cited_by_count": row.get("cited_by_count", None),
        "notes": _safe_str(row.get("notes")),
        "tags": [],
        "core_idea": "",
        "methods": "",
        "findings": "",
    }

def notion_upsert(fields: dict) -> Dict[str, Any]:
    props = build_props_safe(fields)
    existing_id = find_existing_page_id(fields)

    if existing_id:
        page = notion_update_page(existing_id, props)
        return {"status": "updated", "page_id": page.get("id"), "page_url": page.get("url")}
    else:
        page = notion_create_page(NOTION_DB_ID, props)
        return {"status": "created", "page_id": page.get("id"), "page_url": page.get("url")}

def notion_upsert_paper(row: pd.Series, drive_link: str, local_pdf_path: str) -> Dict[str, Any]:
    fields = build_fields_from_row(row, drive_link)

    pdf_text_meta = {"ok": False}
    if NOTION_LLM_ENRICH and OPENAI_API_KEY:
        pdf_text_meta = extract_pdf_text_local(local_pdf_path)
        if pdf_text_meta.get("ok"):
            enrich = openai_summarize_to_fields(fields["title"], pdf_text_meta.get("text", ""))
            fields["core_idea"] = enrich.get("core_idea", "") or ""
            fields["methods"] = enrich.get("methods", "") or ""
            fields["findings"] = enrich.get("findings", "") or ""
            fields["tags"] = enrich.get("tags", []) or []

    up = notion_upsert(fields)
    up["pdf_text_meta"] = pdf_text_meta
    return up

# ============================================================
# FIX (fallback): If authors/year missing, extract from LOCAL PDF via OpenAI
#   - Uses extract_pdf_text_local() already defined in your Section 9
#   - Reads only first pages (as configured) and asks the model to return JSON
# ============================================================
import json
import re
import pandas as pd

PDF_META_LLM_MODEL = os.getenv("PDF_META_LLM_MODEL", OPENAI_MODEL)  # e.g., gpt-4o-mini
PDF_META_MAX_CHARS = int(os.getenv("PDF_META_MAX_CHARS", "12000"))  # only send small chunk
PDF_META_MAX_PAGES = int(os.getenv("PDF_META_MAX_PAGES", "2"))      # first 1-2 pages is enough

def openai_extract_bib_from_pdf(title_hint: str, text: str) -> dict:
    """
    Return strict JSON:
      { "authors": "A, B, C", "year": 1984, "confidence": 0.0-1.0, "evidence": "short quote" }
    """
    if not OPENAI_API_KEY:
        return {"authors": "", "year": None, "confidence": 0.0, "evidence": "OPENAI_API_KEY missing"}

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    prompt = f"""
You extract bibliographic metadata from the FIRST PAGE(S) of a paper.

Return JSON ONLY with keys:
- authors: string (comma-separated author names as they appear)
- year: integer or null (publication year; if uncertain, guess from context)
- confidence: number 0..1
- evidence: short snippet (<=120 chars) from the text that supports the extraction

Rules:
- If you cannot find authors, set authors="".
- If you cannot find year, set year=null.
- Do NOT hallucinate. Use evidence.

Title hint:
{title_hint}

PDF text (first pages):
{text[:PDF_META_MAX_CHARS]}
""".strip()

    r = client.chat.completions.create(
        model=PDF_META_LLM_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content": "Return strict JSON only."},
            {"role": "user", "content": prompt},
        ],
    )
    out = (r.choices[0].message.content or "").strip()
    try:
        return json.loads(out)
    except Exception:
        m = re.search(r"\{.*\}", out, flags=re.DOTALL)
        return json.loads(m.group(0)) if m else {"authors": "", "year": None, "confidence": 0.0, "evidence": "parse_failed"}

def backfill_authors_year_from_pdf(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "authors" not in df.columns:
        df["authors"] = ""
    if "year" not in df.columns:
        df["year"] = pd.NA

    # Missing authors OR missing year
    missing = (df["authors"].astype(str).str.strip().eq("") | df["authors"].isna()) | (df["year"].isna())

    # Only rows that have a local pdf
    has_pdf = df["local_pdf_path"].astype(str).str.strip().ne("") & df["local_pdf_path"].notna()
    todo = df[missing & has_pdf].copy()

    print(f"[INFO] PDF-meta backfill targets: {len(todo)}/{len(df)} rows")

    meta_rows = []
    for idx, row in todo.iterrows():
        pdf_path = str(row.get("local_pdf_path") or "").strip()
        title = str(row.get("title") or "").strip()

        # Extract first pages text (small)
        meta = extract_pdf_text_local(pdf_path, max_chars=PDF_META_MAX_CHARS, max_pages=PDF_META_MAX_PAGES)
        if not meta.get("ok"):
            meta_rows.append({
                "idx": idx,
                "authors_pdf": "",
                "year_pdf": None,
                "confidence_pdf": 0.0,
                "evidence_pdf": f"pdf_text_failed:{meta.get('reason')}",
            })
            continue

        bib = openai_extract_bib_from_pdf(title_hint=title, text=meta["text"])
        meta_rows.append({
            "idx": idx,
            "authors_pdf": (bib.get("authors") or "").strip(),
            "year_pdf": bib.get("year", None),
            "confidence_pdf": float(bib.get("confidence", 0.0) or 0.0),
            "evidence_pdf": (bib.get("evidence") or "")[:200],
        })

    meta_df = pd.DataFrame(meta_rows)
    if len(meta_df) == 0:
        return df

    # Merge back
    df = df.merge(meta_df, left_index=True, right_on="idx", how="left").set_index(df.index)

    # Apply fill if confidence is decent (tune threshold)
    TH = float(os.getenv("PDF_META_CONF_THRESHOLD", "0.75"))
    ok = df["confidence_pdf"].fillna(0.0) >= TH

    # Fill authors if missing
    miss_auth = df["authors"].astype(str).str.strip().eq("") | df["authors"].isna()
    df.loc[ok & miss_auth, "authors"] = df.loc[ok & miss_auth, "authors_pdf"].fillna("")

    # Fill year if missing
    miss_year = df["year"].isna()
    df.loc[ok & miss_year, "year"] = pd.to_numeric(df.loc[ok & miss_year, "year_pdf"], errors="coerce")

    print(f"[INFO] Applied PDF-meta fills (threshold={TH})")
    return df

# ---- run fallback (do this before the Notion upsert loop) ----
targets_df = backfill_authors_year_from_pdf(targets_df)
# ============================================================
# 9.8 Run: Local PDF -> Drive -> Notion (skip Notion if Drive failed)
# ============================================================
MAX_PERSIST = int(os.getenv("MAX_PERSIST", "50"))
print(f"🚀 Starting gated persistence run (MAX_PERSIST={MAX_PERSIST})")

OK_LOCAL = {"downloaded", "skipped_exists"}
OK_DRIVE = {"uploaded", "skipped_duplicate"}

results = []
n = min(len(targets_df), MAX_PERSIST)

for i in range(n):
    row = targets_df.iloc[i]
    wid = _safe_str(row.get("openalex_wid"))
    pid = wid or _safe_str(row.get("paper_id")) or _safe_str(row.get("openalex_id")) or f"row_{i}"

    local_pdf_path = _safe_str(row.get("local_pdf_path"))
    dl_status = _safe_str(row.get("pdf_download_status"))

    # ---- Gate 1: local PDF must exist ----
    if (dl_status not in OK_LOCAL) or (not local_pdf_path) or (not pathlib.Path(local_pdf_path).exists()):
        results.append({
            "paper_id": pid,
            "title": _safe_str(row.get("title")),
            "pdf_download_status": dl_status or "missing",
            "drive_status": "skipped_no_local_pdf",
            "drive_link": None,
            "notion_status": "skipped",
            "notion_page_id": None,
            "notion_page_url": None,
            "error": f"skip: local_pdf_not_ready (status={dl_status or 'missing'}, path={local_pdf_path or 'None'})",
        })
        print(f"[{i+1}/{n}] {pid} | DL={dl_status or 'missing'} | Drive=SKIP | Notion=SKIP")
        continue

    # ---- Gate 2: upload to Drive (with dedupe) ----
    try:
        drive_res = drive_upload_pdf_with_dedupe(
            service=drive_service,
            folder_id=DRIVE_FOLDER_ID,
            local_path=local_pdf_path,
            paper_row=row,
            drive_df=drive_files_df,
            llm_enabled=True,
            confidence_threshold=LLM_CONFIDENCE_THRESHOLD,
        )
    except Exception as e:
        drive_res = {
            "status": "failed",
            "drive_file_id": None,
            "drive_link": None,
            "filename": None,
            "reason": f"{type(e).__name__}: {str(e)[:200]}",
        }

    drive_status = drive_res.get("status")

    # ---- Gate 3: only if Drive succeeded, upsert Notion ----
    if drive_status not in OK_DRIVE:
        results.append({
            "paper_id": pid,
            "title": _safe_str(row.get("title")),
            "pdf_download_status": dl_status,
            "drive_status": drive_status,
            "drive_file_id": drive_res.get("drive_file_id"),
            "drive_link": drive_res.get("drive_link"),
            "notion_status": "skipped",
            "notion_page_id": None,
            "notion_page_url": None,
            "error": f"skip: drive_not_ok ({drive_res.get('reason')})",
        })
        print(f"[{i+1}/{n}] {pid} | DL=OK | Drive={drive_status} | Notion=SKIP")
        continue

    # ---- Notion upsert ----
    drive_link = drive_res.get("drive_link")
    try:
        notion_res = notion_upsert_paper(
            row=row,
            drive_link=drive_link,
            local_pdf_path=local_pdf_path,
        )
    except Exception as e:
        notion_res = {"status": "failed", "page_id": None, "page_url": None, "reason": f"{type(e).__name__}: {str(e)[:200]}", "pdf_text_meta": None}

    results.append({
        "paper_id": pid,
        "title": _safe_str(row.get("title")),
        "pdf_download_status": dl_status,
        "drive_status": drive_status,
        "drive_file_id": drive_res.get("drive_file_id"),
        "drive_link": drive_link,
        "notion_status": notion_res.get("status"),
        "notion_page_id": notion_res.get("page_id"),
        "notion_page_url": notion_res.get("page_url"),
        "error": (drive_res.get("reason") or "") if notion_res.get("status") != "failed" else notion_res.get("reason"),
        "pdf_text_ok": (notion_res.get("pdf_text_meta") or {}).get("ok") if isinstance(notion_res.get("pdf_text_meta"), dict) else None,
    })

    print(f"[{i+1}/{n}] {pid} | DL=OK | Drive={drive_status} | Notion={notion_res.get('status')}")

results_df = pd.DataFrame(results)
display(results_df.head(50))

out_path = ARTIFACTS_DIR / "persistence_results_gated.csv"
results_df.to_csv(out_path, index=False)
print(f"💾 Saved: {out_path}")
print("✅ Done: Notion upsert was executed ONLY when Drive succeeded.")


📥 Downloading PDFs locally from oa_url ...
[INFO] targets_df=8 | dl_df(valid oa_url)=8
[1/8] W2015930340 | failed | reason=http_403
[2/8] W4292808503 | failed | reason=not_pdf_content_type:text/html; charset=UTF-8
[3/8] W2162012454 | failed | reason=not_pdf_content_type:text/html;charset=UTF-8
[4/8] W3121865524 | failed | reason=not_pdf_content_type:text/html;charset=UTF-8
[5/8] W2150291618 | failed | reason=http_403
[6/8] W2141951329 | failed | reason=http_403
[7/8] W1546523058 | failed | reason=http_403
[8/8] W2140358882 | skipped_exists | reason=None
✅ Local PDF download finished: ok=1, fail=7, dir=/Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/pdfs


,openalex_wid,title,oa_status,oa_url,pdf_download_status,pdf_download_reason,local_pdf_path
0,W2015930340,The Nature of the Firm,bronze,https://doi.org/10.1111/j.1468-0335.1937.tb000...,failed,http_403,/Users/yuetoya/Desktop/researchOS100-private/n...
1,W4292808503,Self-efficacy: Toward a unifying theory of beh...,green,https://doi.org/10.1037/0033-295x.84.2.191,failed,not_pdf_content_type:text/html; charset=UTF-8,/Users/yuetoya/Desktop/researchOS100-private/n...
2,W2162012454,Investor protection and corporate governance,green,https://doi.org/10.1016/s0304-405x(00)00065-9,failed,not_pdf_content_type:text/html;charset=UTF-8,/Users/yuetoya/Desktop/researchOS100-private/n...
3,W3121865524,Venture capital and the structure of capital m...,hybrid,https://doi.org/10.1016/s0304-405x(97)00045-7,failed,not_pdf_content_type:text/html;charset=UTF-8,/Users/yuetoya/Desktop/researchOS100-private/n...
4,W2150291618,The central role of the propensity score in ob...,bronze,https://academic.oup.com/biomet/article-pdf/70...,failed,http_403,/Users/yuetoya/Desktop/researchOS100-private/n...
5,W2141951329,Building Theories from Case Study Research,gold,https://doi.org/10.5465/amr.1989.4308385,failed,http_403,/Users/yuetoya/Desktop/researchOS100-private/n...
6,W1546523058,Credit Rationing in Markets with Imperfect Inf...,green,https://www.jstor.org/stable/pdfplus/1802787.pdf,failed,http_403,/Users/yuetoya/Desktop/researchOS100-private/n...
7,W2140358882,Corporate Financing and Investment Decisions W...,gold,https://doi.org/10.3386/w1396,skipped_exists,,/Users/yuetoya/Desktop/researchOS100-private/n...


💾 Saved: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/pdf_download_log.csv
✅ drive_service initialized.
📁 DRIVE_FOLDER_ID: 1SygzpVjCuk-_8oHk9XQOponn7T3ZOsgh
=== Persistence Config ===
Drive: LIST_MAX 5000 | K_PREFILTER 15 | LLM_DUPLICATE_ENABLED True
Notion: NOTION_DB_ID 2a98e0e4d16280cbb6cbdcd1ebedee54 | NOTION_LLM_ENRICH True
Match keys: ['paper_id', 'doi', 'landing_url']

✅ Drive PDFs listed: 53


,id,name,mimeType,webViewLink,createdTime,modifiedTime,md5Checksum,size
0,1sirIOcnWIbWx8-w-t8VdFwnyFMUQiIDK,Unknown (1984). Corporate Financing and Invest...,application/pdf,https://drive.google.com/file/d/1sirIOcnWIbWx8...,2026-01-13T04:16:03.765Z,2026-01-13T04:16:03.765Z,95fbeebcb9056238c5de0d28294ff42d,606715
1,1JgeggD6NH3aSqhgR5uKgPMvV9QsX9siF,Unknown (n.d.). Corporate Financing and Invest...,application/pdf,https://drive.google.com/file/d/1JgeggD6NH3aSq...,2026-01-13T01:50:19.203Z,2026-01-13T01:50:19.203Z,95fbeebcb9056238c5de0d28294ff42d,606715
2,1h3aTKERMMp0wtM-x4H9jibkpFDDGEoMa,"Yébenes, M. (2024). Climate change, ESG criter...",application/pdf,https://drive.google.com/file/d/1h3aTKERMMp0wt...,2026-01-12T04:52:34.067Z,2026-01-12T04:52:34.067Z,113913971bdad0ab8fb34e69cbb5e9c5,1063088
3,1gZdLNOQdQdgVYfxtNxzGrj7FguhoPkHI,"Buchner, A. (2025). Does the same investment t...",application/pdf,https://drive.google.com/file/d/1gZdLNOQdQdgVY...,2026-01-12T04:52:10.713Z,2026-01-12T04:52:10.713Z,da8dcff1062ae1475aa2626016fc8bc8,1098254
4,19EXDu4wyllhaR3BN_SBsBz7UVv1VRDur,"Nair, A. (2025). Geographic and Identity-Based...",application/pdf,https://drive.google.com/file/d/19EXDu4wyllhaR...,2026-01-12T04:51:55.837Z,2026-01-12T04:51:55.837Z,2fe43783d988beca67babcb82c4174d1,303384
5,1vD9duskWs_pqxqhwL-s9eAuLxpvGgF8-,"Fan, L. (2024). Mindfulness, self-efficacy, an...",application/pdf,https://drive.google.com/file/d/1vD9duskWs_pqx...,2026-01-12T04:51:11.217Z,2026-01-12T04:51:11.217Z,042f95873ed5e9a953e39b532d751c19,533417
6,1LteOeI-tBW6H7xJ3fEXUJgib7yJYaK19,"Xian-zhou, Z. (2024). The impact and effective...",application/pdf,https://drive.google.com/file/d/1LteOeI-tBW6H7...,2026-01-12T04:50:33.341Z,2026-01-12T04:50:33.341Z,e7b0c691814ee8d7762e7329c5c5afad,584242
7,1C6MtAkf6CHMtW7tK18gBPTh5tioBU0bY,"Li, R. (2023). The evolution of k-shell in syn...",application/pdf,https://drive.google.com/file/d/1C6MtAkf6CHMtW...,2026-01-12T04:50:18.020Z,2026-01-12T04:50:18.020Z,bce7a35ae3ec5212ba181b393c65124e,3749696
8,1qOwWreMaKeKTJWvLIA2GoaAlHahp2RBb,"Arnold, T. (2024). Endowment asset allocations...",application/pdf,https://drive.google.com/file/d/1qOwWreMaKeKTJ...,2026-01-12T04:49:57.200Z,2026-01-12T04:49:57.200Z,cead83e54c61c4567d3319d818da6ad0,1037660
9,1a8Vg4ThFKD5CweSaIqxShHDyleI-19tM,"Ahmed, N. (2025). Symbiotic synergy- How Arbus...",application/pdf,https://drive.google.com/file/d/1a8Vg4ThFKD5Cw...,2026-01-12T04:49:37.988Z,2026-01-12T04:49:37.988Z,63c9fee2a0c3e9e675e8f14560be1c0d,6610516


✅ Notion property mapping resolved (non-null only):
  - Name         -> Name
  - Authors & Year -> Authors & Year
  - Source       -> Source
  - PDF Link     -> PDF Link
  - Notes        -> Notes
  - Tags         -> Tags
  - Core Idea    -> Core Idea
  - Methods      -> Methods
  - Findings     -> Findings
[INFO] PDF-meta backfill targets: 8/8 rows
[INFO] Applied PDF-meta fills (threshold=0.75)
🚀 Starting gated persistence run (MAX_PERSIST=50)
[1/8] W2015930340 | DL=failed | Drive=SKIP | Notion=SKIP
[2/8] W4292808503 | DL=failed | Drive=SKIP | Notion=SKIP
[3/8] W2162012454 | DL=failed | Drive=SKIP | Notion=SKIP
[4/8] W3121865524 | DL=failed | Drive=SKIP | Notion=SKIP
[5/8] W2150291618 | DL=failed | Drive=SKIP | Notion=SKIP
[6/8] W2141951329 | DL=failed | Drive=SKIP | Notion=SKIP
[7/8] W1546523058 | DL=failed | Drive=SKIP | Notion=SKIP
[8/8] W2140358882 | DL=OK | Drive=skipped_duplicate | Notion=created


,paper_id,title,pdf_download_status,drive_status,drive_link,notion_status,notion_page_id,notion_page_url,error,drive_file_id,pdf_text_ok
0,W2015930340,The Nature of the Firm,failed,skipped_no_local_pdf,None,skipped,None,None,"skip: local_pdf_not_ready (status=failed, path...",NaN,NaN
1,W4292808503,Self-efficacy: Toward a unifying theory of beh...,failed,skipped_no_local_pdf,None,skipped,None,None,"skip: local_pdf_not_ready (status=failed, path...",NaN,NaN
2,W2162012454,Investor protection and corporate governance,failed,skipped_no_local_pdf,None,skipped,None,None,"skip: local_pdf_not_ready (status=failed, path...",NaN,NaN
3,W3121865524,Venture capital and the structure of capital m...,failed,skipped_no_local_pdf,None,skipped,None,None,"skip: local_pdf_not_ready (status=failed, path...",NaN,NaN
4,W2150291618,The central role of the propensity score in ob...,failed,skipped_no_local_pdf,None,skipped,None,None,"skip: local_pdf_not_ready (status=failed, path...",NaN,NaN
5,W2141951329,Building Theories from Case Study Research,failed,skipped_no_local_pdf,None,skipped,None,None,"skip: local_pdf_not_ready (status=failed, path...",NaN,NaN
6,W1546523058,Credit Rationing in Markets with Imperfect Inf...,failed,skipped_no_local_pdf,None,skipped,None,None,"skip: local_pdf_not_ready (status=failed, path...",NaN,NaN
7,W2140358882,Corporate Financing and Investment Decisions W...,skipped_exists,skipped_duplicate,https://drive.google.com/file/d/1sirIOcnWIbWx8...,created,2e78e0e4-d162-8162-837f-f9e70eaccaa7,https://www.notion.so/Corporate-Financing-and-...,Duplicate by LLM (conf=0.95): Same title and y...,1sirIOcnWIbWx8-w-t8VdFwnyFMUQiIDK,True


💾 Saved: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/persistence_results_gated.csv
✅ Done: Notion upsert was executed ONLY when Drive succeeded.


In [36]:
# %% [markdown]
# # 10. Summary tables and reports for human review and decision-making
#
# This section assumes you already defined:
# - RUN_TS, PROJECT_ROOT, CACHE_DIR, ARTIFACTS_DIR, LOG_DIR
#
# It will:
# - Load the best available Day20 artifacts from ARTIFACTS_DIR (or fallback to the latest prior run)
# - Generate human review tables (core / top candidates / fetch diagnostics)
# - Export CSV + Markdown reports into the CURRENT run folder (ARTIFACTS_DIR / reports_<ts>/)

# %%
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd

# %%
# --------------------------------------------------
# 10.0 Sanity checks: required globals
# --------------------------------------------------
_required_globals = ["RUN_TS", "PROJECT_ROOT", "ARTIFACTS_DIR"]
missing = [g for g in _required_globals if g not in globals()]
if missing:
    raise RuntimeError(
        f"Section 10 requires globals {missing}. "
        "Define RUN_TS / PROJECT_ROOT / ARTIFACTS_DIR at the top of the notebook first."
    )

PROJECT_ROOT = Path(PROJECT_ROOT)
ARTIFACTS_DIR = Path(ARTIFACTS_DIR)

REPORT_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
REPORT_DIR = ARTIFACTS_DIR / f"reports_{REPORT_TS}"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[INFO] Using ARTIFACTS_DIR: {ARTIFACTS_DIR}")
print(f"[INFO] Writing reports to:  {REPORT_DIR}")

# %%
# --------------------------------------------------
# 10.1 Artifact registry (your Day20 filenames)
# --------------------------------------------------
ARTIFACT_CANDIDATES = {
    "core": [
        "core_openalex_resolved.csv",
        "day20_core_papers.csv",
    ],
    "candidates_scored": [
        "candidate_pool_enriched_scored.csv",
        "candidate_pool_enriched_deduped.csv",
        "candidate_pool_enriched.csv",
        "candidate_pool_raw.csv",
    ],
    "fetch_results": [
        "pdf_download_results.csv",
        "persistence_results_gated.csv",
        "persistence_results.csv",
    ],
    "fetch_queue": [
        "backfill_fetch_queue_top200.csv",
        "backfill_top200.csv",
        "backfill_ranked.csv",
    ],
    # Optional snapshots (if you add later)
    "existing_snapshot": [
        "existing_corpus_snapshot.csv",
        "corpus_with_rq_scores.csv",
        "corpus_rq_scores.csv",
    ],
}

# %%
# --------------------------------------------------
# 10.2 Helpers: locating & loading artifacts
# --------------------------------------------------
def _fmt_bytes(n: int) -> str:
    for unit in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f}{unit}"
        n /= 1024
    return f"{n:.0f}TB"

def _first_existing(base: Path, names: List[str]) -> Optional[Path]:
    for n in names:
        p = base / n
        if p.exists() and p.is_file():
            return p
    return None

def _list_csvs(folder: Path) -> List[Path]:
    if not folder.exists():
        return []
    return sorted([p for p in folder.glob("*.csv") if p.is_file()])

def _pick_latest_prior_day20_run(project_root: Path, exclude_run_ts: str) -> Optional[Path]:
    """
    If current ARTIFACTS_DIR is empty (fresh run), fall back to the latest previous run under:
      <PROJECT_ROOT>/artifacts/day20/<run_ts>/
    """
    day20_root = project_root / "artifacts" / "day20"
    if not day20_root.exists():
        return None

    run_dirs = [d for d in day20_root.iterdir() if d.is_dir() and d.name != exclude_run_ts]
    if not run_dirs:
        return None

    # sort by modified time (descending)
    run_dirs_sorted = sorted(run_dirs, key=lambda p: p.stat().st_mtime, reverse=True)
    return run_dirs_sorted[0]

def load_df_optional(path: Optional[Path], label: str) -> pd.DataFrame:
    if path is None:
        print(f"[WARN] Missing artifact for '{label}' (no matching file found).")
        return pd.DataFrame()
    try:
        df = pd.read_csv(path)
        st = path.stat()
        mtime = datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        print(f"[OK] Loaded {label}: {path.name} ({len(df):,} rows, {_fmt_bytes(st.st_size)}, mtime={mtime})")
        return df
    except Exception as e:
        print(f"[WARN] Failed to read {label} from {path}: {e}")
        return pd.DataFrame()

def ensure_col(df: pd.DataFrame, col: str, default=None) -> pd.DataFrame:
    if col not in df.columns:
        df[col] = default
    return df

def pct(n: int, d: int) -> float:
    return round((n / d * 100.0), 2) if d else 0.0

def topn(df: pd.DataFrame, n: int, by: str, asc: bool = False) -> pd.DataFrame:
    if df.empty:
        return df
    if by not in df.columns:
        return df.head(n)
    return df.sort_values(by, ascending=asc).head(n)

# %%
# --------------------------------------------------
# 10.3 Decide input folder: current run or fallback
# --------------------------------------------------
current_csvs = _list_csvs(ARTIFACTS_DIR)
if current_csvs:
    INPUT_DIR = ARTIFACTS_DIR
    print(f"[PICK] Using CURRENT run artifacts: {INPUT_DIR} ({len(current_csvs)} CSVs)")
else:
    fallback = _pick_latest_prior_day20_run(PROJECT_ROOT, exclude_run_ts=RUN_TS)
    if fallback is None:
        INPUT_DIR = ARTIFACTS_DIR  # nothing else to use
        print("[PICK] No prior run found. Using CURRENT run artifacts (may be empty).")
    else:
        INPUT_DIR = fallback
        print(f"[PICK] CURRENT run folder is empty → fallback to latest prior run: {INPUT_DIR}")

print(f"[INFO] INPUT_DIR = {INPUT_DIR}")

# %%
# --------------------------------------------------
# 10.4 Load best available artifacts from INPUT_DIR
# --------------------------------------------------
core_path = _first_existing(INPUT_DIR, ARTIFACT_CANDIDATES["core"])
cand_path = _first_existing(INPUT_DIR, ARTIFACT_CANDIDATES["candidates_scored"])
fetch_path = _first_existing(INPUT_DIR, ARTIFACT_CANDIDATES["fetch_results"])
queue_path = _first_existing(INPUT_DIR, ARTIFACT_CANDIDATES["fetch_queue"])
existing_path = _first_existing(INPUT_DIR, ARTIFACT_CANDIDATES["existing_snapshot"])

core_df = load_df_optional(core_path, "core_df")
candidate_df = load_df_optional(cand_path, "candidate_df")
fetch_df = load_df_optional(fetch_path, "fetch_df")
fetch_queue_df = load_df_optional(queue_path, "fetch_queue_df")
existing_df = load_df_optional(existing_path, "existing_df")

# %%
# --------------------------------------------------
# 10.5 Normalize schemas (best-effort)
# --------------------------------------------------
# Candidates
for c in ["paper_uid", "title", "year", "venue", "doi", "openalex_id", "oa_url",
          "cited_by_count", "rq_score", "priority_score", "score_explain", "hop", "source_core_id"]:
    candidate_df = ensure_col(candidate_df, c, None)

# Core
for c in ["paper_uid", "title", "year", "doi", "openalex_id", "cited_by_count", "rq_score"]:
    core_df = ensure_col(core_df, c, None)

# Fetch results
# (pdf_download_results.csv / persistence_results*.csv may have different column names)
fetch_df = ensure_col(fetch_df, "fetch_status", None)
fetch_df = ensure_col(fetch_df, "error_type", None)
fetch_df = ensure_col(fetch_df, "paper_uid", None)
fetch_df = ensure_col(fetch_df, "title", None)
fetch_df = ensure_col(fetch_df, "pdf_url", None)

# If your files use different column names, map them here:
rename_map = {}
if "status" in fetch_df.columns and fetch_df["fetch_status"].isna().all():
    rename_map["status"] = "fetch_status"
if "download_status" in fetch_df.columns and fetch_df["fetch_status"].isna().all():
    rename_map["download_status"] = "fetch_status"
if "error" in fetch_df.columns and fetch_df["error_type"].isna().all():
    rename_map["error"] = "error_type"

if rename_map:
    fetch_df = fetch_df.rename(columns=rename_map)

# %%
# --------------------------------------------------
# 10.6 Executive overview
# --------------------------------------------------
def build_run_overview(
    core_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
    fetch_df: pd.DataFrame,
    fetch_queue_df: pd.DataFrame,
) -> Dict[str, object]:
    attempts = len(fetch_df)
    success = 0
    if attempts and "fetch_status" in fetch_df.columns:
        success = int(fetch_df["fetch_status"].isin(["FOUND_PDF", "SUCCESS", "OK"]).sum())

    return {
        "run_timestamp": datetime.now().isoformat(timespec="seconds"),
        "input_dir": str(INPUT_DIR),
        "report_dir": str(REPORT_DIR),
        "core_papers": int(len(core_df)),
        "candidates_scored": int(len(candidate_df)),
        "fetch_queue_size": int(len(fetch_queue_df)),
        "fetch_attempts": int(attempts),
        "fetch_success": int(success),
        "fetch_success_rate_pct": pct(success, attempts),
    }

overview = build_run_overview(core_df, candidate_df, fetch_df, fetch_queue_df)
pd.DataFrame([overview])

# %%
# --------------------------------------------------
# 10.7 Human review tables
# --------------------------------------------------
def build_core_review(df: pd.DataFrame, topk: int = 30) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    cols = [c for c in ["paper_uid", "title", "year", "cited_by_count", "rq_score", "doi", "openalex_id"] if c in df.columns]
    out = df.copy()
    if "cited_by_count" in out.columns:
        out = out.sort_values("cited_by_count", ascending=False)
    return out[cols].head(topk)

def build_top_candidates(df: pd.DataFrame, topk: int = 50) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    score_col = "priority_score" if "priority_score" in df.columns else None
    out = df.copy()

    if score_col and out[score_col].notna().any():
        out = topn(out, topk, by=score_col, asc=False)
    else:
        out = out.head(topk)

    cols = [
        "paper_uid", "title", "year", "venue",
        "cited_by_count", "rq_score", "priority_score",
        "score_explain", "hop", "source_core_id",
        "oa_url", "doi", "openalex_id",
    ]
    cols = [c for c in cols if c in out.columns]
    out = out[cols].copy()

    if "why" not in out.columns:
        out["why"] = "RQ=" + out["rq_score"].astype(str) + " | cites=" + out["cited_by_count"].astype(str)

    return out

core_review_df = build_core_review(core_df, topk=30)
top_candidates_df = build_top_candidates(candidate_df, topk=50)

core_review_df.head(10), top_candidates_df.head(10)

# %%
# --------------------------------------------------
# 10.8 Fetch diagnostics
# --------------------------------------------------
def build_fetch_diagnostics(fetch_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    if fetch_df.empty:
        return {"by_status": pd.DataFrame(), "by_error": pd.DataFrame()}

    by_status = pd.DataFrame()
    if "fetch_status" in fetch_df.columns:
        by_status = (
            fetch_df["fetch_status"].fillna("NA").astype(str)
            .value_counts()
            .reset_index()
        )
        by_status.columns = ["fetch_status", "count"]

    by_error = pd.DataFrame()
    if "error_type" in fetch_df.columns:
        by_error = (
            fetch_df["error_type"].fillna("NA").astype(str)
            .value_counts()
            .reset_index()
        )
        by_error.columns = ["error_type", "count"]

    return {"by_status": by_status, "by_error": by_error}

fetch_diag = build_fetch_diagnostics(fetch_df)
fetch_diag["by_status"].head(20)

# %%
fetch_diag["by_error"].head(20)

# %%
# --------------------------------------------------
# 10.9 New vs existing diff (optional)
# --------------------------------------------------
def diff_new_vs_existing(
    candidates: pd.DataFrame,
    existing: pd.DataFrame,
    id_cols: Tuple[str, ...] = ("paper_uid", "openalex_id", "doi"),
) -> Dict[str, pd.DataFrame]:
    if candidates.empty or existing.empty:
        return {"new": candidates.copy(), "existing": pd.DataFrame()}

    existing_ids = set()
    for c in id_cols:
        if c in existing.columns:
            existing_ids |= set(existing[c].dropna().astype(str))

    def is_existing(row) -> bool:
        for c in id_cols:
            if c in row and pd.notna(row[c]) and str(row[c]) in existing_ids:
                return True
        return False

    mask = candidates.apply(is_existing, axis=1)
    return {"new": candidates[~mask].copy(), "existing": candidates[mask].copy()}

diff = diff_new_vs_existing(candidate_df, existing_df)
len(diff["new"]), len(diff["existing"])

# %%
# --------------------------------------------------
# 10.10 Export artifacts (CSV + Markdown)
# --------------------------------------------------
def export_reports(
    overview: Dict[str, object],
    core_review_df: pd.DataFrame,
    top_candidates_df: pd.DataFrame,
    fetch_df: pd.DataFrame,
    fetch_diag: Dict[str, pd.DataFrame],
    diff: Dict[str, pd.DataFrame],
) -> Dict[str, Path]:

    paths: Dict[str, Path] = {}

    # CSV exports
    paths["overview_csv"] = REPORT_DIR / "overview.csv"
    pd.DataFrame([overview]).to_csv(paths["overview_csv"], index=False)

    paths["core_review_csv"] = REPORT_DIR / "core_review.csv"
    core_review_df.to_csv(paths["core_review_csv"], index=False)

    paths["top_candidates_csv"] = REPORT_DIR / "top_candidates.csv"
    top_candidates_df.to_csv(paths["top_candidates_csv"], index=False)

    paths["fetch_results_csv"] = REPORT_DIR / "fetch_results.csv"
    fetch_df.to_csv(paths["fetch_results_csv"], index=False)

    paths["fetch_by_status_csv"] = REPORT_DIR / "fetch_by_status.csv"
    fetch_diag["by_status"].to_csv(paths["fetch_by_status_csv"], index=False)

    paths["fetch_by_error_csv"] = REPORT_DIR / "fetch_by_error.csv"
    fetch_diag["by_error"].to_csv(paths["fetch_by_error_csv"], index=False)

    paths["new_candidates_csv"] = REPORT_DIR / "new_candidates.csv"
    diff["new"].to_csv(paths["new_candidates_csv"], index=False)

    # Markdown executive report
    md = []
    md.append(f"# Day20 Run Report ({overview['run_timestamp']})\n")

    md.append("## Overview")
    for k, v in overview.items():
        md.append(f"- **{k}**: {v}")

    md.append("\n## Core papers (preview)")
    md.append(core_review_df.head(15).to_markdown(index=False) if not core_review_df.empty else "_No core papers loaded_")

    md.append("\n## Top candidates (preview)")
    md.append(top_candidates_df.head(15).to_markdown(index=False) if not top_candidates_df.empty else "_No candidates loaded_")

    md.append("\n## Fetch status distribution")
    md.append(fetch_diag["by_status"].to_markdown(index=False) if not fetch_diag["by_status"].empty else "_No fetch status data_")

    md.append("\n## Error distribution")
    md.append(fetch_diag["by_error"].to_markdown(index=False) if not fetch_diag["by_error"].empty else "_No error data_")

    md.append("\n## Human review checklist")
    md.append("- Remove false positives from top candidates")
    md.append("- Mark must-fetch vs nice-to-have")
    md.append("- Decide retry rules for failures")
    md.append("- Spot-check Notion/Drive links for 5–10 papers")
    md.append("- Adjust scoring weights / hop depth for next weekly run")

    paths["executive_report_md"] = REPORT_DIR / "executive_report.md"
    paths["executive_report_md"].write_text("\n".join(md), encoding="utf-8")

    return paths

paths = export_reports(
    overview=overview,
    core_review_df=core_review_df,
    top_candidates_df=top_candidates_df,
    fetch_df=fetch_df,
    fetch_diag=fetch_diag,
    diff=diff,
)

print("[DONE] Exported:")
for k, v in paths.items():
    print(f"  - {k:20s}: {v}")

# %%
# --------------------------------------------------
# 10.11 Print checklist
# --------------------------------------------------
print("Human Review Checklist")
print("----------------------")
print("1. Review core papers (sanity check).")
print("2. Scan top candidates; remove false positives.")
print("3. Mark 'Must-fetch' vs 'Nice-to-have'.")
print("4. Inspect fetch failures; decide retry policy.")
print("5. Spot-check Notion & Drive links.")
print("6. Tune parameters for next weekly run.")


[INFO] Using ARTIFACTS_DIR: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442
[INFO] Writing reports to:  /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/reports_20260113_123414
[PICK] Using CURRENT run artifacts: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442 (15 CSVs)
[INFO] INPUT_DIR = /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442
[OK] Loaded core_df: core_openalex_resolved.csv (30 rows, 4KB, mtime=2026-01-13 07:06:08)
[OK] Loaded candidate_df: candidate_pool_enriched_scored.csv (902 rows, 2MB, mtime=2026-01-13 09:38:53)
[OK] Loaded fetch_df: pdf_download_results.csv (8 rows, 3KB, mtime=2026-01-13 10:05:45)
[OK] Loaded fetch_queue_df: backfill_fetch_queue_top200.csv (200 rows, 49KB, mtime=2026-01-13 09:45:50)
[OK] Loaded existing_df: corpus_with_rq_scores.csv (50 rows, 15KB, mtime=2026-01-13 06:27:36)
[DONE] Exported:
  - overvi

In [40]:
# %%
from pathlib import Path
import pandas as pd
import numpy as np

assert "ARTIFACTS_DIR" in globals(), "ARTIFACTS_DIR is not defined."
ARTIFACTS_DIR = Path(ARTIFACTS_DIR)

fetch_df = pd.read_csv(ARTIFACTS_DIR / "pdf_download_results.csv")
cand_df  = pd.read_csv(ARTIFACTS_DIR / "candidate_pool_enriched_scored.csv")

# -------------------------
# 1) Normalize / derive fields
# -------------------------
# Success flag
if "ok" in fetch_df.columns:
    fetch_df["ok"] = fetch_df["ok"].astype(bool)
else:
    fetch_df["ok"] = False

# Define fetch_status from ok + error info
def infer_status(r):
    if r.get("ok") is True:
        return "SUCCESS"
    code = r.get("final_error_code")
    msg  = r.get("final_error_message")
    if pd.notna(code) and str(code).strip() != "":
        return f"FAIL:{code}"
    if pd.notna(msg) and str(msg).strip() != "":
        return "FAIL"
    return "FAIL:UNKNOWN"

fetch_df["fetch_status"] = fetch_df.apply(infer_status, axis=1)

# error_type / message
fetch_df["error_type"] = fetch_df["final_error_code"]
fetch_df["error_message"] = fetch_df["final_error_message"]

# unify url fields
fetch_df["pdf_url"] = fetch_df.get("final_url", np.nan)
fetch_df["chosen_url"] = fetch_df.get("chosen_url", np.nan)

# Candidates: ensure key fields exist
for c in ["doi", "title", "publication_year", "cited_by_count", "rq_score", "oa_status", "oa_url", "venue",
          "abstract", "abstract_inverted_index", "openalex_id", "openalex_wid", "url_candidates",
          "title_norm", "doi_norm", "openalex_norm"]:
    if c not in cand_df.columns:
        cand_df[c] = np.nan

# Some runs may have abstract only as inverted index; backfill a preview string if needed
def abstract_preview(row, max_chars=400):
    # Prefer "abstract" text if present
    if pd.notna(row.get("abstract")) and str(row.get("abstract")).strip() != "":
        s = str(row.get("abstract"))
        return s[:max_chars]
    # Otherwise show a small preview of inverted index keys (not perfect, but better than None)
    inv = row.get("abstract_inverted_index")
    if pd.isna(inv) or inv is None or str(inv).strip() == "":
        return np.nan
    # inv might be stored as stringified dict; show first chunk
    s = str(inv)
    return (s[:max_chars] + ("..." if len(s) > max_chars else ""))

cand_df["abstract_preview"] = cand_df.apply(abstract_preview, axis=1)

# -------------------------
# 2) Merge by DOI (confirmed 100% match)
# -------------------------
# Normalize DOI form for safety (lowercase, strip)
def norm_doi(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    s = s.replace("https://doi.org/", "").replace("http://doi.org/", "")
    s = s.replace("doi:", "").strip()
    return s if s else np.nan

fetch_df["doi_norm"] = fetch_df["doi"].apply(norm_doi)
cand_df["doi_norm"]  = cand_df["doi"].apply(norm_doi)

merged = fetch_df.merge(
    cand_df.add_suffix("_cand"),
    left_on="doi_norm",
    right_on="doi_norm_cand",
    how="left"
)

# -------------------------
# 3) Build failures table (human follow-up)
# -------------------------
failures = merged[merged["ok"] == False].copy()

failures_enriched_df = pd.DataFrame({
    "title": failures["title"],
    "publication_year": failures.get("publication_year", np.nan),
    "cited_by_count": failures.get("cited_by_count", np.nan),
    "rq_score": failures.get("rq_score", np.nan),
    "priority_score": failures.get("priority_score", np.nan),

    "oa_status": failures.get("oa_status", np.nan),
    "oa_url": failures.get("oa_url", np.nan),

    "url_candidates": failures.get("url_candidates", np.nan),
    "chosen_url": failures.get("chosen_url", np.nan),
    "final_url": failures.get("final_url", np.nan),

    "fetch_status": failures.get("fetch_status", np.nan),
    "error_type": failures.get("error_type", np.nan),
    "error_message": failures.get("error_message", np.nan),

    "local_path": failures.get("local_path", np.nan),
    "bytes": failures.get("bytes", np.nan),

    # from candidate table (if you want)
    "venue": failures.get("venue_cand", np.nan),
    "openalex_id": failures.get("openalex_id_cand", np.nan),
    "openalex_wid": failures.get("openalex_wid_cand", np.nan),
    "abstract": failures.get("abstract_cand", np.nan),
    "abstract_preview": failures.get("abstract_preview_cand", np.nan),
})

# numeric coercion & sort
for col in ["publication_year", "cited_by_count", "rq_score", "priority_score", "bytes"]:
    if col in failures_enriched_df.columns:
        failures_enriched_df[col] = pd.to_numeric(failures_enriched_df[col], errors="coerce")

failures_enriched_df = failures_enriched_df.sort_values(
    ["priority_score", "rq_score", "cited_by_count"],
    ascending=False
)

print("Failures enriched:", len(failures_enriched_df))
failures_enriched_df.head(10)

# -------------------------
# 4) Export CSV + Markdown
# -------------------------
out_csv = ARTIFACTS_DIR / "failures_enriched.csv"
failures_enriched_df.to_csv(out_csv, index=False)

# quick markdown top50
md_path = ARTIFACTS_DIR / "failures_top50.md"
md_cols = [
    "title", "publication_year", "cited_by_count", "rq_score", "priority_score",
    "fetch_status", "error_type", "error_message", "oa_url", "final_url", "chosen_url", "doi_norm"
]
md_cols = [c for c in md_cols if c in failures_enriched_df.columns]
md = failures_enriched_df[md_cols].head(50).to_markdown(index=False)

md_path.write_text("# Failures Top50 (manual follow-up)\n\n" + md + "\n", encoding="utf-8")

print("[DONE] wrote:", out_csv)
print("[DONE] wrote:", md_path)


Failures enriched: 7
[DONE] wrote: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/failures_enriched.csv
[DONE] wrote: /Users/yuetoya/Desktop/researchOS100-private/notebooks/artifacts/day20/20260113_061442/failures_top50.md
